## PORTION-1

In [1]:
# ============================================================
# TOKENIZER PORTION 1 — FINAL VERSION
# Data Registry, Loading, Provenance, Split Isolation
# Bangla Dialect SLM/LLM Tokenizer Pipeline
# ============================================================

from __future__ import annotations

import os
import re
import json
import random
import hashlib
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# ============================================================
# 0. Reproducibility
# ============================================================

SEED = 42

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)


# ============================================================
# 1. Config
# ============================================================

@dataclass(frozen=True)
class Portion1Config:
    kaggle_input_root: str = "/kaggle/input"
    dataset_prime_name: str = "dataset-prime"
    synthetic_dir_name: str = "synthetic_data"

    output_dir: str = "artifacts/tokenizer_p1"

    include_vashantor_target_standard: bool = False
    fail_if_unk_remains: bool = True

    min_text_chars: int = 1
    hash_lowercase: bool = False

    expected_synthetic_files: Tuple[str, ...] = (
        "bangla_dialect_balancing_synthetic_v0.csv",
        "khulna_rajshahi_7000_each (1).csv",
        "mymensingh_noakhali_7000_each (1).csv",
    )


CFG = Portion1Config()
OUT_DIR = Path(CFG.output_dir)
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. Dialect Taxonomy
# ============================================================

TARGET_DIALECTS_10 = [
    "BAR", "CHI", "KHU", "MYM", "NAR",
    "NOA", "RAJ", "RAN", "STD", "SYL",
]

AUXILIARY_DIALECTS = ["KIS", "TAN", "NSD"]

ALL_DIALECTS_FOR_TOKENIZER = sorted(set(TARGET_DIALECTS_10 + AUXILIARY_DIALECTS))

DIALECT_MAP: Dict[str, str] = {
    # Standard
    "standard_bangla": "STD",
    "standard bangla": "STD",
    "standard": "STD",
    "std": "STD",
    "bangla": "STD",
    "bengali": "STD",
    "বাংলা": "STD",
    "প্রমিত বাংলা": "STD",
    "স্ট্যান্ডার্ড বাংলা": "STD",

    # Barishal
    "barishal": "BAR",
    "barisal": "BAR",
    "bar": "BAR",
    "বরিশাল": "BAR",

    # Chittagong / Chattogram
    "chittagong": "CHI",
    "chittagonian": "CHI",
    "chattogram": "CHI",
    "chatgaiyya": "CHI",
    "chatgaiya": "CHI",
    "chi": "CHI",
    "চট্টগ্রাম": "CHI",
    "চাটগাঁ": "CHI",
    "চাটগাঁইয়া": "CHI",
    "চাটগাঁইয়া": "CHI",

    # Khulna
    "khulna": "KHU",
    "khu": "KHU",
    "খুলনা": "KHU",

    # Kishoreganj auxiliary
    "kishoreganj": "KIS",
    "kis": "KIS",
    "কিশোরগঞ্জ": "KIS",

    # Mymensingh
    "mymensingh": "MYM",
    "mymensing": "MYM",
    "mym": "MYM",
    "ময়মনসিংহ": "MYM",
    "ময়মনসিংহ": "MYM",

    # Narail
    "narail": "NAR",
    "nar": "NAR",
    "নড়াইল": "NAR",
    "নড়াইল": "NAR",

    # Noakhali
    "noakhali": "NOA",
    "noa": "NOA",
    "নোয়াখালী": "NOA",
    "নোয়াখালী": "NOA",

    # Narsingdi auxiliary
    "narsingdi": "NSD",
    "nsd": "NSD",
    "নরসিংদী": "NSD",

    # Rajshahi
    "rajshahi": "RAJ",
    "raj": "RAJ",
    "রাজশাহী": "RAJ",

    # Rangpur
    "rangpur": "RAN",
    "rangpuri": "RAN",
    "ran": "RAN",
    "রংপুর": "RAN",
    "রংপুরী": "RAN",

    # Sylhet
    "sylhet": "SYL",
    "sylheti": "SYL",
    "syl": "SYL",
    "সিলেট": "SYL",
    "সিলেটি": "SYL",

    # Tangail auxiliary
    "tangail": "TAN",
    "tan": "TAN",
    "টাঙ্গাইল": "TAN",

    # Dhaka optional if source contains it
    "dhaka": "DHA",
    "dha": "DHA",
    "ঢাকা": "DHA",
}


# ============================================================
# 3. Schema
# ============================================================

REQUIRED_SCHEMA = [
    "text",
    "dialect",
    "source",
    "split_original",
    "is_synthetic",
    "synthetic_method",
    "domain",
    "translation_pair_id",
    "quality_score",
    "text_hash",
    "near_duplicate_group",
]

EXTRA_SCHEMA = [
    "dialect_raw",
    "source_file",
    "source_row_id",
    "text_role",
    "taxonomy_role",
    "eligible_for_tokenizer_train",
    "eligible_for_dialect_routing",
    "eligible_for_main_10dialect_benchmark",
    "load_warning",
]

ALL_COLUMNS = REQUIRED_SCHEMA + EXTRA_SCHEMA


# ============================================================
# 4. Utility Functions
# ============================================================

def norm_col(col: Any) -> str:
    return str(col).strip().lower().replace("-", "_").replace("/", "_")


def minimal_text_normalize(text: Any) -> Optional[str]:
    """
    Portion 1 uses minimal normalization only.
    Full artifact removal and cleaning happen in Portion 2.
    """
    if pd.isna(text):
        return None

    text = str(text)
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    if len(text) < CFG.min_text_chars:
        return None

    return text


def normalize_for_hash(text: str) -> str:
    s = minimal_text_normalize(text) or ""
    if CFG.hash_lowercase:
        s = s.lower()
    return s


def sha256_str(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def make_text_hash(text: str) -> str:
    return sha256_str(normalize_for_hash(text))


def standardize_dialect_label(label: Any) -> Tuple[str, str]:
    """
    Unknown labels are returned as UNK.
    They are never silently mapped to STD.
    """
    raw = "" if pd.isna(label) else str(label).strip()
    key = raw.lower().strip()
    key_space = key.replace("_", " ")
    key_under = key.replace(" ", "_")

    if key in DIALECT_MAP:
        return DIALECT_MAP[key], raw
    if key_space in DIALECT_MAP:
        return DIALECT_MAP[key_space], raw
    if key_under in DIALECT_MAP:
        return DIALECT_MAP[key_under], raw

    return "UNK", raw


def find_first_file(root: Path, filename: str) -> Optional[Path]:
    matches = sorted(root.rglob(filename), key=lambda p: len(str(p)))
    return matches[0] if matches else None


def read_csv_safely(path: Path) -> pd.DataFrame:
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
    last_error = None

    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last_error = e

    raise RuntimeError(f"Could not read CSV file: {path} | last_error={last_error}")


def read_excel_safely(path: Path, sheet_name: Optional[str] = None) -> pd.DataFrame:
    if sheet_name is None:
        return pd.read_excel(path)
    return pd.read_excel(path, sheet_name=sheet_name)


TEXT_COL_CANDIDATES = [
    "text",
    "sentence",
    "sentences",
    "bangla_speech",
    "regional_text",
    "regional_texts",
    "regional texts",
    "regional_text",
    "local bangla dialect",
    "local bangla dialect(sylheti) text",
    "local_bangla_dialect",
    "চট্টগ্রাম",
]

DIALECT_COL_CANDIDATES = [
    "dialect",
    "language",
    "region",
    "region_labels",
    "region labels",
    "labels",
    "label",
    "class",
    "category",
]

STANDARD_TARGET_COL_CANDIDATES = [
    "standard_bangla_text",
    "standard bangla text",
    "standard_bangla",
    "standard bangla",
    "target",
    "target_text",
    "translation",
]


def find_best_column(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    normalized = {norm_col(c): c for c in df.columns}

    for cand in candidates:
        c = norm_col(cand)
        if c in normalized:
            return normalized[c]

    for col in df.columns:
        col_norm = norm_col(col)
        for cand in candidates:
            if norm_col(cand) in col_norm:
                return col

    return None


def infer_text_column(df: pd.DataFrame, preferred: Optional[str] = None) -> str:
    if preferred and preferred in df.columns:
        return preferred

    found = find_best_column(df, TEXT_COL_CANDIDATES)
    if found:
        return found

    object_cols = [c for c in df.columns if df[c].dtype == "object"]
    if not object_cols:
        raise ValueError("No text-like column found.")

    scored = []
    for c in object_cols:
        sample = df[c].dropna().astype(str).head(500)
        avg_len = sample.map(len).mean() if len(sample) else 0
        scored.append((avg_len, c))

    scored.sort(reverse=True)
    return scored[0][1]


def infer_dialect_column(df: pd.DataFrame) -> Optional[str]:
    return find_best_column(df, DIALECT_COL_CANDIDATES)


def infer_standard_target_column(df: pd.DataFrame) -> Optional[str]:
    return find_best_column(df, STANDARD_TARGET_COL_CANDIDATES)


def make_translation_pair_id(source: str, split: str, dialect: str, row_idx: int) -> str:
    return sha256_str(f"{source}|{split}|{dialect}|{row_idx}")[:24]


def taxonomy_role_for_dialect(dialect: str) -> str:
    if dialect in AUXILIARY_DIALECTS:
        return "auxiliary"
    if dialect in TARGET_DIALECTS_10:
        return "target"
    if dialect == "UNK":
        return "unknown"
    return "extra"


def build_row(
    *,
    text: Any,
    dialect: str,
    dialect_raw: str,
    source: str,
    split_original: str,
    is_synthetic: bool,
    synthetic_method: str,
    domain: str,
    translation_pair_id: Optional[str],
    quality_score: Optional[float],
    source_file: str,
    source_row_id: int,
    text_role: str,
    eligible_for_tokenizer_train: bool,
    load_warning: str = "",
) -> Optional[Dict[str, Any]]:

    clean_text = minimal_text_normalize(text)
    if clean_text is None:
        return None

    h = make_text_hash(clean_text)

    if translation_pair_id is None or str(translation_pair_id).strip() == "":
        translation_pair_id = make_translation_pair_id(source, split_original, dialect, source_row_id)

    return {
        "text": clean_text,
        "dialect": dialect,
        "source": source,
        "split_original": split_original,
        "is_synthetic": bool(is_synthetic),
        "synthetic_method": synthetic_method,
        "domain": domain,
        "translation_pair_id": translation_pair_id,
        "quality_score": quality_score,
        "text_hash": h,
        "near_duplicate_group": f"exact:{h[:16]}",

        "dialect_raw": dialect_raw,
        "source_file": source_file,
        "source_row_id": int(source_row_id),
        "text_role": text_role,
        "taxonomy_role": taxonomy_role_for_dialect(dialect),
        "eligible_for_tokenizer_train": bool(eligible_for_tokenizer_train) and dialect != "UNK",
        "eligible_for_dialect_routing": dialect in ALL_DIALECTS_FOR_TOKENIZER,
        "eligible_for_main_10dialect_benchmark": dialect in TARGET_DIALECTS_10,
        "load_warning": load_warning,
    }


# ============================================================
# 5. Registry Builder
# ============================================================

class BanglaDialectRegistryBuilder:
    def __init__(self, cfg: Portion1Config):
        self.cfg = cfg
        self.input_root = Path(cfg.kaggle_input_root)
        self.dataset_root = self._resolve_dataset_root()
        self.synthetic_roots = self._resolve_synthetic_roots()

        self.rows: List[Dict[str, Any]] = []
        self.source_reports: List[Dict[str, Any]] = []

        print("=" * 100)
        print("TOKENIZER PORTION 1 — FINAL: DATA REGISTRY, PROVENANCE, SPLIT ISOLATION")
        print("=" * 100)
        print(f"Input root     : {self.input_root}")
        print(f"Dataset root   : {self.dataset_root}")
        print(f"Synthetic roots: {[str(p) for p in self.synthetic_roots]}")
        print(f"Output dir     : {OUT_DIR}")
        print("=" * 100)

    def _resolve_dataset_root(self) -> Path:
        direct = self.input_root / self.cfg.dataset_prime_name
        if direct.exists():
            return direct

        candidates = [
            p for p in self.input_root.rglob("*")
            if p.is_dir() and self.cfg.dataset_prime_name.lower() in p.name.lower()
        ]
        if candidates:
            return sorted(candidates, key=lambda p: len(str(p)))[0]

        return self.input_root

    def _resolve_synthetic_roots(self) -> List[Path]:
        candidates = []

        direct_1 = self.dataset_root / self.cfg.synthetic_dir_name
        direct_2 = self.input_root / self.cfg.synthetic_dir_name

        if direct_1.exists():
            candidates.append(direct_1)
        if direct_2.exists():
            candidates.append(direct_2)

        for p in self.input_root.rglob("*"):
            if p.is_dir() and "synthetic" in p.name.lower():
                candidates.append(p)

        unique = []
        seen = set()
        for p in candidates:
            rp = str(p.resolve())
            if rp not in seen:
                seen.add(rp)
                unique.append(p)

        return unique

    def _append(self, rows: List[Optional[Dict[str, Any]]]) -> int:
        kept = 0
        for r in rows:
            if r is not None:
                self.rows.append(r)
                kept += 1
        return kept

    def _report(
        self,
        *,
        source: str,
        source_file: Path,
        rows_loaded: int,
        rows_kept: int,
        split_original: str,
        is_synthetic: bool,
        text_col: Optional[str] = None,
        dialect_col: Optional[str] = None,
        warning: str = "",
    ) -> None:
        self.source_reports.append({
            "source": source,
            "source_file": str(source_file),
            "rows_loaded": int(rows_loaded),
            "rows_kept": int(rows_kept),
            "split_original": split_original,
            "is_synthetic": bool(is_synthetic),
            "text_col": text_col,
            "dialect_col": dialect_col,
            "warning": warning,
        })

    # --------------------------------------------------------
    # Real sources
    # --------------------------------------------------------

    def load_bangladial(self) -> None:
        filename = "BanglaDial_ A Merged and Imbalanced text Dataset for Bengali Regional dialect analysis. - Sheet1.csv"
        path = find_first_file(self.dataset_root, filename)

        if path is None:
            print(f"[WARN] Missing BanglaDial: {filename}")
            return

        df = read_csv_safely(path)
        text_col = infer_text_column(df, preferred="Sentence")
        dialect_col = infer_dialect_column(df)

        rows = []
        warning = "" if dialect_col else "Missing dialect column; dialect=UNK."

        for i, row in df.iterrows():
            if dialect_col:
                dialect, raw = standardize_dialect_label(row[dialect_col])
            else:
                dialect, raw = "UNK", ""

            rows.append(build_row(
                text=row[text_col],
                dialect=dialect,
                dialect_raw=raw,
                source="BanglaDial",
                split_original="unspecified",
                is_synthetic=False,
                synthetic_method="none",
                domain="mixed_regional_monolingual",
                translation_pair_id=None,
                quality_score=None,
                source_file=str(path),
                source_row_id=i,
                text_role="monolingual",
                eligible_for_tokenizer_train=True,
                load_warning=warning,
            ))

        kept = self._append(rows)
        self._report(
            source="BanglaDial",
            source_file=path,
            rows_loaded=len(df),
            rows_kept=kept,
            split_original="unspecified",
            is_synthetic=False,
            text_col=text_col,
            dialect_col=dialect_col,
            warning=warning,
        )
        print(f"[OK] BanglaDial: loaded={len(df):,}, kept={kept:,}, text_col={text_col}, dialect_col={dialect_col}")

    def load_vashantor(self) -> None:
        dialect_names = ["Barishal", "Chittagong", "Mymensingh", "Noakhali", "Sylhet"]
        splits = ["Train", "Validation", "Test"]

        for split in splits:
            for dialect_name in dialect_names:
                filename = f"{dialect_name} {split} Translation.csv"
                path = find_first_file(self.dataset_root, filename)

                if path is None:
                    print(f"[WARN] Missing Vashantor file: {filename}")
                    continue

                df = read_csv_safely(path)
                regional_col = infer_text_column(df)
                target_col = infer_standard_target_column(df)

                dialect, raw = standardize_dialect_label(dialect_name)
                eligible = split.lower() == "train"

                rows = []
                for i, row in df.iterrows():
                    pair_id = make_translation_pair_id("Vashantor", split.lower(), dialect, i)

                    rows.append(build_row(
                        text=row[regional_col],
                        dialect=dialect,
                        dialect_raw=raw,
                        source=f"Vashantor_{dialect}_{split}",
                        split_original=split.lower(),
                        is_synthetic=False,
                        synthetic_method="none",
                        domain="parallel_translation",
                        translation_pair_id=pair_id,
                        quality_score=None,
                        source_file=str(path),
                        source_row_id=i,
                        text_role="regional_source",
                        eligible_for_tokenizer_train=eligible,
                    ))

                    if self.cfg.include_vashantor_target_standard and target_col is not None:
                        rows.append(build_row(
                            text=row[target_col],
                            dialect="STD",
                            dialect_raw="standard_bangla",
                            source=f"Vashantor_STD_Target_{split}",
                            split_original=split.lower(),
                            is_synthetic=False,
                            synthetic_method="none",
                            domain="parallel_translation",
                            translation_pair_id=pair_id,
                            quality_score=None,
                            source_file=str(path),
                            source_row_id=i,
                            text_role="target_standard",
                            eligible_for_tokenizer_train=eligible,
                        ))

                kept = self._append(rows)
                self._report(
                    source=f"Vashantor_{dialect}_{split}",
                    source_file=path,
                    rows_loaded=len(df),
                    rows_kept=kept,
                    split_original=split.lower(),
                    is_synthetic=False,
                    text_col=regional_col,
                    dialect_col=None,
                )
                print(
                    f"[OK] Vashantor {dialect_name} {split}: "
                    f"loaded={len(df):,}, kept={kept:,}, regional_col={regional_col}, target_col={target_col}"
                )

    def load_sylheti_parallel(self) -> None:
        filename = "Bangla Dialect Transaction Dataset_v2 - Sheet1.csv"
        path = find_first_file(self.dataset_root, filename)

        if path is None:
            print(f"[WARN] Missing Sylheti parallel: {filename}")
            return

        df = read_csv_safely(path)
        text_col = infer_text_column(df)

        rows = []
        for i, row in df.iterrows():
            rows.append(build_row(
                text=row[text_col],
                dialect="SYL",
                dialect_raw="sylheti",
                source="Sylheti_Parallel",
                split_original="unspecified",
                is_synthetic=False,
                synthetic_method="none",
                domain="parallel_translation",
                translation_pair_id=make_translation_pair_id("Sylheti_Parallel", "unspecified", "SYL", i),
                quality_score=None,
                source_file=str(path),
                source_row_id=i,
                text_role="regional_source",
                eligible_for_tokenizer_train=True,
            ))

        kept = self._append(rows)
        self._report(
            source="Sylheti_Parallel",
            source_file=path,
            rows_loaded=len(df),
            rows_kept=kept,
            split_original="unspecified",
            is_synthetic=False,
            text_col=text_col,
        )
        print(f"[OK] Sylheti_Parallel: loaded={len(df):,}, kept={kept:,}, text_col={text_col}")

    def load_regional_corpus(self) -> None:
        filename = "BanglaRegionalTextCorpus.xlsx"
        path = find_first_file(self.dataset_root, filename)

        if path is None:
            print(f"[WARN] Missing BanglaRegionalTextCorpus: {filename}")
            return

        df = read_excel_safely(path, sheet_name="Sheet1")
        text_col = find_best_column(df, ["Regional_Texts", "Regional Texts"]) or infer_text_column(df)
        dialect_col = find_best_column(df, ["Region_Labels", "Region Labels", "dialect", "region"])

        rows = []
        warning = "" if dialect_col else "Missing dialect column; dialect=UNK."

        for i, row in df.iterrows():
            if dialect_col:
                dialect, raw = standardize_dialect_label(row[dialect_col])
            else:
                dialect, raw = "UNK", ""

            rows.append(build_row(
                text=row[text_col],
                dialect=dialect,
                dialect_raw=raw,
                source="BanglaRegionalTextCorpus",
                split_original="unspecified",
                is_synthetic=False,
                synthetic_method="none",
                domain="regional_corpus",
                translation_pair_id=None,
                quality_score=None,
                source_file=str(path),
                source_row_id=i,
                text_role="monolingual",
                eligible_for_tokenizer_train=True,
                load_warning=warning,
            ))

        kept = self._append(rows)
        self._report(
            source="BanglaRegionalTextCorpus",
            source_file=path,
            rows_loaded=len(df),
            rows_kept=kept,
            split_original="unspecified",
            is_synthetic=False,
            text_col=text_col,
            dialect_col=dialect_col,
            warning=warning,
        )
        print(f"[OK] BanglaRegionalTextCorpus: loaded={len(df):,}, kept={kept:,}, text_col={text_col}, dialect_col={dialect_col}")

    def load_regional_cleaned_dataset(self) -> None:
        filename = "Regional_cleaned_dataset.xlsx"
        path = find_first_file(self.dataset_root, filename)

        if path is None:
            print(f"[INFO] Optional Regional_cleaned_dataset not found: {filename}")
            return

        df = read_excel_safely(path)
        text_col = infer_text_column(df)
        dialect_col = infer_dialect_column(df)

        rows = []
        warning = "" if dialect_col else "Missing dialect column; dialect=UNK."

        for i, row in df.iterrows():
            if dialect_col:
                dialect, raw = standardize_dialect_label(row[dialect_col])
            else:
                dialect, raw = "UNK", ""

            rows.append(build_row(
                text=row[text_col],
                dialect=dialect,
                dialect_raw=raw,
                source="Regional_cleaned_dataset",
                split_original="unspecified",
                is_synthetic=False,
                synthetic_method="none",
                domain="regional_corpus_cleaned",
                translation_pair_id=None,
                quality_score=None,
                source_file=str(path),
                source_row_id=i,
                text_role="monolingual",
                eligible_for_tokenizer_train=True,
                load_warning=warning,
            ))

        kept = self._append(rows)
        self._report(
            source="Regional_cleaned_dataset",
            source_file=path,
            rows_loaded=len(df),
            rows_kept=kept,
            split_original="unspecified",
            is_synthetic=False,
            text_col=text_col,
            dialect_col=dialect_col,
            warning=warning,
        )
        print(f"[OK] Regional_cleaned_dataset: loaded={len(df):,}, kept={kept:,}, text_col={text_col}, dialect_col={dialect_col}")

    def load_chatgaiyya_alap(self) -> None:
        filename = "Dataset_Chittagong_2.0.csv"
        path = find_first_file(self.dataset_root, filename)

        if path is None:
            print(f"[WARN] Missing ChatgaiyyaAlap: {filename}")
            return

        df = read_csv_safely(path)
        text_col = infer_text_column(df)

        rows = []
        for i, row in df.iterrows():
            rows.append(build_row(
                text=row[text_col],
                dialect="CHI",
                dialect_raw="chittagong",
                source="ChatgaiyyaAlap",
                split_original="unspecified",
                is_synthetic=False,
                synthetic_method="none",
                domain="parallel_translation",
                translation_pair_id=make_translation_pair_id("ChatgaiyyaAlap", "unspecified", "CHI", i),
                quality_score=None,
                source_file=str(path),
                source_row_id=i,
                text_role="regional_source",
                eligible_for_tokenizer_train=True,
            ))

        kept = self._append(rows)
        self._report(
            source="ChatgaiyyaAlap",
            source_file=path,
            rows_loaded=len(df),
            rows_kept=kept,
            split_original="unspecified",
            is_synthetic=False,
            text_col=text_col,
        )
        print(f"[OK] ChatgaiyyaAlap: loaded={len(df):,}, kept={kept:,}, text_col={text_col}")

    def register_dictionary(self) -> None:
        filename = "dictionary.csv"
        path = find_first_file(self.dataset_root, filename)

        if path is None:
            print(f"[INFO] dictionary.csv not found.")
            return

        try:
            df = read_csv_safely(path)
            n = len(df)
        except Exception as e:
            n = 0
            warning = f"Could not read dictionary.csv: {e}"
        else:
            warning = "Registered as lexicon metadata only; not loaded into corpus in Portion 1."

        self.source_reports.append({
            "source": "dictionary",
            "source_file": str(path),
            "rows_loaded": int(n),
            "rows_kept": 0,
            "split_original": "lexicon",
            "is_synthetic": False,
            "text_col": None,
            "dialect_col": None,
            "warning": warning,
        })
        print(f"[OK] Registered dictionary.csv metadata: rows={n:,}")

    # --------------------------------------------------------
    # Synthetic sources
    # --------------------------------------------------------

    def _synthetic_csv_files(self) -> List[Path]:
        files = []

        for root in self.synthetic_roots:
            for p in root.rglob("*.csv"):
                files.append(p)

        # Fallback: search entire input root for expected synthetic file names.
        for name in self.cfg.expected_synthetic_files:
            matches = list(self.input_root.rglob(name))
            files.extend(matches)

        unique = []
        seen = set()
        for p in files:
            rp = str(p.resolve())
            if rp not in seen:
                seen.add(rp)
                unique.append(p)

        return sorted(unique, key=lambda p: str(p))

    def load_synthetic_data(self) -> None:
        files = self._synthetic_csv_files()

        if not files:
            print("[WARN] No synthetic CSV files found.")
            return

        found_names = {p.name for p in files}
        missing_expected = [n for n in self.cfg.expected_synthetic_files if n not in found_names]
        if missing_expected:
            print(f"[WARN] Missing expected synthetic files: {missing_expected}")

        for path in files:
            df = read_csv_safely(path)
            text_col = infer_text_column(df)
            dialect_col = infer_dialect_column(df)

            if dialect_col is None:
                print(f"[WARN] Synthetic file skipped because no dialect column was found: {path}")
                self._report(
                    source=f"Synthetic_{path.stem}",
                    source_file=path,
                    rows_loaded=len(df),
                    rows_kept=0,
                    split_original="synthetic_train",
                    is_synthetic=True,
                    text_col=text_col,
                    dialect_col=None,
                    warning="No dialect column.",
                )
                continue

            rows = []
            for i, row in df.iterrows():
                dialect, raw = standardize_dialect_label(row[dialect_col])

                source_value = row["source"] if "source" in df.columns and not pd.isna(row["source"]) else f"Synthetic_{path.stem}"
                split_value = row["split_original"] if "split_original" in df.columns and not pd.isna(row["split_original"]) else "synthetic_train"
                method_value = row["synthetic_method"] if "synthetic_method" in df.columns and not pd.isna(row["synthetic_method"]) else "llm_synthetic"
                domain_value = row["domain"] if "domain" in df.columns and not pd.isna(row["domain"]) else "synthetic_regional"

                if "quality_score" in df.columns and not pd.isna(row["quality_score"]):
                    try:
                        q_score = float(row["quality_score"])
                    except Exception:
                        q_score = None
                else:
                    q_score = None

                pair_id = row["translation_pair_id"] if "translation_pair_id" in df.columns and not pd.isna(row["translation_pair_id"]) else None

                rows.append(build_row(
                    text=row[text_col],
                    dialect=dialect,
                    dialect_raw=raw,
                    source=str(source_value),
                    split_original=str(split_value),
                    is_synthetic=True,
                    synthetic_method=str(method_value),
                    domain=str(domain_value),
                    translation_pair_id=pair_id,
                    quality_score=q_score,
                    source_file=str(path),
                    source_row_id=i,
                    text_role="synthetic_monolingual",
                    eligible_for_tokenizer_train=True,
                ))

            kept = self._append(rows)
            self._report(
                source=f"Synthetic_{path.stem}",
                source_file=path,
                rows_loaded=len(df),
                rows_kept=kept,
                split_original="synthetic_train",
                is_synthetic=True,
                text_col=text_col,
                dialect_col=dialect_col,
            )
            print(f"[OK] Synthetic {path.name}: loaded={len(df):,}, kept={kept:,}, text_col={text_col}, dialect_col={dialect_col}")

    # --------------------------------------------------------
    # Main execution
    # --------------------------------------------------------

    def build(self) -> pd.DataFrame:
        self.load_bangladial()
        self.load_vashantor()
        self.load_sylheti_parallel()
        self.load_regional_corpus()
        self.load_regional_cleaned_dataset()
        self.load_chatgaiyya_alap()
        self.load_synthetic_data()
        self.register_dictionary()

        if not self.rows:
            raise RuntimeError("No rows loaded. Check Kaggle paths.")

        df = pd.DataFrame(self.rows)

        for col in ALL_COLUMNS:
            if col not in df.columns:
                df[col] = None

        df = df[ALL_COLUMNS].copy()

        df = df.sort_values(
            by=["source", "split_original", "dialect", "source_file", "source_row_id"],
            kind="mergesort",
        ).reset_index(drop=True)

        return df

    def run_tests(self, df: pd.DataFrame) -> Dict[str, Any]:
        print("\n" + "=" * 100)
        print("PORTION 1 TESTS")
        print("=" * 100)

        results: Dict[str, Any] = {}

        missing_required = [c for c in REQUIRED_SCHEMA if c not in df.columns]
        missing_dialect = int(df["dialect"].isna().sum() + (df["dialect"].astype(str).str.len() == 0).sum())
        unk_count = int((df["dialect"] == "UNK").sum())
        missing_source = int(df["source"].isna().sum() + (df["source"].astype(str).str.len() == 0).sum())
        missing_synth_flag = int(df["is_synthetic"].isna().sum())
        missing_hash = int(df["text_hash"].isna().sum() + (df["text_hash"].astype(str).str.len() == 0).sum())

        heldout_mask = df["split_original"].isin(["validation", "test"])
        heldout_train_eligible = int(df.loc[heldout_mask, "eligible_for_tokenizer_train"].sum())

        synthetic_df = df[df["is_synthetic"] == True]
        synthetic_path_ok = True
        if len(synthetic_df) > 0:
            synthetic_path_ok = synthetic_df["source_file"].str.lower().str.contains("synthetic").all()

        expected_synthetic_loaded = {}
        for fname in CFG.expected_synthetic_files:
            expected_synthetic_loaded[fname] = bool(df["source_file"].str.contains(re.escape(fname), regex=True).any())

        failures = []
        if missing_required:
            failures.append(f"Missing required columns: {missing_required}")
        if missing_dialect > 0:
            failures.append(f"Missing dialect labels: {missing_dialect}")
        if CFG.fail_if_unk_remains and unk_count > 0:
            failures.append(f"UNK labels remain: {unk_count}")
        if missing_source > 0:
            failures.append(f"Missing source values: {missing_source}")
        if missing_synth_flag > 0:
            failures.append(f"Missing synthetic flags: {missing_synth_flag}")
        if missing_hash > 0:
            failures.append(f"Missing text hashes: {missing_hash}")
        if heldout_train_eligible > 0:
            failures.append(f"Validation/test rows eligible for tokenizer train: {heldout_train_eligible}")
        if not synthetic_path_ok:
            failures.append("Some synthetic rows do not originate from a synthetic path.")
        for fname, ok in expected_synthetic_loaded.items():
            if not ok:
                failures.append(f"Expected synthetic file not loaded: {fname}")

        results = {
            "required_schema_present": len(missing_required) == 0,
            "missing_required_columns": missing_required,
            "missing_dialect_count": missing_dialect,
            "unk_count": unk_count,
            "missing_source_count": missing_source,
            "missing_synthetic_flag_count": missing_synth_flag,
            "missing_text_hash_count": missing_hash,
            "heldout_rows_eligible_for_tokenizer_train_count": heldout_train_eligible,
            "synthetic_rows_only_from_synthetic_path": bool(synthetic_path_ok),
            "expected_synthetic_loaded": expected_synthetic_loaded,
            "hard_failures": failures,
            "passed": len(failures) == 0,
        }

        for k, v in results.items():
            if k != "hard_failures":
                print(f"{k:60s}: {v}")

        if failures:
            print("\n[FAILED]")
            for f in failures:
                print(f"  - {f}")
            raise AssertionError("Tokenizer Portion 1 tests failed.")

        print("\n[OK] Portion 1 tests passed.")
        return results

    def export(self, df: pd.DataFrame, test_results: Dict[str, Any]) -> None:
        OUT_DIR.mkdir(parents=True, exist_ok=True)

        registry = pd.DataFrame(self.source_reports)

        unknown = df[df["dialect"] == "UNK"].copy()

        dialect_count = {
            "overall": df["dialect"].value_counts().sort_index().to_dict(),
            "eligible_for_tokenizer_train": (
                df[df["eligible_for_tokenizer_train"] == True]["dialect"]
                .value_counts()
                .sort_index()
                .to_dict()
            ),
            "real_only": (
                df[df["is_synthetic"] == False]["dialect"]
                .value_counts()
                .sort_index()
                .to_dict()
            ),
            "synthetic_only": (
                df[df["is_synthetic"] == True]["dialect"]
                .value_counts()
                .sort_index()
                .to_dict()
            ),
            "target_10_only": (
                df[df["eligible_for_main_10dialect_benchmark"] == True]["dialect"]
                .value_counts()
                .sort_index()
                .to_dict()
            ),
            "auxiliary_only": (
                df[df["taxonomy_role"] == "auxiliary"]["dialect"]
                .value_counts()
                .sort_index()
                .to_dict()
            ),
            "by_dialect_synthetic_split": (
                df.groupby(["dialect", "is_synthetic", "split_original"], dropna=False)
                .size()
                .reset_index(name="count")
                .sort_values(["dialect", "is_synthetic", "split_original"])
                .to_dict(orient="records")
            ),
        }

        duplicate_exact_text = (
            df.groupby("text_hash")
            .size()
            .reset_index(name="count")
            .query("count > 1")
            .sort_values("count", ascending=False)
        )

        duplicate_exact_text_dialect = (
            df.groupby(["text_hash", "dialect"])
            .size()
            .reset_index(name="count")
            .query("count > 1")
            .sort_values("count", ascending=False)
        )

        taxonomy = {
            "target_dialects_10": TARGET_DIALECTS_10,
            "auxiliary_dialects": {
                "KIS": "Kishoreganj",
                "TAN": "Tangail",
                "NSD": "Narsingdi",
            },
            "all_dialects_for_tokenizer": ALL_DIALECTS_FOR_TOKENIZER,
            "standard_control": "STD",
            "notes": [
                "Tokenizer/pretraining uses target + auxiliary dialect varieties.",
                "Main 10-dialect benchmark can exclude KIS/TAN/NSD if needed.",
                "Validation/test rows from Vashantor are excluded from tokenizer training.",
                "Synthetic rows are train-only candidates and must be audited in Portion 2.",
            ],
        }

        manifest = {
            "portion": "tokenizer_p1_final_data_registry_loading_provenance_split_isolation",
            "seed": SEED,
            "config": asdict(CFG),
            "dataset_root": str(self.dataset_root),
            "synthetic_roots": [str(p) for p in self.synthetic_roots],
            "num_rows_total": int(len(df)),
            "num_sources": int(len(registry)),
            "required_schema": REQUIRED_SCHEMA,
            "extra_schema": EXTRA_SCHEMA,
            "test_results": test_results,
            "taxonomy": taxonomy,
            "source_reports": self.source_reports,
        }

        df.to_csv(OUT_DIR / "merged_raw_with_metadata.csv", index=False, encoding="utf-8")
        registry.to_csv(OUT_DIR / "data_registry.csv", index=False, encoding="utf-8")
        unknown.to_csv(OUT_DIR / "unknown_label_report.csv", index=False, encoding="utf-8")
        duplicate_exact_text.to_csv(OUT_DIR / "duplicate_exact_text_report.csv", index=False, encoding="utf-8")
        duplicate_exact_text_dialect.to_csv(OUT_DIR / "duplicate_exact_text_dialect_report.csv", index=False, encoding="utf-8")

        with open(OUT_DIR / "dialect_count_before_cleaning.json", "w", encoding="utf-8") as f:
            json.dump(dialect_count, f, ensure_ascii=False, indent=2)

        with open(OUT_DIR / "source_manifest.json", "w", encoding="utf-8") as f:
            json.dump(manifest, f, ensure_ascii=False, indent=2)

        with open(OUT_DIR / "dialect_taxonomy.json", "w", encoding="utf-8") as f:
            json.dump(taxonomy, f, ensure_ascii=False, indent=2)

        print("\n" + "=" * 100)
        print("PORTION 1 EXPORTS")
        print("=" * 100)
        print(f"Saved: {OUT_DIR / 'merged_raw_with_metadata.csv'}")
        print(f"Saved: {OUT_DIR / 'data_registry.csv'}")
        print(f"Saved: {OUT_DIR / 'source_manifest.json'}")
        print(f"Saved: {OUT_DIR / 'dialect_count_before_cleaning.json'}")
        print(f"Saved: {OUT_DIR / 'dialect_taxonomy.json'}")
        print(f"Saved: {OUT_DIR / 'unknown_label_report.csv'}")
        print(f"Saved: {OUT_DIR / 'duplicate_exact_text_report.csv'}")
        print(f"Saved: {OUT_DIR / 'duplicate_exact_text_dialect_report.csv'}")

    def checkpoint(self, df: pd.DataFrame) -> None:
        print("\n" + "=" * 100)
        print("CHECKPOINT: PORTION 1 DISTRIBUTION")
        print("=" * 100)

        table = (
            df.groupby(["dialect", "taxonomy_role", "is_synthetic", "eligible_for_tokenizer_train"], dropna=False)
            .size()
            .reset_index(name="count")
            .sort_values(["dialect", "taxonomy_role", "is_synthetic", "eligible_for_tokenizer_train"])
        )

        print(table.to_string(index=False))

        print("\nOverall dialect counts:")
        print(df["dialect"].value_counts().sort_index().to_string())

        print("\nTokenizer-train-eligible dialect counts:")
        print(
            df[df["eligible_for_tokenizer_train"] == True]["dialect"]
            .value_counts()
            .sort_index()
            .to_string()
        )

        print("\nSynthetic counts by source and dialect:")
        synth = df[df["is_synthetic"] == True]
        if len(synth):
            print(synth.groupby(["source", "dialect"]).size().to_string())
        else:
            print("No synthetic rows loaded.")

        print("\nSplit isolation:")
        print(df.groupby(["split_original", "eligible_for_tokenizer_train"]).size().to_string())

        print("\nDuplicate diagnostics:")
        print(f"Exact duplicate text rows       : {int(df.duplicated(subset=['text_hash']).sum()):,}")
        print(f"Exact duplicate text+dialect rows: {int(df.duplicated(subset=['text_hash', 'dialect']).sum()):,}")

        print("\n✅ TOKENIZER PORTION 1 COMPLETE")
        print(f"Total rows loaded: {len(df):,}")
        print(f"Output directory : {OUT_DIR}")


# ============================================================
# 6. Execute Portion 1
# ============================================================

builder = BanglaDialectRegistryBuilder(CFG)
df_p1 = builder.build()
test_results_p1 = builder.run_tests(df_p1)
builder.export(df_p1, test_results_p1)
builder.checkpoint(df_p1)

TOKENIZER PORTION 1 — FINAL: DATA REGISTRY, PROVENANCE, SPLIT ISOLATION
Input root     : /kaggle/input
Dataset root   : /kaggle/input/dataset-prime
Synthetic roots: ['/kaggle/input/datasets/nabidnur/synthetic-data']
Output dir     : artifacts/tokenizer_p1
[OK] BanglaDial: loaded=63,303, kept=63,303, text_col=Sentence, dialect_col=Language
[OK] Vashantor Barishal Train: loaded=1,875, kept=1,875, regional_col=bangla_speech , target_col=None
[OK] Vashantor Chittagong Train: loaded=1,875, kept=1,875, regional_col=bangla_speech , target_col=None
[OK] Vashantor Mymensingh Train: loaded=1,875, kept=1,875, regional_col=bangla_speech , target_col=None
[OK] Vashantor Noakhali Train: loaded=1,875, kept=1,875, regional_col=bangla_speech , target_col=None
[OK] Vashantor Sylhet Train: loaded=1,875, kept=1,875, regional_col=bangla_speech , target_col=None
[WARN] Missing Vashantor file: Barishal Validation Translation.csv
[OK] Vashantor Chittagong Validation: loaded=250, kept=250, regional_col=bangla_

In [2]:
from pathlib import Path
import pandas as pd
import json

P1_DIR = Path("artifacts/tokenizer_p1")
path = P1_DIR / "merged_raw_with_metadata.csv"

df = pd.read_csv(path)

# Preserve old synthetic source names.
df["source_original"] = df["source"]

# Make synthetic source explicit from filename.
mask_syn = df["is_synthetic"] == True
df.loc[mask_syn, "source"] = (
    "Synthetic_" +
    df.loc[mask_syn, "source_file"]
      .astype(str)
      .str.split("/")
      .str[-1]
      .str.replace(".csv", "", regex=False)
)

# Re-export fixed Portion 1 artifact.
fixed_path = P1_DIR / "merged_raw_with_metadata_fixed.csv"
df.to_csv(fixed_path, index=False, encoding="utf-8")

# Save synthetic provenance check.
syn_table = (
    df[df["is_synthetic"] == True]
    .groupby(["source", "source_original", "dialect"])
    .size()
    .reset_index(name="count")
    .sort_values(["source", "dialect", "source_original"])
)

syn_table.to_csv(P1_DIR / "synthetic_source_provenance_fixed.csv", index=False, encoding="utf-8")

print("Saved:", fixed_path)
print("Saved:", P1_DIR / "synthetic_source_provenance_fixed.csv")

print("\nSynthetic counts by fixed source:")
print(
    df[df["is_synthetic"] == True]
    .groupby(["source", "dialect"])
    .size()
    .to_string()
)

Saved: artifacts/tokenizer_p1/merged_raw_with_metadata_fixed.csv
Saved: artifacts/tokenizer_p1/synthetic_source_provenance_fixed.csv

Synthetic counts by fixed source:
source                                           dialect
Synthetic_bangla_dialect_balancing_synthetic_v0  BAR        1583
                                                 KHU        1166
                                                 KIS        1249
                                                 NSD        4138
                                                 RAJ        2109
                                                 RAN        1191
                                                 STD        5455
                                                 SYL        3003
                                                 TAN        3207
Synthetic_khulna_rajshahi_7000_each (1)          KHU        7000
                                                 RAJ        7000
Synthetic_mymensingh_noakhali_7000_each (1)      MYM        

## portion-2 ##

In [3]:
# ============================================================
# TOKENIZER PORTION 2 — FINAL VERSION
# Dialect-Preserving Cleaning, Artifact Removal,
# Duplicate Policy, Synthetic Quality Audit
# ============================================================

from __future__ import annotations

import os
import re
import json
import html
import math
import random
import string
import hashlib
import unicodedata
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


# ============================================================
# 0. Reproducibility
# ============================================================

SEED = 42

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)


# ============================================================
# 1. Config
# ============================================================

@dataclass(frozen=True)
class Portion2Config:
    input_dir: str = "artifacts/tokenizer_p1"
    output_dir: str = "artifacts/tokenizer_p2"

    # Prefer fixed source-provenance file if it exists.
    preferred_inputs: Tuple[str, ...] = (
        "merged_raw_with_metadata_fixed.csv",
        "merged_raw_with_metadata.csv",
    )

    min_chars_after_cleaning: int = 1
    min_bangla_chars: int = 1

    # For code-mix tolerance. Do not set too high.
    min_bangla_char_ratio_for_keep: float = 0.15
    max_latin_char_ratio_review: float = 0.35

    max_repeated_punctuation: int = 3
    max_same_char_run_review: int = 8

    # Rows below this are dropped from tokenizer training.
    min_quality_for_train: float = 0.25

    # Synthetic rows below this remain in full audit but are removed from tokenizer_train_clean.
    min_synthetic_quality_for_train: float = 0.35

    # Dedup policy.
    deduplicate_train_by_text_and_dialect: bool = True
    preserve_cross_dialect_duplicates: bool = True

    # Whether to keep validation/test in a separate clean holdout artifact.
    export_holdout_clean: bool = True

    # Drop bad rows from tokenizer training, but keep reports.
    fail_if_artifact_placeholder_remains: bool = True
    fail_if_train_duplicate_text_dialect_remains: bool = True

    # Plot outputs.
    save_plots: bool = True


CFG = Portion2Config()
P1_DIR = Path(CFG.input_dir)
P2_DIR = Path(CFG.output_dir)
P2_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. Taxonomy
# ============================================================

TARGET_DIALECTS_10 = [
    "BAR", "CHI", "KHU", "MYM", "NAR",
    "NOA", "RAJ", "RAN", "STD", "SYL",
]

AUXILIARY_DIALECTS = ["KIS", "TAN", "NSD"]

ALL_DIALECTS_FOR_TOKENIZER = sorted(set(TARGET_DIALECTS_10 + AUXILIARY_DIALECTS))


# ============================================================
# 3. Input Resolver
# ============================================================

def resolve_input_path() -> Path:
    for fname in CFG.preferred_inputs:
        p = P1_DIR / fname
        if p.exists():
            return p
    raise FileNotFoundError(
        f"No Portion 1 input found in {P1_DIR}. Tried: {CFG.preferred_inputs}"
    )


INPUT_PATH = resolve_input_path()

print("=" * 100)
print("TOKENIZER PORTION 2 — CLEANING, DEDUPLICATION, SYNTHETIC QUALITY AUDIT")
print("=" * 100)
print(f"Input file : {INPUT_PATH}")
print(f"Output dir : {P2_DIR}")
print("GPU needed : NO — CPU is sufficient for this portion.")
print("=" * 100)


# ============================================================
# 4. Unicode / Text Utility Functions
# ============================================================

BENGALI_START = "\u0980"
BENGALI_END = "\u09FF"

BENGALI_PUNCT = "।॥"
ASCII_PUNCT = string.punctuation
COMMON_PUNCT = ASCII_PUNCT + BENGALI_PUNCT + "“”‘’—–…"
ARTIFACT_PATTERNS = [
    r"<>",
    r"&lt;&gt;",
    r"\[\s*\]",
    r"\{\s*\}",
]


def is_bangla_char(ch: str) -> bool:
    return "\u0980" <= ch <= "\u09FF"


def is_latin_char(ch: str) -> bool:
    return ("A" <= ch <= "Z") or ("a" <= ch <= "z")


def is_punctuation_or_symbol(ch: str) -> bool:
    cat = unicodedata.category(ch)
    return cat.startswith("P") or cat.startswith("S")


def stable_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def normalize_unicode_nfc(text: str) -> str:
    return unicodedata.normalize("NFC", text)


def remove_control_and_invisible(text: str) -> str:
    """
    Remove control/invisible formatting artifacts.
    This intentionally removes zero-width variants because we do not want
    tokenizer statistics fragmented by invisible code points.
    """
    # C0/C1 controls
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", "", text)

    # Zero-width and BOM-like artifacts
    text = re.sub(r"[\u200b\u200c\u200d\u200e\u200f\ufeff]", "", text)

    return text


def has_artifact_placeholder(text: str) -> bool:
    if pd.isna(text):
        return False
    s = str(text)
    return bool(re.search(r"<>|&lt;&gt;", s))


def clean_text_dialect_preserving(text: Any) -> Optional[str]:
    """
    Dialect-preserving cleaning.
    Keep:
      - dialect spellings
      - regional morphology
      - Bangla phonological variation
      - code-mixed terms if mostly Bangla
    Remove:
      - control chars
      - zero-width artifacts
      - HTML/XML fragments
      - placeholder <>
      - repeated whitespace
      - extreme punctuation spam
    """
    if pd.isna(text):
        return None

    s = str(text)

    # HTML entity decode first.
    s = html.unescape(s)

    # Unicode normalization.
    s = normalize_unicode_nfc(s)

    # Remove control/invisible chars.
    s = remove_control_and_invisible(s)

    # Remove placeholder artifacts.
    for pat in ARTIFACT_PATTERNS:
        s = re.sub(pat, " ", s)

    # Remove HTML/XML-like tags, but only tag-shaped Latin/meta fragments.
    s = re.sub(r"</?[A-Za-z][A-Za-z0-9_\-:]*[^>]*>", " ", s)

    # Remove stray angle brackets left after placeholder removal.
    s = s.replace("<", " ").replace(">", " ")

    # Collapse repeated punctuation beyond configured cap.
    cap = CFG.max_repeated_punctuation
    s = re.sub(r"([!?।॥.,;:])\1{" + str(cap) + r",}", lambda m: m.group(1) * cap, s)

    # Normalize spacing around punctuation.
    s = re.sub(r"\s+([।॥!?.,;:])", r"\1", s)
    s = re.sub(r"([।॥!?.,;:])([^\s।॥!?.,;:])", r"\1 \2", s)

    # Normalize whitespace.
    s = re.sub(r"\s+", " ", s).strip()

    if len(s) < CFG.min_chars_after_cleaning:
        return None

    return s


def loose_normalize_for_near_duplicate(text: str) -> str:
    """
    Fast near-exact normalization:
      - remove punctuation/symbols
      - remove whitespace
      - normalize all digits to 0
      - preserve Bangla and Latin characters
    This is not semantic deduplication; it catches punctuation/spacing variants.
    """
    text = normalize_unicode_nfc(text)
    text = remove_control_and_invisible(text)

    chars = []
    for ch in text:
        if ch.isspace():
            continue
        if ch.isdigit():
            chars.append("0")
            continue
        if is_punctuation_or_symbol(ch):
            continue
        chars.append(ch)

    return "".join(chars)


def max_same_char_run(text: str) -> int:
    if not text:
        return 0

    best = 1
    cur = 1
    prev = text[0]

    for ch in text[1:]:
        if ch == prev:
            cur += 1
            best = max(best, cur)
        else:
            prev = ch
            cur = 1

    return best


def token_repetition_score(text: str, n: int = 3) -> float:
    tokens = text.split()
    if len(tokens) < n * 2:
        return 0.0

    grams = [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
    if not grams:
        return 0.0

    return 1.0 - (len(set(grams)) / len(grams))


def char_stats(text: Optional[str]) -> Dict[str, Any]:
    if text is None or pd.isna(text):
        return {
            "char_len": 0,
            "nonspace_len": 0,
            "bangla_chars": 0,
            "latin_chars": 0,
            "digit_chars": 0,
            "punct_symbol_chars": 0,
            "bangla_char_ratio": 0.0,
            "latin_char_ratio": 0.0,
            "digit_char_ratio": 0.0,
            "punct_symbol_ratio": 0.0,
            "token_count_ws": 0,
            "unique_token_ratio": 0.0,
            "max_same_char_run": 0,
            "token_trigram_repetition": 0.0,
        }

    s = str(text)
    nonspace = [ch for ch in s if not ch.isspace()]
    denom = max(1, len(nonspace))

    bangla = sum(is_bangla_char(ch) for ch in nonspace)
    latin = sum(is_latin_char(ch) for ch in nonspace)
    digit = sum(ch.isdigit() for ch in nonspace)
    punct_symbol = sum(is_punctuation_or_symbol(ch) for ch in nonspace)

    tokens = s.split()
    unique_token_ratio = len(set(tokens)) / max(1, len(tokens))

    return {
        "char_len": len(s),
        "nonspace_len": len(nonspace),
        "bangla_chars": bangla,
        "latin_chars": latin,
        "digit_chars": digit,
        "punct_symbol_chars": punct_symbol,
        "bangla_char_ratio": bangla / denom,
        "latin_char_ratio": latin / denom,
        "digit_char_ratio": digit / denom,
        "punct_symbol_ratio": punct_symbol / denom,
        "token_count_ws": len(tokens),
        "unique_token_ratio": unique_token_ratio,
        "max_same_char_run": max_same_char_run(s),
        "token_trigram_repetition": token_repetition_score(s, n=3),
    }


def safe_bool(x: Any) -> bool:
    if isinstance(x, bool):
        return x
    if pd.isna(x):
        return False
    return str(x).strip().lower() in {"true", "1", "yes", "y"}


# ============================================================
# 5. Quality Scoring
# ============================================================

def compute_quality_score(row: pd.Series) -> Tuple[float, List[str], List[str]]:
    """
    Returns:
      quality_score_cleaned
      flags
      removal_reasons

    The scoring is conservative. It does not punish dialect spellings.
    It punishes artifacts, empty text, no Bangla content, excessive Latin,
    extreme repetition, and very low signal.
    """
    flags = []
    removal = []

    score = 1.0

    text = row["clean_text"]
    raw_text = row["text_raw"]

    if text is None or pd.isna(text) or len(str(text).strip()) == 0:
        removal.append("empty_after_cleaning")
        return 0.0, flags, removal

    bangla_chars = int(row["bangla_chars"])
    bangla_ratio = float(row["bangla_char_ratio"])
    latin_ratio = float(row["latin_char_ratio"])
    token_count = int(row["token_count_ws"])
    char_len = int(row["char_len"])
    max_run = int(row["max_same_char_run"])
    rep_score = float(row["token_trigram_repetition"])
    punct_ratio = float(row["punct_symbol_ratio"])

    if bangla_chars < CFG.min_bangla_chars:
        removal.append("no_bangla_characters")
        score -= 0.70

    if bangla_ratio < CFG.min_bangla_char_ratio_for_keep:
        flags.append("low_bangla_ratio")
        score -= 0.35

    if latin_ratio > CFG.max_latin_char_ratio_review:
        flags.append("high_latin_ratio")
        score -= 0.20

    if has_artifact_placeholder(raw_text):
        flags.append("artifact_placeholder_removed")
        score -= 0.10

    if char_len <= 1:
        flags.append("very_short")
        score -= 0.25
    elif char_len <= 3:
        flags.append("short_utterance")
        score -= 0.08

    if token_count == 1 and char_len <= 4:
        flags.append("single_short_token")
        score -= 0.10

    if max_run >= CFG.max_same_char_run_review:
        flags.append("excessive_same_char_run")
        score -= 0.20

    if rep_score >= 0.30:
        flags.append("high_token_repetition")
        score -= 0.25

    if punct_ratio > 0.60:
        flags.append("punctuation_symbol_dominant")
        score -= 0.40

    # Synthetic rows get an extra conservatism penalty if formulaic/low signal.
    if safe_bool(row["is_synthetic"]):
        original_q = row.get("quality_score", np.nan)
        if not pd.isna(original_q):
            try:
                original_q = float(original_q)
                score = 0.70 * score + 0.30 * original_q
            except Exception:
                pass

        if token_count <= 2:
            flags.append("synthetic_too_short")
            score -= 0.10

    score = max(0.0, min(1.0, score))

    if score < CFG.min_quality_for_train:
        removal.append("quality_below_minimum")

    if safe_bool(row["is_synthetic"]) and score < CFG.min_synthetic_quality_for_train:
        removal.append("synthetic_quality_below_minimum")

    return score, flags, removal


# ============================================================
# 6. Source Provenance Fix for Synthetic Rows
# ============================================================

def standardize_synthetic_provenance(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "source_original" not in df.columns:
        df["source_original"] = df["source"]

    syn_mask = df["is_synthetic"].map(safe_bool)

    # Force synthetic source to be file-based for clean provenance.
    synthetic_file_stem = (
        df.loc[syn_mask, "source_file"]
        .astype(str)
        .str.split("/")
        .str[-1]
        .str.replace(".csv", "", regex=False)
    )

    df.loc[syn_mask, "source"] = "Synthetic_" + synthetic_file_stem

    return df


# ============================================================
# 7. Main Cleaner Class
# ============================================================

class TokenizerPortion2Cleaner:
    def __init__(self, cfg: Portion2Config, input_path: Path):
        self.cfg = cfg
        self.input_path = input_path
        self.output_dir = Path(cfg.output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def load(self) -> pd.DataFrame:
        df = pd.read_csv(self.input_path)

        # Normalize bool columns.
        for col in [
            "is_synthetic",
            "eligible_for_tokenizer_train",
            "eligible_for_dialect_routing",
            "eligible_for_main_10dialect_benchmark",
        ]:
            if col in df.columns:
                df[col] = df[col].map(safe_bool)

        df = standardize_synthetic_provenance(df)

        print(f"[LOAD] Rows loaded from Portion 1: {len(df):,}")
        print(f"[LOAD] Dialects: {sorted(df['dialect'].unique().tolist())}")

        return df

    def clean_and_score(self, df: pd.DataFrame) -> pd.DataFrame:
        print("\n" + "=" * 100)
        print("STEP 1: DIALECT-PRESERVING CLEANING + QUALITY SCORING")
        print("=" * 100)

        df = df.copy()
        df["text_raw"] = df["text"]

        df["raw_has_placeholder_artifact"] = df["text_raw"].map(has_artifact_placeholder)
        df["clean_text"] = df["text_raw"].map(clean_text_dialect_preserving)

        # Hashes after cleaning.
        df["clean_text_hash"] = df["clean_text"].fillna("").map(stable_hash)
        df["loose_text_norm"] = df["clean_text"].fillna("").map(loose_normalize_for_near_duplicate)
        df["loose_text_hash"] = df["loose_text_norm"].map(stable_hash)
        df["near_duplicate_group_p2"] = "loose:" + df["loose_text_hash"].str[:16]

        # Character/token statistics.
        stats = df["clean_text"].map(char_stats)
        stats_df = pd.DataFrame(stats.tolist())
        df = pd.concat([df.reset_index(drop=True), stats_df.reset_index(drop=True)], axis=1)

        # Quality scoring.
        quality_results = df.apply(compute_quality_score, axis=1)
        df["quality_score_cleaned"] = [x[0] for x in quality_results]
        df["quality_flags"] = [";".join(x[1]) for x in quality_results]
        df["removal_reasons"] = [";".join(x[2]) for x in quality_results]

        df["drop_from_clean_corpus"] = df["removal_reasons"].str.len() > 0

        # Final text column for downstream portions.
        df["text"] = df["clean_text"]

        print(f"[CLEAN] Placeholder artifact rows before cleaning: {int(df['raw_has_placeholder_artifact'].sum()):,}")
        print(f"[CLEAN] Empty/drop rows after cleaning/scoring     : {int(df['drop_from_clean_corpus'].sum()):,}")
        print(f"[CLEAN] Mean cleaned quality score               : {df['quality_score_cleaned'].mean():.4f}")

        return df

    def build_train_and_holdout(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        print("\n" + "=" * 100)
        print("STEP 2: SOURCE-PRIORITY DEDUPLICATION")
        print("=" * 100)

        kept = df[df["drop_from_clean_corpus"] == False].copy()
        removed = df[df["drop_from_clean_corpus"] == True].copy()

        train = kept[kept["eligible_for_tokenizer_train"] == True].copy()

        if self.cfg.export_holdout_clean:
            holdout = kept[kept["split_original"].isin(["validation", "test"])].copy()
        else:
            holdout = pd.DataFrame(columns=kept.columns)

        # Source priority:
        #   0: real train
        #   1: real unspecified
        #   2: synthetic non-balancing
        #   3: synthetic balancing v0
        # Higher quality preferred inside same priority.
        def source_priority(row: pd.Series) -> int:
            is_syn = safe_bool(row["is_synthetic"])
            split = str(row["split_original"]).lower()
            source_file = str(row["source_file"]).lower()

            if not is_syn and split == "train":
                return 0
            if not is_syn and split == "unspecified":
                return 1
            if is_syn and "bangla_dialect_balancing_synthetic_v0" not in source_file:
                return 2
            if is_syn and "bangla_dialect_balancing_synthetic_v0" in source_file:
                return 3
            return 4

        train["_source_priority"] = train.apply(source_priority, axis=1)
        train["_quality_sort"] = -train["quality_score_cleaned"].astype(float)
        train["_char_len_sort"] = -train["char_len"].astype(int)

        before_train = len(train)

        if self.cfg.deduplicate_train_by_text_and_dialect:
            duplicate_mask = train.duplicated(subset=["clean_text_hash", "dialect"], keep=False)
            duplicate_candidates = train[duplicate_mask].copy()

            train = (
                train.sort_values(
                    by=[
                        "clean_text_hash",
                        "dialect",
                        "_source_priority",
                        "_quality_sort",
                        "_char_len_sort",
                        "source_file",
                        "source_row_id",
                    ],
                    kind="mergesort",
                )
                .drop_duplicates(subset=["clean_text_hash", "dialect"], keep="first")
                .reset_index(drop=True)
            )

            kept_keys = set(zip(train["clean_text_hash"], train["dialect"], train["source_file"], train["source_row_id"]))

            if len(duplicate_candidates):
                duplicate_candidates["kept_after_dedup"] = duplicate_candidates.apply(
                    lambda r: (r["clean_text_hash"], r["dialect"], r["source_file"], r["source_row_id"]) in kept_keys,
                    axis=1,
                )
            else:
                duplicate_candidates["kept_after_dedup"] = []

        else:
            duplicate_candidates = pd.DataFrame(columns=train.columns.tolist() + ["kept_after_dedup"])

        after_train = len(train)

        # Remove helper columns from final train.
        for col in ["_source_priority", "_quality_sort", "_char_len_sort"]:
            if col in train.columns:
                train = train.drop(columns=[col])

        print(f"[DEDUP] Train candidates before dedup : {before_train:,}")
        print(f"[DEDUP] Train rows after dedup        : {after_train:,}")
        print(f"[DEDUP] Removed by text+dialect dedup : {before_train - after_train:,}")

        self.duplicate_resolution_report = duplicate_candidates

        return df, train, holdout

    def export_reports(self, df_all: pd.DataFrame, train: pd.DataFrame, holdout: pd.DataFrame) -> None:
        print("\n" + "=" * 100)
        print("STEP 3: EXPORT REPORTS")
        print("=" * 100)

        # Core exports.
        all_path = self.output_dir / "cleaned_all_with_quality.csv"
        train_path = self.output_dir / "tokenizer_train_clean.csv"
        holdout_path = self.output_dir / "tokenizer_eval_holdout_clean.csv"

        removed_path = self.output_dir / "removed_rows_report.csv"
        dup_path = self.output_dir / "duplicate_resolution_report.csv"
        synthetic_quality_path = self.output_dir / "synthetic_quality_report.csv"
        distribution_path = self.output_dir / "dialect_distribution_after_cleaning.csv"
        artifact_report_path = self.output_dir / "artifact_report.csv"
        source_distribution_path = self.output_dir / "source_distribution_after_cleaning.csv"

        df_all.to_csv(all_path, index=False, encoding="utf-8")
        train.to_csv(train_path, index=False, encoding="utf-8")
        holdout.to_csv(holdout_path, index=False, encoding="utf-8")

        removed = df_all[df_all["drop_from_clean_corpus"] == True].copy()
        removed.to_csv(removed_path, index=False, encoding="utf-8")

        if hasattr(self, "duplicate_resolution_report"):
            self.duplicate_resolution_report.to_csv(dup_path, index=False, encoding="utf-8")
        else:
            pd.DataFrame().to_csv(dup_path, index=False, encoding="utf-8")

        synthetic_quality = df_all[df_all["is_synthetic"] == True].copy()
        synthetic_quality.to_csv(synthetic_quality_path, index=False, encoding="utf-8")

        dist = (
            train.groupby(["dialect", "taxonomy_role", "is_synthetic"], dropna=False)
            .size()
            .reset_index(name="count")
            .sort_values(["dialect", "taxonomy_role", "is_synthetic"])
        )
        dist.to_csv(distribution_path, index=False, encoding="utf-8")

        source_dist = (
            train.groupby(["source", "dialect", "is_synthetic"], dropna=False)
            .size()
            .reset_index(name="count")
            .sort_values(["source", "dialect", "is_synthetic"])
        )
        source_dist.to_csv(source_distribution_path, index=False, encoding="utf-8")

        artifact_report = (
            df_all.groupby(["dialect", "is_synthetic"], dropna=False)
            .agg(
                rows=("text_raw", "count"),
                raw_placeholder_artifacts=("raw_has_placeholder_artifact", "sum"),
                dropped_rows=("drop_from_clean_corpus", "sum"),
                mean_quality=("quality_score_cleaned", "mean"),
                mean_bangla_ratio=("bangla_char_ratio", "mean"),
                mean_latin_ratio=("latin_char_ratio", "mean"),
                mean_token_count=("token_count_ws", "mean"),
            )
            .reset_index()
            .sort_values(["dialect", "is_synthetic"])
        )
        artifact_report.to_csv(artifact_report_path, index=False, encoding="utf-8")

        # Summary JSON.
        summary = {
            "portion": "tokenizer_p2_cleaning_dedup_synthetic_quality",
            "seed": SEED,
            "config": asdict(CFG),
            "input_path": str(INPUT_PATH),
            "rows_input": int(len(df_all)),
            "rows_dropped_cleaning": int(df_all["drop_from_clean_corpus"].sum()),
            "rows_train_clean": int(len(train)),
            "rows_holdout_clean": int(len(holdout)),
            "artifact_placeholder_rows_raw": int(df_all["raw_has_placeholder_artifact"].sum()),
            "exact_duplicate_text_rows_input_clean_hash": int(df_all.duplicated(subset=["clean_text_hash"]).sum()),
            "exact_duplicate_text_dialect_rows_train": int(train.duplicated(subset=["clean_text_hash", "dialect"]).sum()),
            "train_dialect_counts": train["dialect"].value_counts().sort_index().to_dict(),
            "train_real_counts": train[train["is_synthetic"] == False]["dialect"].value_counts().sort_index().to_dict(),
            "train_synthetic_counts": train[train["is_synthetic"] == True]["dialect"].value_counts().sort_index().to_dict(),
            "mean_quality_by_dialect": (
                train.groupby("dialect")["quality_score_cleaned"].mean().sort_index().to_dict()
            ),
            "outputs": {
                "cleaned_all_with_quality": str(all_path),
                "tokenizer_train_clean": str(train_path),
                "tokenizer_eval_holdout_clean": str(holdout_path),
                "removed_rows_report": str(removed_path),
                "duplicate_resolution_report": str(dup_path),
                "synthetic_quality_report": str(synthetic_quality_path),
                "dialect_distribution_after_cleaning": str(distribution_path),
                "artifact_report": str(artifact_report_path),
                "source_distribution_after_cleaning": str(source_distribution_path),
            },
        }

        with open(self.output_dir / "cleaning_summary.json", "w", encoding="utf-8") as f:
            json.dump(summary, f, ensure_ascii=False, indent=2)

        print(f"Saved: {all_path}")
        print(f"Saved: {train_path}")
        print(f"Saved: {holdout_path}")
        print(f"Saved: {removed_path}")
        print(f"Saved: {dup_path}")
        print(f"Saved: {synthetic_quality_path}")
        print(f"Saved: {distribution_path}")
        print(f"Saved: {artifact_report_path}")
        print(f"Saved: {source_distribution_path}")
        print(f"Saved: {self.output_dir / 'cleaning_summary.json'}")

    def save_plots(self, train: pd.DataFrame, df_all: pd.DataFrame) -> None:
        if not self.cfg.save_plots:
            return

        print("\n" + "=" * 100)
        print("STEP 4: SAVE DIAGNOSTIC PLOTS")
        print("=" * 100)

        # Plot 1: train dialect counts.
        counts = train["dialect"].value_counts().sort_index()

        plt.figure(figsize=(14, 6))
        plt.bar(counts.index, counts.values)
        plt.title("Tokenizer Train Clean Rows per Dialect")
        plt.xlabel("Dialect")
        plt.ylabel("Rows")
        plt.xticks(rotation=45)
        plt.tight_layout()
        p = self.output_dir / "dialect_distribution_after_cleaning.png"
        plt.savefig(p, dpi=200)
        plt.close()
        print(f"Saved: {p}")

        # Plot 2: real vs synthetic stacked-style separate bars.
        rs = (
            train.groupby(["dialect", "is_synthetic"])
            .size()
            .reset_index(name="count")
        )

        dialects = sorted(train["dialect"].unique())
        real_counts = []
        syn_counts = []

        for d in dialects:
            sub = rs[rs["dialect"] == d]
            real_counts.append(int(sub[sub["is_synthetic"] == False]["count"].sum()))
            syn_counts.append(int(sub[sub["is_synthetic"] == True]["count"].sum()))

        x = np.arange(len(dialects))

        plt.figure(figsize=(14, 6))
        plt.bar(x, real_counts, label="real")
        plt.bar(x, syn_counts, bottom=real_counts, label="synthetic")
        plt.title("Real vs Synthetic Composition after Cleaning")
        plt.xlabel("Dialect")
        plt.ylabel("Rows")
        plt.xticks(x, dialects, rotation=45)
        plt.legend()
        plt.tight_layout()
        p = self.output_dir / "real_synthetic_distribution_after_cleaning.png"
        plt.savefig(p, dpi=200)
        plt.close()
        print(f"Saved: {p}")

        # Plot 3: quality by dialect.
        q = train.groupby("dialect")["quality_score_cleaned"].mean().sort_index()

        plt.figure(figsize=(14, 6))
        plt.bar(q.index, q.values)
        plt.title("Mean Cleaned Quality Score by Dialect")
        plt.xlabel("Dialect")
        plt.ylabel("Mean quality score")
        plt.ylim(0, 1.05)
        plt.xticks(rotation=45)
        plt.tight_layout()
        p = self.output_dir / "mean_quality_by_dialect.png"
        plt.savefig(p, dpi=200)
        plt.close()
        print(f"Saved: {p}")

    def run_tests(self, df_all: pd.DataFrame, train: pd.DataFrame, holdout: pd.DataFrame) -> Dict[str, Any]:
        print("\n" + "=" * 100)
        print("PORTION 2 TESTS")
        print("=" * 100)

        failures = []

        # Test 1: no null train text.
        null_train_text = int(train["text"].isna().sum() + (train["text"].astype(str).str.len() == 0).sum())
        if null_train_text > 0:
            failures.append(f"Null/empty train text rows: {null_train_text}")

        # Test 2: no UNK.
        unk_train = int((train["dialect"] == "UNK").sum())
        if unk_train > 0:
            failures.append(f"UNK rows remain in train: {unk_train}")

        # Test 3: no validation/test in tokenizer train.
        heldout_train = int(train["split_original"].isin(["validation", "test"]).sum())
        if heldout_train > 0:
            failures.append(f"Validation/test rows leaked into tokenizer train: {heldout_train}")

        # Test 4: placeholder artifact removed from final train text.
        artifact_remaining = int(train["text"].astype(str).str.contains(r"<>|&lt;&gt;", regex=True).sum())
        if self.cfg.fail_if_artifact_placeholder_remains and artifact_remaining > 0:
            failures.append(f"Placeholder artifact remains in train text: {artifact_remaining}")

        # Test 5: no exact train duplicate by clean_text_hash+dialect.
        dup_train_text_dialect = int(train.duplicated(subset=["clean_text_hash", "dialect"]).sum())
        if self.cfg.fail_if_train_duplicate_text_dialect_remains and dup_train_text_dialect > 0:
            failures.append(f"Duplicate clean_text_hash+dialect remains in train: {dup_train_text_dialect}")

        # Test 6: each dialect has at least one row.
        missing_dialects = sorted(set(ALL_DIALECTS_FOR_TOKENIZER) - set(train["dialect"].unique()))
        if missing_dialects:
            failures.append(f"Missing dialects in train: {missing_dialects}")

        # Test 7: synthetic rows only synthetic split/source file.
        syn = train[train["is_synthetic"] == True]
        bad_syn_split = int((syn["split_original"] != "synthetic_train").sum())
        if bad_syn_split > 0:
            failures.append(f"Synthetic rows with non-synthetic_train split: {bad_syn_split}")

        results = {
            "train_rows": int(len(train)),
            "all_rows": int(len(df_all)),
            "holdout_rows": int(len(holdout)),
            "null_train_text": null_train_text,
            "unk_train": unk_train,
            "heldout_rows_in_train": heldout_train,
            "artifact_placeholder_remaining_train": artifact_remaining,
            "duplicate_clean_text_hash_dialect_train": dup_train_text_dialect,
            "missing_dialects": missing_dialects,
            "synthetic_bad_split_rows": bad_syn_split,
            "hard_failures": failures,
            "passed": len(failures) == 0,
        }

        for k, v in results.items():
            if k != "hard_failures":
                print(f"{k:60s}: {v}")

        if failures:
            print("\n[FAILED]")
            for f in failures:
                print(f"  - {f}")
            raise AssertionError("Tokenizer Portion 2 tests failed.")

        print("\n[OK] Portion 2 tests passed.")
        return results

    def checkpoint(self, train: pd.DataFrame, df_all: pd.DataFrame) -> None:
        print("\n" + "=" * 100)
        print("CHECKPOINT: PORTION 2 CLEAN CORPUS")
        print("=" * 100)

        print("\nTrain-clean dialect counts:")
        print(train["dialect"].value_counts().sort_index().to_string())

        print("\nTrain-clean real/synthetic by dialect:")
        print(
            train.groupby(["dialect", "is_synthetic"])
            .size()
            .reset_index(name="count")
            .sort_values(["dialect", "is_synthetic"])
            .to_string(index=False)
        )

        print("\nMean quality by dialect:")
        print(train.groupby("dialect")["quality_score_cleaned"].mean().sort_index().round(4).to_string())

        print("\nRows dropped by reason:")
        dropped = df_all[df_all["drop_from_clean_corpus"] == True].copy()
        if len(dropped) == 0:
            print("No rows dropped by quality/removal rules.")
        else:
            print(dropped["removal_reasons"].value_counts().head(30).to_string())

        print("\nArtifact placeholder rows in raw input:")
        print(int(df_all["raw_has_placeholder_artifact"].sum()))

        print("\nExact duplicate diagnostics after cleaning:")
        print(f"All rows duplicate clean_text_hash               : {int(df_all.duplicated(subset=['clean_text_hash']).sum()):,}")
        print(f"Train duplicate clean_text_hash+dialect          : {int(train.duplicated(subset=['clean_text_hash', 'dialect']).sum()):,}")
        print(f"Train duplicate loose_text_hash+dialect          : {int(train.duplicated(subset=['loose_text_hash', 'dialect']).sum()):,}")

        print("\n✅ TOKENIZER PORTION 2 COMPLETE")
        print(f"Output directory: {P2_DIR}")
        print("Next: Tokenizer Portion 3 — dialect-aware token mining and protected vocabulary construction.")


# ============================================================
# 8. Execute Portion 2
# ============================================================

cleaner = TokenizerPortion2Cleaner(CFG, INPUT_PATH)

df_p2_raw = cleaner.load()
df_p2_all = cleaner.clean_and_score(df_p2_raw)
df_p2_all, df_tokenizer_train_clean, df_tokenizer_holdout_clean = cleaner.build_train_and_holdout(df_p2_all)

test_results_p2 = cleaner.run_tests(
    df_all=df_p2_all,
    train=df_tokenizer_train_clean,
    holdout=df_tokenizer_holdout_clean,
)

cleaner.export_reports(
    df_all=df_p2_all,
    train=df_tokenizer_train_clean,
    holdout=df_tokenizer_holdout_clean,
)

cleaner.save_plots(
    train=df_tokenizer_train_clean,
    df_all=df_p2_all,
)

cleaner.checkpoint(
    train=df_tokenizer_train_clean,
    df_all=df_p2_all,
)

TOKENIZER PORTION 2 — CLEANING, DEDUPLICATION, SYNTHETIC QUALITY AUDIT
Input file : artifacts/tokenizer_p1/merged_raw_with_metadata_fixed.csv
Output dir : artifacts/tokenizer_p2
GPU needed : NO — CPU is sufficient for this portion.
[LOAD] Rows loaded from Portion 1: 141,171
[LOAD] Dialects: ['BAR', 'CHI', 'KHU', 'KIS', 'MYM', 'NAR', 'NOA', 'NSD', 'RAJ', 'RAN', 'STD', 'SYL', 'TAN']

STEP 1: DIALECT-PRESERVING CLEANING + QUALITY SCORING
[CLEAN] Placeholder artifact rows before cleaning: 3,872
[CLEAN] Empty/drop rows after cleaning/scoring     : 25
[CLEAN] Mean cleaned quality score               : 0.9776

STEP 2: SOURCE-PRIORITY DEDUPLICATION
[DEDUP] Train candidates before dedup : 138,271
[DEDUP] Train rows after dedup        : 123,833
[DEDUP] Removed by text+dialect dedup : 14,438

PORTION 2 TESTS
train_rows                                                  : 123833
all_rows                                                    : 141171
holdout_rows                                         

In [4]:
# ============================================================
# TOKENIZER PORTION 2.1 — FINAL VERSION
# Loose Duplicate Audit + Conservative Synthetic Thinning
# ============================================================
#
# Purpose:
#   - Audit near/loose duplicates after Portion 2.
#   - Preserve real rows aggressively.
#   - Thin only high-confidence synthetic near-duplicates.
#   - Export both:
#       1. audit-only original train file copy
#       2. recommended p2.1 thinned train file
#
# Input:
#   artifacts/tokenizer_p2/tokenizer_train_clean.csv
#   artifacts/tokenizer_p2/cleaned_all_with_quality.csv
#
# Output:
#   artifacts/tokenizer_p21/tokenizer_train_clean_p21_audit_only.csv
#   artifacts/tokenizer_p21/tokenizer_train_clean_p21_recommended.csv
#   artifacts/tokenizer_p21/loose_duplicate_group_report.csv
#   artifacts/tokenizer_p21/loose_duplicate_members_report.csv
#   artifacts/tokenizer_p21/thinning_decision_report.csv
#   artifacts/tokenizer_p21/dialect_distribution_p21_recommended.csv
#   artifacts/tokenizer_p21/p21_summary.json
# ============================================================

from __future__ import annotations

import os
import re
import json
import random
import hashlib
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


# ============================================================
# 0. Reproducibility
# ============================================================

SEED = 42

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)


# ============================================================
# 1. Config
# ============================================================

@dataclass(frozen=True)
class Portion21Config:
    input_dir: str = "artifacts/tokenizer_p2"
    output_dir: str = "artifacts/tokenizer_p21"

    train_file: str = "tokenizer_train_clean.csv"
    all_quality_file: str = "cleaned_all_with_quality.csv"

    # Key used for loose duplicates.
    loose_key_cols: Tuple[str, ...] = ("dialect", "loose_text_hash")

    # Conservative thinning policy.
    # Real rows are always kept.
    keep_all_real_rows: bool = True

    # If a loose duplicate group contains real + synthetic rows:
    # keep all real rows and at most this many synthetic rows.
    max_synthetic_rows_when_real_exists: int = 1

    # If a loose duplicate group contains only synthetic rows:
    # keep at most this many synthetic rows.
    max_synthetic_rows_when_synthetic_only: int = 2

    # Groups smaller than this are only audited, not thinned, unless synthetic-only.
    min_group_size_for_thinning: int = 2

    # Do not thin groups where all rows are real.
    thin_real_only_groups: bool = False

    # Ranking fields for choosing synthetic rows to keep.
    # Higher quality and longer text are preferred.
    quality_col: str = "quality_score_cleaned"

    # Safety threshold after thinning.
    min_rows_per_dialect_after_thinning: int = 7500

    # Plot outputs.
    save_plots: bool = True


CFG = Portion21Config()

P2_DIR = Path(CFG.input_dir)
P21_DIR = Path(CFG.output_dir)
P21_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = P2_DIR / CFG.train_file
ALL_QUALITY_PATH = P2_DIR / CFG.all_quality_file

print("=" * 100)
print("TOKENIZER PORTION 2.1 — LOOSE DUPLICATE AUDIT + CONSERVATIVE SYNTHETIC THINNING")
print("=" * 100)
print(f"Train input      : {TRAIN_PATH}")
print(f"All-quality input: {ALL_QUALITY_PATH}")
print(f"Output dir       : {P21_DIR}")
print("GPU needed       : NO — CPU is sufficient for this portion.")
print("=" * 100)

assert TRAIN_PATH.exists(), f"Missing train input: {TRAIN_PATH}"


# ============================================================
# 2. Utility Functions
# ============================================================

def safe_bool(x: Any) -> bool:
    if isinstance(x, bool):
        return x
    if pd.isna(x):
        return False
    return str(x).strip().lower() in {"true", "1", "yes", "y"}


def stable_hash(text: str) -> str:
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()


def source_priority(row: pd.Series) -> int:
    """
    Lower is better.
    Priority:
      0 = real Vashantor train / real labeled training split
      1 = real unspecified
      2 = synthetic older/large synthetic files
      3 = synthetic balancing v0
      4 = other
    """
    is_syn = safe_bool(row.get("is_synthetic", False))
    split = str(row.get("split_original", "")).lower()
    source_file = str(row.get("source_file", "")).lower()

    if not is_syn and split == "train":
        return 0
    if not is_syn and split == "unspecified":
        return 1
    if is_syn and "bangla_dialect_balancing_synthetic_v0" not in source_file:
        return 2
    if is_syn and "bangla_dialect_balancing_synthetic_v0" in source_file:
        return 3
    return 4


def compact_text_preview(text: Any, n: int = 90) -> str:
    s = "" if pd.isna(text) else str(text)
    s = re.sub(r"\s+", " ", s).strip()
    if len(s) <= n:
        return s
    return s[:n] + "..."


def ensure_required_cols(df: pd.DataFrame) -> None:
    required = [
        "text",
        "dialect",
        "is_synthetic",
        "source",
        "source_file",
        "split_original",
        "clean_text_hash",
        "loose_text_hash",
        "quality_score_cleaned",
        "char_len",
        "token_count_ws",
    ]
    missing = [c for c in required if c not in df.columns]
    assert not missing, f"Missing required columns in train file: {missing}"


# ============================================================
# 3. Main Portion 2.1 Class
# ============================================================

class LooseDuplicateAuditor:
    def __init__(self, cfg: Portion21Config):
        self.cfg = cfg
        self.output_dir = Path(cfg.output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def load(self) -> pd.DataFrame:
        train = pd.read_csv(TRAIN_PATH)

        ensure_required_cols(train)

        for col in [
            "is_synthetic",
            "eligible_for_tokenizer_train",
            "eligible_for_dialect_routing",
            "eligible_for_main_10dialect_benchmark",
        ]:
            if col in train.columns:
                train[col] = train[col].map(safe_bool)

        train["p21_row_id"] = np.arange(len(train))
        train["_source_priority"] = train.apply(source_priority, axis=1)

        print(f"[LOAD] Train rows from Portion 2: {len(train):,}")
        print(f"[LOAD] Dialects: {sorted(train['dialect'].unique().tolist())}")

        return train

    def build_group_report(self, train: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
        print("\n" + "=" * 100)
        print("STEP 1: BUILD LOOSE DUPLICATE GROUP REPORT")
        print("=" * 100)

        key_cols = list(self.cfg.loose_key_cols)

        group_size = (
            train.groupby(key_cols)
            .size()
            .reset_index(name="group_size")
        )

        dup_keys = group_size[group_size["group_size"] > 1].copy()

        members = train.merge(dup_keys[key_cols + ["group_size"]], on=key_cols, how="inner").copy()

        if len(members) == 0:
            group_report = pd.DataFrame()
            members_report = pd.DataFrame()
            print("[AUDIT] No loose duplicate groups found.")
            return group_report, members_report

        # Member-level helper fields.
        members["is_real"] = ~members["is_synthetic"].map(safe_bool)
        members["text_preview"] = members["text"].map(compact_text_preview)

        agg = (
            members.groupby(key_cols)
            .agg(
                group_size=("p21_row_id", "count"),
                real_count=("is_real", "sum"),
                synthetic_count=("is_synthetic", "sum"),
                unique_clean_text_hashes=("clean_text_hash", "nunique"),
                unique_texts=("text", "nunique"),
                unique_sources=("source", "nunique"),
                unique_source_files=("source_file", "nunique"),
                mean_quality=("quality_score_cleaned", "mean"),
                min_quality=("quality_score_cleaned", "min"),
                max_quality=("quality_score_cleaned", "max"),
                mean_char_len=("char_len", "mean"),
                min_char_len=("char_len", "min"),
                max_char_len=("char_len", "max"),
                mean_token_count=("token_count_ws", "mean"),
                min_token_count=("token_count_ws", "min"),
                max_token_count=("token_count_ws", "max"),
            )
            .reset_index()
        )

        # Add group type.
        def classify_group(row: pd.Series) -> str:
            if row["real_count"] > 0 and row["synthetic_count"] > 0:
                return "real_synthetic_collision"
            if row["real_count"] > 0 and row["synthetic_count"] == 0:
                return "real_only_loose_duplicate"
            if row["real_count"] == 0 and row["synthetic_count"] > 0:
                return "synthetic_only_loose_duplicate"
            return "unknown"

        agg["group_type"] = agg.apply(classify_group, axis=1)
        agg["likely_template_pressure"] = (
            (agg["synthetic_count"] >= 2) &
            (agg["synthetic_count"] >= agg["real_count"])
        )

        # Representative examples.
        examples = (
            members.sort_values(
                key_cols + ["_source_priority", "quality_score_cleaned", "char_len"],
                ascending=[True] * len(key_cols) + [True, False, False],
                kind="mergesort",
            )
            .groupby(key_cols)
            .head(5)
            .groupby(key_cols)["text_preview"]
            .apply(lambda xs: " || ".join(xs.tolist()))
            .reset_index(name="representative_examples")
        )

        group_report = agg.merge(examples, on=key_cols, how="left")
        group_report = group_report.sort_values(
            ["group_size", "synthetic_count", "real_count"],
            ascending=False,
        ).reset_index(drop=True)

        members_report = members.sort_values(
            key_cols + ["_source_priority", "quality_score_cleaned", "char_len"],
            ascending=[True] * len(key_cols) + [True, False, False],
            kind="mergesort",
        ).reset_index(drop=True)

        print(f"[AUDIT] Loose duplicate groups            : {len(group_report):,}")
        print(f"[AUDIT] Rows inside loose duplicate groups: {len(members_report):,}")
        print("\n[AUDIT] Group type counts:")
        print(group_report["group_type"].value_counts().to_string())

        return group_report, members_report

    def choose_rows_to_keep(self, group_df: pd.DataFrame) -> pd.DataFrame:
        """
        Conservative within-group selection.
        Real rows are always kept unless thin_real_only_groups=True.
        Synthetic rows are capped depending on group composition.
        """
        df = group_df.copy()

        df["is_real"] = ~df["is_synthetic"].map(safe_bool)
        real_count = int(df["is_real"].sum())
        syn_count = int(df["is_synthetic"].sum())
        group_size = len(df)

        df["p21_keep"] = False
        df["p21_decision_reason"] = ""

        # Sort best-first.
        df = df.sort_values(
            by=[
                "_source_priority",
                self.cfg.quality_col,
                "char_len",
                "token_count_ws",
                "source_file",
                "source_row_id",
            ],
            ascending=[True, False, False, False, True, True],
            kind="mergesort",
        ).copy()

        if group_size < self.cfg.min_group_size_for_thinning:
            df["p21_keep"] = True
            df["p21_decision_reason"] = "group_below_thinning_threshold"
            return df

        # Real-only group.
        if real_count > 0 and syn_count == 0:
            if self.cfg.thin_real_only_groups:
                # Not recommended. Keep best real row only.
                keep_ids = df.head(1)["p21_row_id"].tolist()
                df.loc[df["p21_row_id"].isin(keep_ids), "p21_keep"] = True
                df["p21_decision_reason"] = np.where(
                    df["p21_keep"],
                    "real_only_keep_best",
                    "real_only_thinned",
                )
            else:
                df["p21_keep"] = True
                df["p21_decision_reason"] = "real_only_preserved"
            return df

        # Mixed real + synthetic group.
        if real_count > 0 and syn_count > 0:
            if self.cfg.keep_all_real_rows:
                df.loc[df["is_real"], "p21_keep"] = True

            syn = df[df["is_synthetic"] == True].copy()
            syn_keep_n = min(self.cfg.max_synthetic_rows_when_real_exists, len(syn))
            syn_keep_ids = syn.head(syn_keep_n)["p21_row_id"].tolist()

            df.loc[df["p21_row_id"].isin(syn_keep_ids), "p21_keep"] = True

            df["p21_decision_reason"] = np.where(
                df["p21_keep"] & df["is_real"],
                "mixed_keep_real",
                np.where(
                    df["p21_keep"] & df["is_synthetic"],
                    "mixed_keep_top_synthetic",
                    "mixed_drop_excess_synthetic",
                ),
            )
            return df

        # Synthetic-only group.
        if real_count == 0 and syn_count > 0:
            keep_n = min(self.cfg.max_synthetic_rows_when_synthetic_only, len(df))
            keep_ids = df.head(keep_n)["p21_row_id"].tolist()
            df.loc[df["p21_row_id"].isin(keep_ids), "p21_keep"] = True

            df["p21_decision_reason"] = np.where(
                df["p21_keep"],
                "synthetic_only_keep_top",
                "synthetic_only_drop_excess",
            )
            return df

        df["p21_keep"] = True
        df["p21_decision_reason"] = "fallback_keep"
        return df

    def thin_recommended(self, train: pd.DataFrame, members_report: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
        print("\n" + "=" * 100)
        print("STEP 2: CONSERVATIVE SYNTHETIC THINNING")
        print("=" * 100)

        if len(members_report) == 0:
            train["p21_keep"] = True
            train["p21_decision_reason"] = "no_loose_duplicate"
            return train.copy(), train[["p21_row_id", "p21_keep", "p21_decision_reason"]].copy()

        key_cols = list(self.cfg.loose_key_cols)

        # Apply group selection only to duplicate members.
        decisions = (
            members_report
            .groupby(key_cols, group_keys=False)
            .apply(self.choose_rows_to_keep)
            .reset_index(drop=True)
        )

        decision_cols = [
            "p21_row_id",
            "dialect",
            "loose_text_hash",
            "text",
            "source",
            "source_file",
            "is_synthetic",
            "quality_score_cleaned",
            "char_len",
            "token_count_ws",
            "p21_keep",
            "p21_decision_reason",
        ]

        decision_report = decisions[decision_cols].copy()

        duplicate_row_ids = set(decision_report["p21_row_id"].tolist())
        keep_duplicate_ids = set(decision_report[decision_report["p21_keep"] == True]["p21_row_id"].tolist())

        train = train.copy()
        train["p21_keep"] = True
        train["p21_decision_reason"] = "not_in_loose_duplicate_group"

        train.loc[train["p21_row_id"].isin(duplicate_row_ids), "p21_keep"] = False
        train.loc[train["p21_row_id"].isin(duplicate_row_ids), "p21_decision_reason"] = "loose_duplicate_pending_decision"

        train.loc[train["p21_row_id"].isin(keep_duplicate_ids), "p21_keep"] = True

        reason_map = dict(zip(decision_report["p21_row_id"], decision_report["p21_decision_reason"]))
        train.loc[train["p21_row_id"].isin(duplicate_row_ids), "p21_decision_reason"] = (
            train.loc[train["p21_row_id"].isin(duplicate_row_ids), "p21_row_id"].map(reason_map)
        )

        recommended = train[train["p21_keep"] == True].copy().reset_index(drop=True)

        print(f"[THIN] Train rows before p2.1 thinning: {len(train):,}")
        print(f"[THIN] Train rows after p2.1 thinning : {len(recommended):,}")
        print(f"[THIN] Rows removed by p2.1           : {len(train) - len(recommended):,}")

        print("\n[THIN] Decision reason counts:")
        print(train["p21_decision_reason"].value_counts().to_string())

        return recommended, decision_report

    def run_tests(self, train: pd.DataFrame, recommended: pd.DataFrame) -> Dict[str, Any]:
        print("\n" + "=" * 100)
        print("PORTION 2.1 TESTS")
        print("=" * 100)

        failures = []

        null_text = int(recommended["text"].isna().sum() + (recommended["text"].astype(str).str.len() == 0).sum())
        if null_text > 0:
            failures.append(f"Null/empty text in recommended train: {null_text}")

        unk_rows = int((recommended["dialect"] == "UNK").sum())
        if unk_rows > 0:
            failures.append(f"UNK rows in recommended train: {unk_rows}")

        heldout_rows = int(recommended["split_original"].isin(["validation", "test"]).sum())
        if heldout_rows > 0:
            failures.append(f"Validation/test rows leaked into recommended train: {heldout_rows}")

        # Exact duplicates must remain zero.
        exact_dup = int(recommended.duplicated(subset=["clean_text_hash", "dialect"]).sum())
        if exact_dup > 0:
            failures.append(f"Exact clean_text_hash+dialect duplicates after p2.1: {exact_dup}")

        # Count remaining loose duplicates. This is allowed, but reported.
        loose_dup = int(recommended.duplicated(subset=["loose_text_hash", "dialect"]).sum())

        # Dialect coverage.
        before_counts = train["dialect"].value_counts().sort_index()
        after_counts = recommended["dialect"].value_counts().sort_index()

        low_dialects = after_counts[after_counts < self.cfg.min_rows_per_dialect_after_thinning].to_dict()
        if low_dialects:
            failures.append(
                f"Dialects below min_rows_per_dialect_after_thinning={self.cfg.min_rows_per_dialect_after_thinning}: {low_dialects}"
            )

        missing_dialects = sorted(set(train["dialect"].unique()) - set(recommended["dialect"].unique()))
        if missing_dialects:
            failures.append(f"Dialects missing after p2.1: {missing_dialects}")

        results = {
            "input_train_rows": int(len(train)),
            "recommended_train_rows": int(len(recommended)),
            "rows_removed_p21": int(len(train) - len(recommended)),
            "null_text_recommended": null_text,
            "unk_rows_recommended": unk_rows,
            "heldout_rows_recommended": heldout_rows,
            "exact_duplicate_clean_text_hash_dialect_recommended": exact_dup,
            "remaining_loose_duplicate_hash_dialect_recommended": loose_dup,
            "min_rows_per_dialect_after_thinning": self.cfg.min_rows_per_dialect_after_thinning,
            "dialect_counts_before": before_counts.to_dict(),
            "dialect_counts_after": after_counts.to_dict(),
            "low_dialects": low_dialects,
            "missing_dialects": missing_dialects,
            "hard_failures": failures,
            "passed": len(failures) == 0,
        }

        for k, v in results.items():
            if k not in {"hard_failures", "dialect_counts_before", "dialect_counts_after"}:
                print(f"{k:70s}: {v}")

        if failures:
            print("\n[FAILED]")
            for f in failures:
                print(f"  - {f}")
            raise AssertionError("Tokenizer Portion 2.1 tests failed.")

        print("\n[OK] Portion 2.1 tests passed.")
        return results

    def export(
        self,
        train: pd.DataFrame,
        recommended: pd.DataFrame,
        group_report: pd.DataFrame,
        members_report: pd.DataFrame,
        decision_report: pd.DataFrame,
        test_results: Dict[str, Any],
    ) -> None:
        print("\n" + "=" * 100)
        print("STEP 3: EXPORT PORTION 2.1 ARTIFACTS")
        print("=" * 100)

        audit_only_path = self.output_dir / "tokenizer_train_clean_p21_audit_only.csv"
        recommended_path = self.output_dir / "tokenizer_train_clean_p21_recommended.csv"
        group_path = self.output_dir / "loose_duplicate_group_report.csv"
        members_path = self.output_dir / "loose_duplicate_members_report.csv"
        decision_path = self.output_dir / "thinning_decision_report.csv"
        dist_path = self.output_dir / "dialect_distribution_p21_recommended.csv"
        source_dist_path = self.output_dir / "source_distribution_p21_recommended.csv"
        summary_path = self.output_dir / "p21_summary.json"

        train.to_csv(audit_only_path, index=False, encoding="utf-8")
        recommended.to_csv(recommended_path, index=False, encoding="utf-8")
        group_report.to_csv(group_path, index=False, encoding="utf-8")
        members_report.to_csv(members_path, index=False, encoding="utf-8")
        decision_report.to_csv(decision_path, index=False, encoding="utf-8")

        dist = (
            recommended.groupby(["dialect", "taxonomy_role", "is_synthetic"], dropna=False)
            .size()
            .reset_index(name="count")
            .sort_values(["dialect", "taxonomy_role", "is_synthetic"])
        )
        dist.to_csv(dist_path, index=False, encoding="utf-8")

        source_dist = (
            recommended.groupby(["source", "dialect", "is_synthetic"], dropna=False)
            .size()
            .reset_index(name="count")
            .sort_values(["source", "dialect", "is_synthetic"])
        )
        source_dist.to_csv(source_dist_path, index=False, encoding="utf-8")

        summary = {
            "portion": "tokenizer_p21_loose_duplicate_audit_conservative_synthetic_thinning",
            "seed": SEED,
            "config": asdict(self.cfg),
            "input_train_path": str(TRAIN_PATH),
            "rows_input_train": int(len(train)),
            "rows_recommended_train": int(len(recommended)),
            "rows_removed_p21": int(len(train) - len(recommended)),
            "loose_duplicate_groups": int(len(group_report)),
            "loose_duplicate_member_rows": int(len(members_report)),
            "decision_reason_counts": (
                recommended["p21_decision_reason"].value_counts().to_dict()
                if "p21_decision_reason" in recommended.columns else {}
            ),
            "test_results": test_results,
            "recommended_train_dialect_counts": recommended["dialect"].value_counts().sort_index().to_dict(),
            "recommended_train_real_counts": (
                recommended[recommended["is_synthetic"] == False]["dialect"]
                .value_counts()
                .sort_index()
                .to_dict()
            ),
            "recommended_train_synthetic_counts": (
                recommended[recommended["is_synthetic"] == True]["dialect"]
                .value_counts()
                .sort_index()
                .to_dict()
            ),
            "outputs": {
                "audit_only_train": str(audit_only_path),
                "recommended_train": str(recommended_path),
                "group_report": str(group_path),
                "members_report": str(members_path),
                "decision_report": str(decision_path),
                "distribution": str(dist_path),
                "source_distribution": str(source_dist_path),
            },
        }

        with open(summary_path, "w", encoding="utf-8") as f:
            json.dump(summary, f, ensure_ascii=False, indent=2)

        print(f"Saved: {audit_only_path}")
        print(f"Saved: {recommended_path}")
        print(f"Saved: {group_path}")
        print(f"Saved: {members_path}")
        print(f"Saved: {decision_path}")
        print(f"Saved: {dist_path}")
        print(f"Saved: {source_dist_path}")
        print(f"Saved: {summary_path}")

    def save_plots(self, train: pd.DataFrame, recommended: pd.DataFrame, group_report: pd.DataFrame) -> None:
        if not self.cfg.save_plots:
            return

        print("\n" + "=" * 100)
        print("STEP 4: SAVE PORTION 2.1 PLOTS")
        print("=" * 100)

        # Before/after counts.
        before = train["dialect"].value_counts().sort_index()
        after = recommended["dialect"].value_counts().sort_index()
        dialects = sorted(set(before.index) | set(after.index))

        before_vals = [before.get(d, 0) for d in dialects]
        after_vals = [after.get(d, 0) for d in dialects]

        x = np.arange(len(dialects))
        width = 0.38

        plt.figure(figsize=(15, 6))
        plt.bar(x - width / 2, before_vals, width, label="Before P2.1")
        plt.bar(x + width / 2, after_vals, width, label="Recommended P2.1")
        plt.title("Dialect Counts Before vs After P2.1")
        plt.xlabel("Dialect")
        plt.ylabel("Rows")
        plt.xticks(x, dialects, rotation=45)
        plt.legend()
        plt.tight_layout()
        p = self.output_dir / "dialect_counts_before_after_p21.png"
        plt.savefig(p, dpi=200)
        plt.close()
        print(f"Saved: {p}")

        # Group type counts.
        if len(group_report):
            gt = group_report["group_type"].value_counts()

            plt.figure(figsize=(10, 5))
            plt.bar(gt.index, gt.values)
            plt.title("Loose Duplicate Group Types")
            plt.xlabel("Group Type")
            plt.ylabel("Groups")
            plt.xticks(rotation=35, ha="right")
            plt.tight_layout()
            p = self.output_dir / "loose_duplicate_group_types.png"
            plt.savefig(p, dpi=200)
            plt.close()
            print(f"Saved: {p}")

    def checkpoint(self, train: pd.DataFrame, recommended: pd.DataFrame, group_report: pd.DataFrame) -> None:
        print("\n" + "=" * 100)
        print("CHECKPOINT: PORTION 2.1")
        print("=" * 100)

        print("\nLoose duplicate group summary:")
        if len(group_report):
            print(group_report["group_type"].value_counts().to_string())
            print("\nTop 20 largest loose duplicate groups:")
            cols = [
                "dialect",
                "loose_text_hash",
                "group_size",
                "real_count",
                "synthetic_count",
                "unique_texts",
                "unique_sources",
                "group_type",
                "representative_examples",
            ]
            print(group_report[cols].head(20).to_string(index=False))
        else:
            print("No loose duplicate groups found.")

        print("\nBefore P2.1 dialect counts:")
        print(train["dialect"].value_counts().sort_index().to_string())

        print("\nRecommended P2.1 dialect counts:")
        print(recommended["dialect"].value_counts().sort_index().to_string())

        print("\nRecommended P2.1 real/synthetic by dialect:")
        print(
            recommended.groupby(["dialect", "is_synthetic"])
            .size()
            .reset_index(name="count")
            .sort_values(["dialect", "is_synthetic"])
            .to_string(index=False)
        )

        print("\nRemaining duplicate diagnostics in recommended file:")
        print(f"Exact clean_text_hash+dialect duplicates : {int(recommended.duplicated(subset=['clean_text_hash', 'dialect']).sum()):,}")
        print(f"Loose loose_text_hash+dialect duplicates : {int(recommended.duplicated(subset=['loose_text_hash', 'dialect']).sum()):,}")

        print("\nUse for Portion 3 if accepted:")
        print(P21_DIR / "tokenizer_train_clean_p21_recommended.csv")

        print("\n✅ TOKENIZER PORTION 2.1 COMPLETE")
        print(f"Output directory: {P21_DIR}")


# ============================================================
# 4. Execute Portion 2.1
# ============================================================

auditor = LooseDuplicateAuditor(CFG)

df_train_p2 = auditor.load()
group_report_p21, members_report_p21 = auditor.build_group_report(df_train_p2)
df_train_p21_recommended, decision_report_p21 = auditor.thin_recommended(df_train_p2, members_report_p21)

test_results_p21 = auditor.run_tests(
    train=df_train_p2,
    recommended=df_train_p21_recommended,
)

auditor.export(
    train=df_train_p2,
    recommended=df_train_p21_recommended,
    group_report=group_report_p21,
    members_report=members_report_p21,
    decision_report=decision_report_p21,
    test_results=test_results_p21,
)

auditor.save_plots(
    train=df_train_p2,
    recommended=df_train_p21_recommended,
    group_report=group_report_p21,
)

auditor.checkpoint(
    train=df_train_p2,
    recommended=df_train_p21_recommended,
    group_report=group_report_p21,
)

TOKENIZER PORTION 2.1 — LOOSE DUPLICATE AUDIT + CONSERVATIVE SYNTHETIC THINNING
Train input      : artifacts/tokenizer_p2/tokenizer_train_clean.csv
All-quality input: artifacts/tokenizer_p2/cleaned_all_with_quality.csv
Output dir       : artifacts/tokenizer_p21
GPU needed       : NO — CPU is sufficient for this portion.
[LOAD] Train rows from Portion 2: 123,833
[LOAD] Dialects: ['BAR', 'CHI', 'KHU', 'KIS', 'MYM', 'NAR', 'NOA', 'NSD', 'RAJ', 'RAN', 'STD', 'SYL', 'TAN']

STEP 1: BUILD LOOSE DUPLICATE GROUP REPORT
[AUDIT] Loose duplicate groups            : 4,946
[AUDIT] Rows inside loose duplicate groups: 12,265

[AUDIT] Group type counts:
group_type
synthetic_only_loose_duplicate    4506
real_only_loose_duplicate          413
real_synthetic_collision            27

STEP 2: CONSERVATIVE SYNTHETIC THINNING
[THIN] Train rows before p2.1 thinning: 123,833
[THIN] Train rows after p2.1 thinning : 121,503
[THIN] Rows removed by p2.1           : 2,330

[THIN] Decision reason counts:
p21_decisio

## portion-3

In [5]:
# ============================================================
# TOKENIZER PORTION 3 — FIXED FINAL VERSION
# Dialect-Aware Token Mining + Protected Vocabulary Construction
# With Adaptive Strict Backfill for Low-Real-Resource Dialects
# ============================================================
#
# Input:
#   artifacts/tokenizer_p21/tokenizer_train_clean_p21_recommended.csv
#
# Main fix:
#   Previous strict filters were too rigid for KHU and RAJ.
#   This version:
#     - mines strict real/mixed candidates first
#     - mines expanded candidates
#     - adaptively backfills strict lists from high-quality expanded tokens
#       only when a dialect has too few strict candidates
#     - marks such tokens as adaptive_synthetic_assisted
#
# GPU:
#   No GPU required. CPU is enough.
# ============================================================

from __future__ import annotations

import os
import re
import json
import math
import random
import hashlib
import warnings
import unicodedata
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Tuple
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


# ============================================================
# 0. Reproducibility
# ============================================================

SEED = 42

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)


# ============================================================
# 1. Config
# ============================================================

@dataclass(frozen=True)
class Portion3Config:
    input_path: str = "artifacts/tokenizer_p21/tokenizer_train_clean_p21_recommended.csv"
    output_dir: str = "artifacts/tokenizer_p3"

    # Tokenization/statistics
    lowercase_latin: bool = True
    keep_numeric_tokens_in_stats: bool = True
    protect_numeric_tokens: bool = False

    # Smoothing for dialect association
    alpha: float = 0.10

    # Core/shared-token mining
    min_core_total_count: int = 15
    min_core_real_count: int = 5
    min_core_dialect_coverage: int = 5
    max_core_dialect_purity: float = 0.65
    min_core_normalized_entropy: float = 0.45

    # Strict dialect-token mining
    min_dialect_token_count: int = 5
    min_dialect_real_count_strict: int = 2
    min_dialect_purity_strict: float = 0.55
    min_dialect_log_odds_strict: float = 0.50
    min_dialect_weighted_pmi_strict: float = 1.00
    max_synthetic_fraction_strict: float = 0.80

    # Expanded dialect-token mining
    min_dialect_token_count_expanded: int = 5
    min_dialect_purity_expanded: float = 0.45
    min_dialect_log_odds_expanded: float = 0.20
    max_synthetic_fraction_expanded: float = 0.95

    # Adaptive backfill for low-resource/synthetic-supported dialects
    enable_adaptive_strict_backfill: bool = True
    min_strict_candidates_per_dialect: int = 20
    adaptive_min_count_td: int = 5
    adaptive_min_purity: float = 0.45
    adaptive_min_log_odds: float = 0.15
    adaptive_max_synthetic_fraction: float = 0.98

    # Synthetic penalties for association score
    synthetic_only_penalty: float = 0.45
    high_synthetic_penalty: float = 0.70

    # Caps
    max_core_tokens_strict: int = 1800
    max_core_tokens_expanded: int = 4000
    max_dialect_tokens_strict_per_dialect: int = 250
    max_dialect_tokens_expanded_per_dialect: int = 700
    max_total_lexical_strict: int = 6000
    max_total_lexical_expanded: int = 14000

    # Protectable token rules
    min_chars_for_protected_token: int = 2
    max_chars_for_protected_token: int = 40

    # Diagnostics
    top_k_export_per_dialect: int = 2000
    save_plots: bool = True

    # Decoder-only SLM/LLM special tokens
    special_tokens: Tuple[str, ...] = (
        "<pad>",
        "<unk>",
        "<bos>",
        "<eos>",
    )

    dialect_tag_prefix: str = "<dial:"
    dialect_tag_suffix: str = ">"

    target_dialects_10: Tuple[str, ...] = (
        "BAR", "CHI", "KHU", "MYM", "NAR",
        "NOA", "RAJ", "RAN", "STD", "SYL",
    )

    auxiliary_dialects: Tuple[str, ...] = (
        "KIS", "TAN", "NSD",
    )


CFG = Portion3Config()
INPUT_PATH = Path(CFG.input_path)
P3_DIR = Path(CFG.output_dir)
P3_DIR.mkdir(parents=True, exist_ok=True)

ALL_DIALECTS = sorted(set(CFG.target_dialects_10 + CFG.auxiliary_dialects))
DIALECT_TAGS = [f"{CFG.dialect_tag_prefix}{d}{CFG.dialect_tag_suffix}" for d in ALL_DIALECTS]

print("=" * 100)
print("TOKENIZER PORTION 3 — FIXED FINAL: DIALECT-AWARE TOKEN MINING")
print("=" * 100)
print(f"Input path : {INPUT_PATH}")
print(f"Output dir : {P3_DIR}")
print(f"Dialects   : {ALL_DIALECTS}")
print("GPU needed : NO — CPU is sufficient for this portion.")
print("=" * 100)

assert INPUT_PATH.exists(), f"Missing input file: {INPUT_PATH}"


# ============================================================
# 2. Utility Functions
# ============================================================

TOKEN_RE = re.compile(
    r"[\u0980-\u09FF]+|[A-Za-z]+(?:'[A-Za-z]+)?|\d+(?:[.,]\d+)?",
    flags=re.UNICODE,
)

BAD_PROTECTED_PATTERNS = [
    r"<>",
    r"&lt;&gt;",
    r"^\s*$",
]

def safe_bool(x: Any) -> bool:
    if isinstance(x, bool):
        return x
    if pd.isna(x):
        return False
    return str(x).strip().lower() in {"true", "1", "yes", "y"}

def is_bangla_char(ch: str) -> bool:
    return "\u0980" <= ch <= "\u09FF"

def has_bangla(text: str) -> bool:
    return any(is_bangla_char(ch) for ch in str(text))

def is_latin_only(text: str) -> bool:
    s = str(text)
    chars = [c for c in s if not c.isspace()]
    return len(chars) > 0 and all(("a" <= c.lower() <= "z") for c in chars)

def is_numeric_like(text: str) -> bool:
    return bool(re.fullmatch(r"\d+(?:[.,]\d+)?", str(text)))

def is_punctuation_only(text: str) -> bool:
    s = str(text).strip()
    if not s:
        return True
    return all(unicodedata.category(ch).startswith(("P", "S")) for ch in s)

def normalize_token(token: str) -> str:
    token = unicodedata.normalize("NFC", str(token).strip())
    if CFG.lowercase_latin and is_latin_only(token):
        token = token.lower()
    return token

def lexical_tokenize(text: Any) -> List[str]:
    if pd.isna(text):
        return []
    s = unicodedata.normalize("NFC", str(text))
    toks = [normalize_token(m.group(0)) for m in TOKEN_RE.finditer(s)]
    return [t for t in toks if t]

def is_bad_artifact_token(token: str) -> bool:
    s = str(token).strip()
    for pat in BAD_PROTECTED_PATTERNS:
        if re.search(pat, s):
            return True
    return False

def is_protectable_token(token: str) -> bool:
    token = str(token).strip()

    if is_bad_artifact_token(token):
        return False
    if len(token) < CFG.min_chars_for_protected_token:
        return False
    if len(token) > CFG.max_chars_for_protected_token:
        return False
    if is_punctuation_only(token):
        return False
    if is_numeric_like(token) and not CFG.protect_numeric_tokens:
        return False
    if not has_bangla(token):
        return False

    return True

def entropy_from_counts(counts: List[int]) -> float:
    total = sum(counts)
    if total <= 0:
        return 0.0

    h = 0.0
    for c in counts:
        if c <= 0:
            continue
        p = c / total
        h -= p * math.log(p + 1e-12)

    return h

def normalized_entropy(counts: List[int]) -> float:
    if len(counts) <= 1:
        return 0.0
    return entropy_from_counts(counts) / math.log(len(counts))

def write_lines(path: Path, lines: List[str]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for x in lines:
            f.write(str(x).strip() + "\n")

def dedup_keep_order(xs: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in xs:
        x = str(x).strip()
        if not x:
            continue
        if x in seen:
            continue
        seen.add(x)
        out.append(x)
    return out


# ============================================================
# 3. Token Counter
# ============================================================

class TokenCounterBuilder:
    def __init__(self, cfg: Portion3Config):
        self.cfg = cfg

        self.token_total = Counter()
        self.token_real_total = Counter()
        self.token_syn_total = Counter()
        self.token_doc_count = Counter()

        self.token_dialect = Counter()
        self.token_dialect_real = Counter()
        self.token_dialect_syn = Counter()
        self.token_dialect_doc = Counter()

        self.dialect_total_tokens = Counter()
        self.dialect_total_docs = Counter()

    def add_row(self, row: pd.Series) -> None:
        text = row["text"]
        dialect = row["dialect"]
        is_syn = safe_bool(row["is_synthetic"])

        tokens = lexical_tokenize(text)

        if not self.cfg.keep_numeric_tokens_in_stats:
            tokens = [t for t in tokens if not is_numeric_like(t)]

        self.dialect_total_docs[dialect] += 1
        self.dialect_total_tokens[dialect] += len(tokens)

        local_counts = Counter(tokens)
        unique_tokens = set(tokens)

        for tok, c in local_counts.items():
            self.token_total[tok] += c
            self.token_dialect[(tok, dialect)] += c

            if is_syn:
                self.token_syn_total[tok] += c
                self.token_dialect_syn[(tok, dialect)] += c
            else:
                self.token_real_total[tok] += c
                self.token_dialect_real[(tok, dialect)] += c

        for tok in unique_tokens:
            self.token_doc_count[tok] += 1
            self.token_dialect_doc[(tok, dialect)] += 1

    def build(self, df: pd.DataFrame) -> None:
        print("\n" + "=" * 100)
        print("STEP 1: BUILD TOKEN COUNTERS")
        print("=" * 100)

        for i, row in df.iterrows():
            self.add_row(row)
            if (i + 1) % 25000 == 0:
                print(f"[COUNT] Processed {i + 1:,} rows...")

        print(f"[COUNT] Processed rows        : {len(df):,}")
        print(f"[COUNT] Unique lexical tokens : {len(self.token_total):,}")
        print(f"[COUNT] Total lexical tokens  : {sum(self.token_total.values()):,}")

    def to_dataframes(self) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        print("\n" + "=" * 100)
        print("STEP 2: CONVERT COUNTERS TO DATAFRAMES")
        print("=" * 100)

        total_tokens_all = sum(self.token_total.values())
        vocab_size = max(1, len(self.token_total))

        token_rows = []
        score_rows = []

        for tok, total_count in self.token_total.items():
            dialect_counts = [self.token_dialect[(tok, d)] for d in ALL_DIALECTS]
            dialect_coverage_1 = sum(c > 0 for c in dialect_counts)
            dialect_coverage_min = sum(c >= self.cfg.min_dialect_token_count for c in dialect_counts)
            max_dialect_count = max(dialect_counts) if dialect_counts else 0
            max_dialect = ALL_DIALECTS[int(np.argmax(dialect_counts))] if dialect_counts else None
            max_purity = max_dialect_count / max(1, total_count)

            h_norm = normalized_entropy(dialect_counts)

            real_total = self.token_real_total[tok]
            syn_total = self.token_syn_total[tok]
            syn_frac = syn_total / max(1, total_count)

            protectable = is_protectable_token(tok)

            token_rows.append({
                "token": tok,
                "total_count": int(total_count),
                "real_count": int(real_total),
                "synthetic_count": int(syn_total),
                "synthetic_fraction": syn_frac,
                "doc_count": int(self.token_doc_count[tok]),
                "dialect_coverage_1": int(dialect_coverage_1),
                "dialect_coverage_min": int(dialect_coverage_min),
                "max_dialect": max_dialect,
                "max_dialect_count": int(max_dialect_count),
                "max_dialect_purity": max_purity,
                "normalized_dialect_entropy": h_norm,
                "is_protectable": protectable,
                "has_bangla": has_bangla(tok),
                "is_numeric": is_numeric_like(tok),
                "is_latin_only": is_latin_only(tok),
                "is_bad_artifact": is_bad_artifact_token(tok),
            })

            for d in ALL_DIALECTS:
                c_td = self.token_dialect[(tok, d)]
                if c_td <= 0:
                    continue

                c_td_real = self.token_dialect_real[(tok, d)]
                c_td_syn = self.token_dialect_syn[(tok, d)]
                doc_td = self.token_dialect_doc[(tok, d)]

                n_d = self.dialect_total_tokens[d]
                c_not = total_count - c_td
                n_not = total_tokens_all - n_d

                p_tok_given_d = (c_td + self.cfg.alpha) / max(1.0, n_d + self.cfg.alpha * vocab_size)
                p_tok_given_not_d = (c_not + self.cfg.alpha) / max(1.0, n_not + self.cfg.alpha * vocab_size)
                p_tok_all = total_count / max(1, total_tokens_all)

                smoothed_log_odds = math.log(p_tok_given_d + 1e-12) - math.log(p_tok_given_not_d + 1e-12)
                pmi = math.log((p_tok_given_d + 1e-12) / (p_tok_all + 1e-12))
                weighted_pmi = pmi * math.log1p(c_td)

                purity = c_td / max(1, total_count)
                dialect_syn_frac = c_td_syn / max(1, c_td)

                if c_td_real == 0 and c_td_syn > 0:
                    synthetic_penalty = self.cfg.synthetic_only_penalty
                    support_type = "synthetic_only"
                elif dialect_syn_frac >= self.cfg.max_synthetic_fraction_strict:
                    synthetic_penalty = self.cfg.high_synthetic_penalty
                    support_type = "synthetic_dominant"
                else:
                    synthetic_penalty = 1.0
                    support_type = "real_or_mixed"

                association_score = (
                    smoothed_log_odds
                    * math.log1p(c_td)
                    * (0.50 + 0.50 * purity)
                    * synthetic_penalty
                )

                score_rows.append({
                    "token": tok,
                    "dialect": d,
                    "count_td": int(c_td),
                    "real_count_td": int(c_td_real),
                    "synthetic_count_td": int(c_td_syn),
                    "doc_count_td": int(doc_td),
                    "dialect_total_tokens": int(n_d),
                    "token_total_count": int(total_count),
                    "token_real_count": int(real_total),
                    "token_synthetic_count": int(syn_total),
                    "token_synthetic_fraction": syn_frac,
                    "dialect_synthetic_fraction": dialect_syn_frac,
                    "purity_token_to_dialect": purity,
                    "p_token_given_dialect": p_tok_given_d,
                    "pmi": pmi,
                    "weighted_pmi": weighted_pmi,
                    "smoothed_log_odds": smoothed_log_odds,
                    "association_score": association_score,
                    "support_type": support_type,
                    "is_protectable": protectable,
                    "is_bad_artifact": is_bad_artifact_token(tok),
                    "is_numeric": is_numeric_like(tok),
                    "has_bangla": has_bangla(tok),
                })

        token_stats = pd.DataFrame(token_rows)
        dialect_scores = pd.DataFrame(score_rows)

        freq_rows = []
        for tok in self.token_total.keys():
            row = {"token": tok, "total_count": int(self.token_total[tok])}
            for d in ALL_DIALECTS:
                row[d] = int(self.token_dialect[(tok, d)])
            freq_rows.append(row)

        freq_by_dialect = pd.DataFrame(freq_rows)

        print(f"[DF] token_statistics rows      : {len(token_stats):,}")
        print(f"[DF] token_dialect_scores rows  : {len(dialect_scores):,}")
        print(f"[DF] token_frequency rows       : {len(freq_by_dialect):,}")

        return token_stats, dialect_scores, freq_by_dialect


# ============================================================
# 4. Token Miner
# ============================================================

class DialectAwareTokenMiner:
    def __init__(self, cfg: Portion3Config):
        self.cfg = cfg
        self.output_dir = Path(cfg.output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def load(self) -> pd.DataFrame:
        df = pd.read_csv(INPUT_PATH)

        for col in [
            "is_synthetic",
            "eligible_for_tokenizer_train",
            "eligible_for_dialect_routing",
            "eligible_for_main_10dialect_benchmark",
        ]:
            if col in df.columns:
                df[col] = df[col].map(safe_bool)

        df = df[df["eligible_for_tokenizer_train"] == True].copy()
        df = df[~df["split_original"].isin(["validation", "test"])].copy()
        df = df[df["dialect"].isin(ALL_DIALECTS)].copy()
        df = df[df["text"].notna()].copy()
        df = df[df["text"].astype(str).str.len() > 0].copy()

        print(f"[LOAD] Rows loaded for Portion 3: {len(df):,}")
        print("[LOAD] Dialect counts:")
        print(df["dialect"].value_counts().sort_index().to_string())

        return df.reset_index(drop=True)

    def mine_core_tokens(self, token_stats: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
        print("\n" + "=" * 100)
        print("STEP 3: MINE CORE SHARED TOKENS")
        print("=" * 100)

        ts = token_stats.copy()

        ts["real_support_factor"] = np.log1p(ts["real_count"]) / np.log1p(ts["total_count"].clip(lower=1))
        ts["coverage_factor"] = ts["dialect_coverage_min"] / len(ALL_DIALECTS)
        ts["entropy_factor"] = ts["normalized_dialect_entropy"].clip(0, 1)

        ts["core_score"] = (
            np.log1p(ts["total_count"])
            * (0.40 + 0.60 * ts["coverage_factor"])
            * (0.40 + 0.60 * ts["entropy_factor"])
            * (0.50 + 0.50 * ts["real_support_factor"])
            * (1.00 - 0.25 * ts["synthetic_fraction"].clip(0, 1))
        )

        core_candidates = ts[
            (ts["is_protectable"] == True)
            & (ts["is_bad_artifact"] == False)
            & (ts["total_count"] >= self.cfg.min_core_total_count)
            & (ts["real_count"] >= self.cfg.min_core_real_count)
            & (ts["dialect_coverage_min"] >= self.cfg.min_core_dialect_coverage)
            & (ts["max_dialect_purity"] <= self.cfg.max_core_dialect_purity)
            & (ts["normalized_dialect_entropy"] >= self.cfg.min_core_normalized_entropy)
        ].copy()

        core_candidates = core_candidates.sort_values(
            ["core_score", "total_count", "dialect_coverage_min", "real_count"],
            ascending=False,
            kind="mergesort",
        ).reset_index(drop=True)

        core_strict = core_candidates.head(self.cfg.max_core_tokens_strict).copy()
        core_expanded = core_candidates.head(self.cfg.max_core_tokens_expanded).copy()

        core_strict["protected_class"] = "core_strict"
        core_strict["candidate_tier"] = "core_strict"

        core_expanded["protected_class"] = "core_expanded"
        core_expanded["candidate_tier"] = "core_expanded"

        print(f"[CORE] Candidates       : {len(core_candidates):,}")
        print(f"[CORE] Strict selected  : {len(core_strict):,}")
        print(f"[CORE] Expanded selected: {len(core_expanded):,}")

        return core_strict, core_expanded

    def _base_strict_filter(self, sub: pd.DataFrame) -> pd.DataFrame:
        return sub[
            (sub["is_protectable"] == True)
            & (sub["is_bad_artifact"] == False)
            & (sub["has_bangla"] == True)
            & (sub["is_numeric"] == False)
            & (sub["count_td"] >= self.cfg.min_dialect_token_count)
            & (sub["real_count_td"] >= self.cfg.min_dialect_real_count_strict)
            & (sub["purity_token_to_dialect"] >= self.cfg.min_dialect_purity_strict)
            & (sub["smoothed_log_odds"] >= self.cfg.min_dialect_log_odds_strict)
            & (sub["weighted_pmi"] >= self.cfg.min_dialect_weighted_pmi_strict)
            & (sub["dialect_synthetic_fraction"] <= self.cfg.max_synthetic_fraction_strict)
        ].copy()

    def _expanded_filter(self, sub: pd.DataFrame) -> pd.DataFrame:
        return sub[
            (sub["is_protectable"] == True)
            & (sub["is_bad_artifact"] == False)
            & (sub["has_bangla"] == True)
            & (sub["is_numeric"] == False)
            & (sub["count_td"] >= self.cfg.min_dialect_token_count_expanded)
            & (sub["purity_token_to_dialect"] >= self.cfg.min_dialect_purity_expanded)
            & (sub["smoothed_log_odds"] >= self.cfg.min_dialect_log_odds_expanded)
            & (sub["dialect_synthetic_fraction"] <= self.cfg.max_synthetic_fraction_expanded)
        ].copy()

    def _adaptive_filter(self, sub: pd.DataFrame) -> pd.DataFrame:
        return sub[
            (sub["is_protectable"] == True)
            & (sub["is_bad_artifact"] == False)
            & (sub["has_bangla"] == True)
            & (sub["is_numeric"] == False)
            & (sub["count_td"] >= self.cfg.adaptive_min_count_td)
            & (sub["purity_token_to_dialect"] >= self.cfg.adaptive_min_purity)
            & (sub["smoothed_log_odds"] >= self.cfg.adaptive_min_log_odds)
            & (sub["dialect_synthetic_fraction"] <= self.cfg.adaptive_max_synthetic_fraction)
        ].copy()

    def mine_dialect_tokens(self, dialect_scores: pd.DataFrame) -> Tuple[Dict[str, pd.DataFrame], Dict[str, pd.DataFrame]]:
        print("\n" + "=" * 100)
        print("STEP 4: MINE DIALECT-SPECIFIC TOKENS")
        print("=" * 100)

        strict_by_dialect: Dict[str, pd.DataFrame] = {}
        expanded_by_dialect: Dict[str, pd.DataFrame] = {}

        ds = dialect_scores.copy()

        for d in ALL_DIALECTS:
            sub = ds[ds["dialect"] == d].copy()

            strict = self._base_strict_filter(sub)
            strict["candidate_tier"] = "strict_real_or_mixed"

            expanded = self._expanded_filter(sub)
            expanded["candidate_tier"] = np.where(
                expanded["support_type"].isin(["synthetic_only", "synthetic_dominant"]),
                "expanded_synthetic_assisted",
                "expanded_real_or_mixed",
            )

            # Sort strict and expanded.
            strict = strict.sort_values(
                ["association_score", "count_td", "real_count_td", "purity_token_to_dialect"],
                ascending=False,
                kind="mergesort",
            ).reset_index(drop=True)

            expanded = expanded.sort_values(
                ["association_score", "count_td", "real_count_td", "purity_token_to_dialect"],
                ascending=False,
                kind="mergesort",
            ).reset_index(drop=True)

            # Adaptive backfill if strict is too small.
            if self.cfg.enable_adaptive_strict_backfill and len(strict) < self.cfg.min_strict_candidates_per_dialect:
                needed = self.cfg.min_strict_candidates_per_dialect - len(strict)
                strict_tokens = set(strict["token"].tolist())

                adaptive = self._adaptive_filter(sub)
                adaptive = adaptive[~adaptive["token"].isin(strict_tokens)].copy()

                adaptive["candidate_tier"] = np.where(
                    adaptive["support_type"].isin(["synthetic_only", "synthetic_dominant"]),
                    "adaptive_synthetic_assisted",
                    "adaptive_real_or_mixed",
                )

                adaptive = adaptive.sort_values(
                    [
                        "association_score",
                        "count_td",
                        "real_count_td",
                        "purity_token_to_dialect",
                    ],
                    ascending=False,
                    kind="mergesort",
                ).head(needed).reset_index(drop=True)

                if len(adaptive) > 0:
                    strict = pd.concat([strict, adaptive], axis=0).reset_index(drop=True)

            strict = strict.head(self.cfg.max_dialect_tokens_strict_per_dialect).copy()
            expanded = expanded.head(self.cfg.max_dialect_tokens_expanded_per_dialect).copy()

            strict["protected_class"] = f"dialect_{d}_strict"
            expanded["protected_class"] = f"dialect_{d}_expanded"

            strict_by_dialect[d] = strict
            expanded_by_dialect[d] = expanded

            adaptive_count = int(strict["candidate_tier"].astype(str).str.contains("adaptive").sum()) if len(strict) else 0
            synth_assisted_count = int(strict["candidate_tier"].astype(str).str.contains("synthetic").sum()) if len(strict) else 0

            print(
                f"[DIALECT] {d:3s} | "
                f"strict={len(strict):4d} | expanded={len(expanded):4d} | "
                f"adaptive={adaptive_count:3d} | synth_assist={synth_assisted_count:3d} | "
                f"top={strict['token'].head(5).tolist() if len(strict) else []}"
            )

        return strict_by_dialect, expanded_by_dialect

    def build_protected_lists(
        self,
        core_strict: pd.DataFrame,
        core_expanded: pd.DataFrame,
        strict_by_dialect: Dict[str, pd.DataFrame],
        expanded_by_dialect: Dict[str, pd.DataFrame],
    ) -> Dict[str, List[str]]:

        print("\n" + "=" * 100)
        print("STEP 5: BUILD STRICT + EXPANDED PROTECTED VOCABULARIES")
        print("=" * 100)

        special = list(self.cfg.special_tokens)
        tags = DIALECT_TAGS

        strict_lexical = []
        strict_lexical.extend(core_strict["token"].tolist())
        for d in ALL_DIALECTS:
            strict_lexical.extend(strict_by_dialect[d]["token"].tolist())

        expanded_lexical = []
        expanded_lexical.extend(core_expanded["token"].tolist())
        for d in ALL_DIALECTS:
            expanded_lexical.extend(expanded_by_dialect[d]["token"].tolist())

        strict_lexical = dedup_keep_order(strict_lexical)
        expanded_lexical = dedup_keep_order(expanded_lexical)

        strict_lexical = strict_lexical[: self.cfg.max_total_lexical_strict]
        expanded_lexical = expanded_lexical[: self.cfg.max_total_lexical_expanded]

        protected_strict = dedup_keep_order(special + tags + strict_lexical)
        protected_expanded = dedup_keep_order(special + tags + expanded_lexical)

        print(f"[PROTECTED] Special tokens          : {len(special)}")
        print(f"[PROTECTED] Dialect tags             : {len(tags)}")
        print(f"[PROTECTED] Strict lexical tokens    : {len(strict_lexical):,}")
        print(f"[PROTECTED] Expanded lexical tokens  : {len(expanded_lexical):,}")
        print(f"[PROTECTED] Strict total symbols     : {len(protected_strict):,}")
        print(f"[PROTECTED] Expanded total symbols   : {len(protected_expanded):,}")

        return {
            "special": special,
            "dialect_tags": tags,
            "strict_lexical": strict_lexical,
            "expanded_lexical": expanded_lexical,
            "protected_strict": protected_strict,
            "protected_expanded": protected_expanded,
        }

    def compute_protected_coverage(self, df: pd.DataFrame, protected: Dict[str, List[str]]) -> pd.DataFrame:
        print("\n" + "=" * 100)
        print("STEP 6: COMPUTE PROTECTED-TOKEN COVERAGE")
        print("=" * 100)

        strict_set = set(protected["strict_lexical"])
        expanded_set = set(protected["expanded_lexical"])

        rows = []
        for d, sub in df.groupby("dialect"):
            total_tokens = 0
            strict_hits = 0
            expanded_hits = 0
            unique_tokens = set()
            unique_strict = set()
            unique_expanded = set()

            for text in sub["text"].tolist():
                toks = lexical_tokenize(text)
                total_tokens += len(toks)

                for t in toks:
                    unique_tokens.add(t)
                    if t in strict_set:
                        strict_hits += 1
                        unique_strict.add(t)
                    if t in expanded_set:
                        expanded_hits += 1
                        unique_expanded.add(t)

            rows.append({
                "dialect": d,
                "rows": int(len(sub)),
                "total_lexical_tokens": int(total_tokens),
                "unique_lexical_tokens": int(len(unique_tokens)),
                "strict_token_hits": int(strict_hits),
                "expanded_token_hits": int(expanded_hits),
                "strict_token_coverage": strict_hits / max(1, total_tokens),
                "expanded_token_coverage": expanded_hits / max(1, total_tokens),
                "strict_type_coverage": len(unique_strict) / max(1, len(unique_tokens)),
                "expanded_type_coverage": len(unique_expanded) / max(1, len(unique_tokens)),
                "unique_strict_hit_types": int(len(unique_strict)),
                "unique_expanded_hit_types": int(len(unique_expanded)),
            })

        cov = pd.DataFrame(rows).sort_values("dialect").reset_index(drop=True)

        print("[COVERAGE] Protected lexical-token coverage by dialect:")
        print(
            cov[
                [
                    "dialect",
                    "rows",
                    "total_lexical_tokens",
                    "unique_lexical_tokens",
                    "strict_token_coverage",
                    "expanded_token_coverage",
                    "strict_type_coverage",
                    "expanded_type_coverage",
                ]
            ].round(4).to_string(index=False)
        )

        return cov

    def run_tests(
        self,
        df: pd.DataFrame,
        token_stats: pd.DataFrame,
        dialect_scores: pd.DataFrame,
        core_strict: pd.DataFrame,
        strict_by_dialect: Dict[str, pd.DataFrame],
        expanded_by_dialect: Dict[str, pd.DataFrame],
        protected: Dict[str, List[str]],
    ) -> Dict[str, Any]:

        print("\n" + "=" * 100)
        print("PORTION 3 TESTS")
        print("=" * 100)

        failures = []
        warnings_list = []

        null_text = int(df["text"].isna().sum() + (df["text"].astype(str).str.len() == 0).sum())
        if null_text > 0:
            failures.append(f"Null/empty input text rows: {null_text}")

        heldout_rows = int(df["split_original"].isin(["validation", "test"]).sum())
        if heldout_rows > 0:
            failures.append(f"Held-out rows leaked into Portion 3 input: {heldout_rows}")

        unk_rows = int((df["dialect"] == "UNK").sum())
        if unk_rows > 0:
            failures.append(f"UNK rows in Portion 3 input: {unk_rows}")

        bad_strict = [t for t in protected["protected_strict"] if is_bad_artifact_token(t)]
        if bad_strict:
            failures.append(f"Bad artifact tokens in protected_strict: {bad_strict[:10]}")

        missing_special = [t for t in self.cfg.special_tokens if t not in protected["protected_strict"]]
        missing_tags = [t for t in DIALECT_TAGS if t not in protected["protected_strict"]]

        if missing_special:
            failures.append(f"Missing special tokens: {missing_special}")
        if missing_tags:
            failures.append(f"Missing dialect tags: {missing_tags}")

        if len(protected["protected_strict"]) != len(set(protected["protected_strict"])):
            failures.append("Duplicate tokens in protected_strict.")

        if len(protected["protected_expanded"]) != len(set(protected["protected_expanded"])):
            failures.append("Duplicate tokens in protected_expanded.")

        punct_only = [t for t in protected["strict_lexical"] if is_punctuation_only(t)]
        if punct_only:
            failures.append(f"Punctuation-only lexical protected tokens: {punct_only[:10]}")

        important_cols = ["association_score", "smoothed_log_odds", "weighted_pmi", "purity_token_to_dialect"]
        nan_counts = dialect_scores[important_cols].isna().sum().to_dict()
        bad_nan = {k: int(v) for k, v in nan_counts.items() if v > 0}
        if bad_nan:
            failures.append(f"NaN values in dialect score columns: {bad_nan}")

        dialect_strict_counts = {d: int(len(strict_by_dialect[d])) for d in ALL_DIALECTS}
        dialect_expanded_counts = {d: int(len(expanded_by_dialect[d])) for d in ALL_DIALECTS}

        low_strict = {
            d: c for d, c in dialect_strict_counts.items()
            if c < self.cfg.min_strict_candidates_per_dialect
        }

        if low_strict:
            failures.append(f"Too few strict dialect candidates after adaptive backfill: {low_strict}")

        adaptive_counts = {
            d: int(strict_by_dialect[d]["candidate_tier"].astype(str).str.contains("adaptive").sum())
            if len(strict_by_dialect[d]) else 0
            for d in ALL_DIALECTS
        }

        synthetic_assisted_strict_counts = {
            d: int(strict_by_dialect[d]["candidate_tier"].astype(str).str.contains("synthetic").sum())
            if len(strict_by_dialect[d]) else 0
            for d in ALL_DIALECTS
        }

        synthetic_assisted_dialects = {
            d: c for d, c in synthetic_assisted_strict_counts.items() if c > 0
        }

        if synthetic_assisted_dialects:
            warnings_list.append(
                f"Synthetic-assisted strict candidates used: {synthetic_assisted_dialects}"
            )

        if len(token_stats) == 0:
            failures.append("token_statistics is empty.")

        if len(core_strict) == 0:
            failures.append("core_strict is empty.")

        results = {
            "input_rows": int(len(df)),
            "unique_lexical_tokens": int(len(token_stats)),
            "dialect_score_rows": int(len(dialect_scores)),
            "core_strict_count": int(len(core_strict)),
            "protected_strict_total": int(len(protected["protected_strict"])),
            "protected_expanded_total": int(len(protected["protected_expanded"])),
            "null_text_rows": null_text,
            "heldout_rows": heldout_rows,
            "unk_rows": unk_rows,
            "missing_special": missing_special,
            "missing_dialect_tags": missing_tags,
            "bad_artifact_protected_count": len(bad_strict),
            "punct_only_protected_count": len(punct_only),
            "nan_counts": nan_counts,
            "dialect_strict_counts": dialect_strict_counts,
            "dialect_expanded_counts": dialect_expanded_counts,
            "adaptive_counts": adaptive_counts,
            "synthetic_assisted_strict_counts": synthetic_assisted_strict_counts,
            "low_strict_after_backfill": low_strict,
            "warnings": warnings_list,
            "hard_failures": failures,
            "passed": len(failures) == 0,
        }

        for k, v in results.items():
            if k not in {
                "hard_failures",
                "dialect_strict_counts",
                "dialect_expanded_counts",
                "nan_counts",
                "adaptive_counts",
                "synthetic_assisted_strict_counts",
            }:
                print(f"{k:60s}: {v}")

        if warnings_list:
            print("\n[WARNINGS]")
            for w in warnings_list:
                print(f"  - {w}")

        if failures:
            print("\n[FAILED]")
            for f in failures:
                print(f"  - {f}")
            raise AssertionError("Tokenizer Portion 3 tests failed.")

        print("\n[OK] Portion 3 tests passed.")
        return results

    def export_all(
        self,
        df: pd.DataFrame,
        token_stats: pd.DataFrame,
        dialect_scores: pd.DataFrame,
        freq_by_dialect: pd.DataFrame,
        core_strict: pd.DataFrame,
        core_expanded: pd.DataFrame,
        strict_by_dialect: Dict[str, pd.DataFrame],
        expanded_by_dialect: Dict[str, pd.DataFrame],
        protected: Dict[str, List[str]],
        coverage: pd.DataFrame,
        test_results: Dict[str, Any],
    ) -> None:

        print("\n" + "=" * 100)
        print("STEP 7: EXPORT PORTION 3 ARTIFACTS")
        print("=" * 100)

        token_stats_path = self.output_dir / "token_statistics.csv"
        dialect_scores_path = self.output_dir / "token_dialect_scores.csv"
        freq_path = self.output_dir / "token_frequency_by_dialect.csv"
        core_strict_path = self.output_dir / "core_tokens_strict.csv"
        core_expanded_path = self.output_dir / "core_tokens_expanded.csv"
        coverage_path = self.output_dir / "protected_token_coverage_by_dialect.csv"

        token_stats.to_csv(token_stats_path, index=False, encoding="utf-8")
        dialect_scores.to_csv(dialect_scores_path, index=False, encoding="utf-8")
        freq_by_dialect.to_csv(freq_path, index=False, encoding="utf-8")
        core_strict.to_csv(core_strict_path, index=False, encoding="utf-8")
        core_expanded.to_csv(core_expanded_path, index=False, encoding="utf-8")
        coverage.to_csv(coverage_path, index=False, encoding="utf-8")

        write_lines(self.output_dir / "special_tokens.txt", protected["special"])
        write_lines(self.output_dir / "dialect_tags.txt", protected["dialect_tags"])
        write_lines(self.output_dir / "core_tokens_strict.txt", core_strict["token"].tolist())
        write_lines(self.output_dir / "core_tokens_expanded.txt", core_expanded["token"].tolist())
        write_lines(self.output_dir / "protected_tokens_strict.txt", protected["protected_strict"])
        write_lines(self.output_dir / "protected_tokens_expanded.txt", protected["protected_expanded"])

        for d in ALL_DIALECTS:
            strict_path = self.output_dir / f"dialect_tokens_{d}_strict.csv"
            expanded_path = self.output_dir / f"dialect_tokens_{d}_expanded.csv"

            strict_by_dialect[d].to_csv(strict_path, index=False, encoding="utf-8")
            expanded_by_dialect[d].to_csv(expanded_path, index=False, encoding="utf-8")

            write_lines(self.output_dir / f"dialect_tokens_{d}_strict.txt", strict_by_dialect[d]["token"].tolist())
            write_lines(self.output_dir / f"dialect_tokens_{d}_expanded.txt", expanded_by_dialect[d]["token"].tolist())

        synthetic_review = dialect_scores[
            (dialect_scores["is_protectable"] == True)
            & (
                (dialect_scores["support_type"].isin(["synthetic_only", "synthetic_dominant"]))
                | (dialect_scores["dialect_synthetic_fraction"] >= 0.75)
            )
        ].copy()

        synthetic_review = synthetic_review.sort_values(
            ["association_score", "count_td"],
            ascending=False,
            kind="mergesort",
        )

        synthetic_review.to_csv(
            self.output_dir / "synthetic_assisted_tokens_for_review.csv",
            index=False,
            encoding="utf-8",
        )

        adaptive_rows = []
        for d in ALL_DIALECTS:
            s = strict_by_dialect[d].copy()
            if len(s):
                s = s[s["candidate_tier"].astype(str).str.contains("adaptive|synthetic", regex=True)].copy()
                adaptive_rows.append(s)

        if adaptive_rows:
            adaptive_report = pd.concat(adaptive_rows, axis=0).reset_index(drop=True)
        else:
            adaptive_report = pd.DataFrame()

        adaptive_report.to_csv(
            self.output_dir / "adaptive_synthetic_assisted_strict_tokens.csv",
            index=False,
            encoding="utf-8",
        )

        mining_reports = []
        for d in ALL_DIALECTS:
            top = dialect_scores[
                (dialect_scores["dialect"] == d)
                & (dialect_scores["is_protectable"] == True)
            ].copy()

            top = top.sort_values(
                ["association_score", "count_td", "real_count_td"],
                ascending=False,
                kind="mergesort",
            ).head(self.cfg.top_k_export_per_dialect)

            mining_reports.append(top)

        token_mining_report = pd.concat(mining_reports, axis=0).reset_index(drop=True)
        token_mining_report.to_csv(
            self.output_dir / "token_mining_report_top_by_dialect.csv",
            index=False,
            encoding="utf-8",
        )

        strict_candidate_tier_counts = {}
        for d in ALL_DIALECTS:
            if len(strict_by_dialect[d]) == 0:
                strict_candidate_tier_counts[d] = {}
            else:
                strict_candidate_tier_counts[d] = strict_by_dialect[d]["candidate_tier"].value_counts().to_dict()

        summary = {
            "portion": "tokenizer_p3_fixed_dialect_aware_token_mining_protected_vocab",
            "seed": SEED,
            "config": asdict(self.cfg),
            "input_path": str(INPUT_PATH),
            "rows_input": int(len(df)),
            "dialects": ALL_DIALECTS,
            "total_lexical_tokens": int(token_stats["total_count"].sum()),
            "unique_lexical_tokens": int(len(token_stats)),
            "core_strict_count": int(len(core_strict)),
            "core_expanded_count": int(len(core_expanded)),
            "dialect_strict_counts": {d: int(len(strict_by_dialect[d])) for d in ALL_DIALECTS},
            "dialect_expanded_counts": {d: int(len(expanded_by_dialect[d])) for d in ALL_DIALECTS},
            "strict_candidate_tier_counts": strict_candidate_tier_counts,
            "protected_strict_total": int(len(protected["protected_strict"])),
            "protected_expanded_total": int(len(protected["protected_expanded"])),
            "protected_strict_lexical_count": int(len(protected["strict_lexical"])),
            "protected_expanded_lexical_count": int(len(protected["expanded_lexical"])),
            "coverage_mean": {
                "strict_token_coverage": float(coverage["strict_token_coverage"].mean()),
                "expanded_token_coverage": float(coverage["expanded_token_coverage"].mean()),
                "strict_type_coverage": float(coverage["strict_type_coverage"].mean()),
                "expanded_type_coverage": float(coverage["expanded_type_coverage"].mean()),
            },
            "test_results": test_results,
            "outputs": {
                "token_statistics": str(token_stats_path),
                "token_dialect_scores": str(dialect_scores_path),
                "token_frequency_by_dialect": str(freq_path),
                "core_tokens_strict": str(core_strict_path),
                "core_tokens_expanded": str(core_expanded_path),
                "protected_tokens_strict": str(self.output_dir / "protected_tokens_strict.txt"),
                "protected_tokens_expanded": str(self.output_dir / "protected_tokens_expanded.txt"),
                "coverage": str(coverage_path),
                "synthetic_review": str(self.output_dir / "synthetic_assisted_tokens_for_review.csv"),
                "adaptive_report": str(self.output_dir / "adaptive_synthetic_assisted_strict_tokens.csv"),
            },
        }

        with open(self.output_dir / "token_mining_summary.json", "w", encoding="utf-8") as f:
            json.dump(summary, f, ensure_ascii=False, indent=2)

        print(f"Saved: {token_stats_path}")
        print(f"Saved: {dialect_scores_path}")
        print(f"Saved: {freq_path}")
        print(f"Saved: {core_strict_path}")
        print(f"Saved: {core_expanded_path}")
        print(f"Saved: {coverage_path}")
        print(f"Saved: {self.output_dir / 'protected_tokens_strict.txt'}")
        print(f"Saved: {self.output_dir / 'protected_tokens_expanded.txt'}")
        print(f"Saved: {self.output_dir / 'synthetic_assisted_tokens_for_review.csv'}")
        print(f"Saved: {self.output_dir / 'adaptive_synthetic_assisted_strict_tokens.csv'}")
        print(f"Saved: {self.output_dir / 'token_mining_summary.json'}")

    def save_plots(
        self,
        token_stats: pd.DataFrame,
        strict_by_dialect: Dict[str, pd.DataFrame],
        expanded_by_dialect: Dict[str, pd.DataFrame],
        coverage: pd.DataFrame,
    ) -> None:

        if not self.cfg.save_plots:
            return

        print("\n" + "=" * 100)
        print("STEP 8: SAVE DIAGNOSTIC PLOTS")
        print("=" * 100)

        dialects = ALL_DIALECTS
        strict_counts = [len(strict_by_dialect[d]) for d in dialects]
        expanded_counts = [len(expanded_by_dialect[d]) for d in dialects]

        x = np.arange(len(dialects))
        width = 0.38

        plt.figure(figsize=(15, 6))
        plt.bar(x - width / 2, strict_counts, width, label="strict")
        plt.bar(x + width / 2, expanded_counts, width, label="expanded")
        plt.title("Dialect-Specific Protected Token Candidate Counts")
        plt.xlabel("Dialect")
        plt.ylabel("Tokens")
        plt.xticks(x, dialects, rotation=45)
        plt.legend()
        plt.tight_layout()
        p = self.output_dir / "dialect_candidate_counts.png"
        plt.savefig(p, dpi=200)
        plt.close()
        print(f"Saved: {p}")

        plt.figure(figsize=(15, 6))
        plt.bar(x - width / 2, coverage["strict_token_coverage"], width, label="strict")
        plt.bar(x + width / 2, coverage["expanded_token_coverage"], width, label="expanded")
        plt.title("Protected Lexical Token Coverage by Dialect")
        plt.xlabel("Dialect")
        plt.ylabel("Token coverage")
        plt.xticks(x, coverage["dialect"], rotation=45)
        plt.ylim(0, 1)
        plt.legend()
        plt.tight_layout()
        p = self.output_dir / "protected_token_coverage_by_dialect.png"
        plt.savefig(p, dpi=200)
        plt.close()
        print(f"Saved: {p}")

        counts = token_stats["total_count"].values
        plt.figure(figsize=(10, 6))
        plt.hist(np.log10(counts + 1), bins=60)
        plt.title("Log10 Token Frequency Distribution")
        plt.xlabel("log10(total_count + 1)")
        plt.ylabel("Token types")
        plt.tight_layout()
        p = self.output_dir / "token_frequency_distribution_log10.png"
        plt.savefig(p, dpi=200)
        plt.close()
        print(f"Saved: {p}")

    def checkpoint(
        self,
        token_stats: pd.DataFrame,
        core_strict: pd.DataFrame,
        core_expanded: pd.DataFrame,
        strict_by_dialect: Dict[str, pd.DataFrame],
        expanded_by_dialect: Dict[str, pd.DataFrame],
        protected: Dict[str, List[str]],
        coverage: pd.DataFrame,
    ) -> None:

        print("\n" + "=" * 100)
        print("CHECKPOINT: PORTION 3 TOKEN MINING")
        print("=" * 100)

        print(f"Unique lexical tokens      : {len(token_stats):,}")
        print(f"Core strict tokens         : {len(core_strict):,}")
        print(f"Core expanded tokens       : {len(core_expanded):,}")
        print(f"Protected strict total     : {len(protected['protected_strict']):,}")
        print(f"Protected expanded total   : {len(protected['protected_expanded']):,}")

        print("\nStrict dialect-specific token counts:")
        for d in ALL_DIALECTS:
            top = strict_by_dialect[d]["token"].head(12).tolist() if len(strict_by_dialect[d]) else []
            tier_counts = strict_by_dialect[d]["candidate_tier"].value_counts().to_dict() if len(strict_by_dialect[d]) else {}
            print(f"{d:3s}: {len(strict_by_dialect[d]):4d} | tiers={tier_counts} | top={top}")

        print("\nExpanded dialect-specific token counts:")
        for d in ALL_DIALECTS:
            print(f"{d:3s}: {len(expanded_by_dialect[d]):4d}")

        print("\nTop core strict tokens:")
        cols = [
            "token",
            "total_count",
            "real_count",
            "dialect_coverage_min",
            "max_dialect",
            "max_dialect_purity",
            "core_score",
        ]
        print(core_strict[cols].head(30).to_string(index=False))

        print("\nProtected coverage by dialect:")
        print(
            coverage[
                [
                    "dialect",
                    "strict_token_coverage",
                    "expanded_token_coverage",
                    "strict_type_coverage",
                    "expanded_type_coverage",
                ]
            ].round(4).to_string(index=False)
        )

        print("\nUse in Portion 4:")
        print(P3_DIR / "protected_tokens_strict.txt")
        print(P3_DIR / "protected_tokens_expanded.txt")

        print("\nReview synthetic/adaptive candidates before final paper freeze:")
        print(P3_DIR / "adaptive_synthetic_assisted_strict_tokens.csv")
        print(P3_DIR / "synthetic_assisted_tokens_for_review.csv")

        print("\n✅ TOKENIZER PORTION 3 COMPLETE")
        print(f"Output directory: {P3_DIR}")


# ============================================================
# 5. Execute Portion 3
# ============================================================

miner = DialectAwareTokenMiner(CFG)

df_p3 = miner.load()

counter_builder = TokenCounterBuilder(CFG)
counter_builder.build(df_p3)

token_stats_df, dialect_scores_df, freq_by_dialect_df = counter_builder.to_dataframes()

core_strict_df, core_expanded_df = miner.mine_core_tokens(token_stats_df)

strict_by_dialect, expanded_by_dialect = miner.mine_dialect_tokens(dialect_scores_df)

protected_lists = miner.build_protected_lists(
    core_strict=core_strict_df,
    core_expanded=core_expanded_df,
    strict_by_dialect=strict_by_dialect,
    expanded_by_dialect=expanded_by_dialect,
)

coverage_df = miner.compute_protected_coverage(df_p3, protected_lists)

test_results_p3 = miner.run_tests(
    df=df_p3,
    token_stats=token_stats_df,
    dialect_scores=dialect_scores_df,
    core_strict=core_strict_df,
    strict_by_dialect=strict_by_dialect,
    expanded_by_dialect=expanded_by_dialect,
    protected=protected_lists,
)

miner.export_all(
    df=df_p3,
    token_stats=token_stats_df,
    dialect_scores=dialect_scores_df,
    freq_by_dialect=freq_by_dialect_df,
    core_strict=core_strict_df,
    core_expanded=core_expanded_df,
    strict_by_dialect=strict_by_dialect,
    expanded_by_dialect=expanded_by_dialect,
    protected=protected_lists,
    coverage=coverage_df,
    test_results=test_results_p3,
)

miner.save_plots(
    token_stats=token_stats_df,
    strict_by_dialect=strict_by_dialect,
    expanded_by_dialect=expanded_by_dialect,
    coverage=coverage_df,
)

miner.checkpoint(
    token_stats=token_stats_df,
    core_strict=core_strict_df,
    core_expanded=core_expanded_df,
    strict_by_dialect=strict_by_dialect,
    expanded_by_dialect=expanded_by_dialect,
    protected=protected_lists,
    coverage=coverage_df,
)

TOKENIZER PORTION 3 — FIXED FINAL: DIALECT-AWARE TOKEN MINING
Input path : artifacts/tokenizer_p21/tokenizer_train_clean_p21_recommended.csv
Output dir : artifacts/tokenizer_p3
Dialects   : ['BAR', 'CHI', 'KHU', 'KIS', 'MYM', 'NAR', 'NOA', 'NSD', 'RAJ', 'RAN', 'STD', 'SYL', 'TAN']
GPU needed : NO — CPU is sufficient for this portion.
[LOAD] Rows loaded for Portion 3: 121,503
[LOAD] Dialect counts:
dialect
BAR     8466
CHI    14592
KHU     8126
KIS     9899
MYM     8993
NAR     8952
NOA     8663
NSD     9486
RAJ     8955
RAN     8449
STD     9011
SYL     9558
TAN     8353

STEP 1: BUILD TOKEN COUNTERS
[COUNT] Processed 25,000 rows...
[COUNT] Processed 50,000 rows...
[COUNT] Processed 75,000 rows...
[COUNT] Processed 100,000 rows...
[COUNT] Processed rows        : 121,503
[COUNT] Unique lexical tokens : 48,987
[COUNT] Total lexical tokens  : 718,201

STEP 2: CONVERT COUNTERS TO DATAFRAMES
[DF] token_statistics rows      : 48,987
[DF] token_dialect_scores rows  : 91,592
[DF] token_frequen

## portion-4

In [6]:
# ============================================================
# TOKENIZER PORTION 4 — DETERMINISTIC WORDPIECE FINAL VERSION
# Fast Non-Iterative Dialect-Aware Tokenizer Construction
# Bangla Dialect-Aware SLM/LLM Tokenizer
# ============================================================
#
# Why this version:
#   - SentencePiece BPE/Unigram timed out.
#   - HF BPE can also be slow/error-prone on this setup.
#   - This builds a deterministic WordPiece tokenizer without iterative training.
#   - It preserves dialect tags and Portion 3 protected lexical tokens.
#   - It guarantees Bengali character fallback through base + continuation chars.
#
# GPU:
#   Not needed. Do not use T4 for this portion.
#
# Inputs:
#   artifacts/tokenizer_p21/tokenizer_train_clean_p21_recommended.csv
#   artifacts/tokenizer_p2/tokenizer_eval_holdout_clean.csv
#   artifacts/tokenizer_p3/protected_tokens_strict.txt
#   artifacts/tokenizer_p3/protected_tokens_expanded.txt
#   artifacts/tokenizer_p3/special_tokens.txt
#   artifacts/tokenizer_p3/dialect_tags.txt
#
# Outputs:
#   artifacts/tokenizer_p4_wordpiece/
#     training_corpus/
#     candidates/
#     validation/
#     final_tokenizer/
#     tokenizer_candidate_comparison.csv
#     tokenizer_selection_summary.json
#     p4_test_results.json
# ============================================================

from __future__ import annotations

import os

os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"

import re
import gc
import json
import time
import shutil
import random
import hashlib
import warnings
import unicodedata
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


# ============================================================
# 0. Reproducibility
# ============================================================

SEED = 42

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)


# ============================================================
# 1. Dependency
# ============================================================

def require_package(import_name: str, pip_name: Optional[str] = None) -> None:
    try:
        __import__(import_name)
    except Exception as e:
        pip_name = pip_name or import_name
        raise ImportError(
            f"Missing package: {import_name}. Run this first:\n"
            f"!pip install -q {pip_name}"
        ) from e

require_package("tokenizers")

from tokenizers import Tokenizer, AddedToken
from tokenizers import models, pre_tokenizers, decoders, normalizers


# ============================================================
# 2. Config
# ============================================================

@dataclass(frozen=True)
class Portion4WPConfig:
    train_path: str = "artifacts/tokenizer_p21/tokenizer_train_clean_p21_recommended.csv"
    holdout_path: str = "artifacts/tokenizer_p2/tokenizer_eval_holdout_clean.csv"
    p3_dir: str = "artifacts/tokenizer_p3"
    output_dir: str = "artifacts/tokenizer_p4_wordpiece"

    strict_protected_file: str = "protected_tokens_strict.txt"
    expanded_protected_file: str = "protected_tokens_expanded.txt"
    special_tokens_file: str = "special_tokens.txt"
    dialect_tags_file: str = "dialect_tags.txt"

    # Balanced corpus/vocab statistics.
    max_rows_per_dialect: int = 8500
    prefix_dialect_tag: bool = True

    # Synthetic control.
    max_synthetic_ratio_default: float = 0.65
    max_synthetic_ratio_low_real: float = 0.88
    low_real_threshold: int = 2000
    max_synthetic_ratio_std: float = 0.55

    # Deterministic vocab candidates.
    candidate_vocab_sizes: Tuple[int, ...] = (32000, 48000)

    # Full-word lexical token selection.
    min_whole_token_count: int = 2
    max_whole_token_chars: int = 40
    whole_token_budget_ratio: float = 0.70

    # Subword mining.
    min_subword_count: int = 3
    min_subword_len: int = 2
    max_subword_len: int = 7
    max_word_chars_for_subword_mining: int = 32
    max_subword_candidates_keep: int = 80000

    # WordPiece.
    wordpiece_prefix: str = "##"
    max_input_chars_per_word: int = 128

    # Validation.
    validation_train_sample_per_dialect: int = 300
    max_symbols_to_validate: int = 50000

    # Hard constraints.
    max_allowed_unk_rate: float = 0.001
    min_special_vocab_rate: float = 1.0
    min_dialect_tag_vocab_rate: float = 1.0
    min_dialect_tag_atomic_rate: float = 1.0
    min_strict_protected_vocab_rate: float = 1.0
    min_strict_protected_atomic_rate: float = 0.995

    # Selection.
    prefer_smaller_vocab_if_close: bool = True
    close_score_margin: float = 1.0

    save_plots: bool = True

    pad_token: str = "<pad>"
    unk_token: str = "<unk>"
    bos_token: str = "<bos>"
    eos_token: str = "<eos>"

    target_dialects_10: Tuple[str, ...] = (
        "BAR", "CHI", "KHU", "MYM", "NAR",
        "NOA", "RAJ", "RAN", "STD", "SYL",
    )
    auxiliary_dialects: Tuple[str, ...] = ("KIS", "TAN", "NSD")


CFG = Portion4WPConfig()

TRAIN_PATH = Path(CFG.train_path)
HOLDOUT_PATH = Path(CFG.holdout_path)
P3_DIR = Path(CFG.p3_dir)
P4_DIR = Path(CFG.output_dir)

CORPUS_DIR = P4_DIR / "training_corpus"
CANDIDATE_DIR = P4_DIR / "candidates"
VALIDATION_DIR = P4_DIR / "validation"
FINAL_DIR = P4_DIR / "final_tokenizer"

for p in [P4_DIR, CORPUS_DIR, CANDIDATE_DIR, VALIDATION_DIR]:
    p.mkdir(parents=True, exist_ok=True)

ALL_DIALECTS = sorted(set(CFG.target_dialects_10 + CFG.auxiliary_dialects))
SPECIAL_TOKENS = [CFG.pad_token, CFG.unk_token, CFG.bos_token, CFG.eos_token]
DIALECT_TAGS = [f"<dial:{d}>" for d in ALL_DIALECTS]
SPECIAL_PLUS_TAGS = SPECIAL_TOKENS + DIALECT_TAGS

print("=" * 100)
print("TOKENIZER PORTION 4 — DETERMINISTIC WORDPIECE FINAL VERSION")
print("=" * 100)
print(f"Train input : {TRAIN_PATH}")
print(f"Holdout     : {HOLDOUT_PATH}")
print(f"P3 dir      : {P3_DIR}")
print(f"Output dir  : {P4_DIR}")
print(f"Dialects    : {ALL_DIALECTS}")
print("GPU needed  : NO — deterministic CPU vocabulary construction.")
print("=" * 100)

assert TRAIN_PATH.exists(), f"Missing train file: {TRAIN_PATH}"
assert P3_DIR.exists(), f"Missing Portion 3 directory: {P3_DIR}"


# ============================================================
# 3. Utilities
# ============================================================

TOKEN_RE = re.compile(
    r"[\u0980-\u09FF]+|[A-Za-z]+(?:'[A-Za-z]+)?|\d+(?:[.,]\d+)?|[^\s]",
    flags=re.UNICODE,
)

LEXICAL_RE = re.compile(
    r"[\u0980-\u09FF]+|[A-Za-z]+(?:'[A-Za-z]+)?|\d+(?:[.,]\d+)?",
    flags=re.UNICODE,
)

def now() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def safe_bool(x: Any) -> bool:
    if isinstance(x, bool):
        return x
    if pd.isna(x):
        return False
    return str(x).strip().lower() in {"true", "1", "yes", "y"}

def stable_hash(text: str) -> str:
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()

def stable_int(text: str, mod: int = 100_000) -> int:
    return int(stable_hash(text)[:8], 16) % mod

def clean_line(text: Any) -> str:
    if pd.isna(text):
        return ""
    s = str(text)
    s = unicodedata.normalize("NFC", s)
    s = re.sub(r"[\x00-\x1f\x7f-\x9f]", "", s)
    s = re.sub(r"[\u200b\u200c\u200d\u200e\u200f\ufeff]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def read_lines(path: Path) -> List[str]:
    assert path.exists(), f"Missing file: {path}"
    with open(path, "r", encoding="utf-8") as f:
        return [x.strip() for x in f if x.strip()]

def write_lines(path: Path, lines: List[str]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for x in lines:
            x = str(x).strip()
            if x:
                f.write(x + "\n")

def dedup_keep_order(xs: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in xs:
        x = str(x).strip()
        if not x:
            continue
        if x in seen:
            continue
        seen.add(x)
        out.append(x)
    return out

def make_dialect_tag(dialect: str) -> str:
    return f"<dial:{dialect}>"

def is_dialect_tag(token: str) -> bool:
    return bool(re.fullmatch(r"<dial:[A-Z]{3}>", str(token)))

def is_special_token(token: str) -> bool:
    return token in set(SPECIAL_TOKENS)

def is_bangla_char(ch: str) -> bool:
    return "\u0980" <= ch <= "\u09FF"

def has_bangla(text: str) -> bool:
    return any(is_bangla_char(ch) for ch in str(text))

def is_numeric_like(text: str) -> bool:
    return bool(re.fullmatch(r"\d+(?:[.,]\d+)?", str(text)))

def is_punctuation_or_symbol(ch: str) -> bool:
    cat = unicodedata.category(ch)
    return cat.startswith("P") or cat.startswith("S")

def lexical_tokenize(text: Any) -> List[str]:
    if pd.isna(text):
        return []
    s = unicodedata.normalize("NFC", str(text))
    return [m.group(0) for m in LEXICAL_RE.finditer(s)]

def mixed_tokenize(text: Any) -> List[str]:
    if pd.isna(text):
        return []
    s = unicodedata.normalize("NFC", str(text))
    return [m.group(0) for m in TOKEN_RE.finditer(s)]

def make_added_token(token: str, special: bool) -> AddedToken:
    try:
        return AddedToken(
            token,
            single_word=False,
            lstrip=False,
            rstrip=False,
            normalized=True,
            special=special,
        )
    except TypeError:
        return AddedToken(token)

def make_wordpiece_decoder():
    try:
        return decoders.WordPiece(prefix=CFG.wordpiece_prefix, cleanup=False)
    except TypeError:
        return decoders.WordPiece(prefix=CFG.wordpiece_prefix)


# ============================================================
# 4. Load Data and Protected Symbols
# ============================================================

def load_train_minimal() -> pd.DataFrame:
    print("\n" + "=" * 100)
    print("STEP 0: LOAD MINIMAL TRAIN DATA")
    print("=" * 100)

    needed = {
        "text",
        "dialect",
        "is_synthetic",
        "split_original",
        "eligible_for_tokenizer_train",
    }

    df = pd.read_csv(TRAIN_PATH, usecols=lambda c: c in needed)

    for col in ["is_synthetic", "eligible_for_tokenizer_train"]:
        if col in df.columns:
            df[col] = df[col].map(safe_bool)

    df = df[df["eligible_for_tokenizer_train"] == True].copy()
    df = df[~df["split_original"].isin(["validation", "test"])].copy()
    df = df[df["dialect"].isin(ALL_DIALECTS)].copy()
    df = df[df["text"].notna()].copy()
    df["text"] = df["text"].map(clean_line)
    df = df[df["text"].str.len() > 0].copy()

    df = df.reset_index(drop=True)
    df["row_id_p4"] = np.arange(len(df), dtype=np.int64)

    print(f"[{now()}] Train rows loaded: {len(df):,}")
    print(df["dialect"].value_counts().sort_index().to_string())

    return df

def load_holdout_minimal() -> pd.DataFrame:
    print("\n" + "=" * 100)
    print("STEP 0.1: LOAD HOLDOUT DATA")
    print("=" * 100)

    if not HOLDOUT_PATH.exists():
        print(f"[WARN] Missing holdout: {HOLDOUT_PATH}")
        return pd.DataFrame(columns=["text", "dialect", "is_synthetic", "split_original"])

    needed = {"text", "dialect", "is_synthetic", "split_original"}
    df = pd.read_csv(HOLDOUT_PATH, usecols=lambda c: c in needed)

    if "is_synthetic" in df.columns:
        df["is_synthetic"] = df["is_synthetic"].map(safe_bool)

    df = df[df["dialect"].isin(ALL_DIALECTS)].copy()
    df = df[df["text"].notna()].copy()
    df["text"] = df["text"].map(clean_line)
    df = df[df["text"].str.len() > 0].copy()
    df = df.reset_index(drop=True)

    print(f"[{now()}] Holdout rows loaded: {len(df):,}")
    if len(df):
        print(df["dialect"].value_counts().sort_index().to_string())

    return df

def load_protected_symbols() -> Dict[str, List[str]]:
    print("\n" + "=" * 100)
    print("STEP 0.2: LOAD PROTECTED SYMBOLS")
    print("=" * 100)

    strict_all = read_lines(P3_DIR / CFG.strict_protected_file)
    expanded_all = read_lines(P3_DIR / CFG.expanded_protected_file)

    special = read_lines(P3_DIR / CFG.special_tokens_file) if (P3_DIR / CFG.special_tokens_file).exists() else SPECIAL_TOKENS
    tags = read_lines(P3_DIR / CFG.dialect_tags_file) if (P3_DIR / CFG.dialect_tags_file).exists() else DIALECT_TAGS

    special = dedup_keep_order(special)
    tags = dedup_keep_order(tags)

    reserved = set(special + tags)

    strict_lexical = dedup_keep_order([
        x for x in strict_all
        if x not in reserved and not is_special_token(x) and not is_dialect_tag(x)
    ])

    expanded_lexical = dedup_keep_order([
        x for x in expanded_all
        if x not in reserved and not is_special_token(x) and not is_dialect_tag(x)
    ])

    print(f"Special tokens           : {len(special)}")
    print(f"Dialect tags             : {len(tags)}")
    print(f"Strict lexical protected : {len(strict_lexical):,}")
    print(f"Expanded lexical         : {len(expanded_lexical):,}")

    return {
        "special": special,
        "dialect_tags": tags,
        "strict_lexical": strict_lexical,
        "expanded_lexical": expanded_lexical,
        "strict_all": strict_all,
        "expanded_all": expanded_all,
    }

df_train = load_train_minimal()
df_holdout = load_holdout_minimal()
protected = load_protected_symbols()


# ============================================================
# 5. Balanced Corpus Sampler
# ============================================================

class BalancedCorpusSampler:
    def __init__(self, cfg: Portion4WPConfig):
        self.cfg = cfg

    def _synthetic_cap(self, dialect: str, real_count: int) -> float:
        if dialect == "STD":
            return self.cfg.max_synthetic_ratio_std
        if real_count < self.cfg.low_real_threshold:
            return self.cfg.max_synthetic_ratio_low_real
        return self.cfg.max_synthetic_ratio_default

    def _sample_one_dialect(self, df: pd.DataFrame, dialect: str) -> Tuple[np.ndarray, Dict[str, Any]]:
        sub = df[df["dialect"] == dialect]
        real = sub[sub["is_synthetic"] == False]
        syn = sub[sub["is_synthetic"] == True]

        real_idx = real.index.to_numpy(dtype=np.int64)
        syn_idx = syn.index.to_numpy(dtype=np.int64)

        real_count = len(real_idx)
        syn_count = len(syn_idx)
        available = real_count + syn_count

        if available == 0:
            return np.array([], dtype=np.int64), {
                "dialect": dialect,
                "available": 0,
                "available_real": 0,
                "available_synthetic": 0,
                "target": 0,
                "sampled_real": 0,
                "sampled_synthetic": 0,
                "sampled_total": 0,
                "synthetic_ratio": 0.0,
            }

        rng = np.random.default_rng(SEED + stable_int(dialect))
        target = min(self.cfg.max_rows_per_dialect, available)

        if real_count == 0:
            chosen = rng.choice(syn_idx, size=min(target, syn_count), replace=False)

        elif syn_count == 0:
            chosen = rng.choice(real_idx, size=min(target, real_count), replace=False)

        else:
            cap = self._synthetic_cap(dialect, real_count)
            max_syn_n = int(round(target * cap))

            target_syn = min(max_syn_n, syn_count)
            target_real = min(target - target_syn, real_count)

            chosen_real = rng.choice(real_idx, size=target_real, replace=False) if target_real > 0 else np.array([], dtype=np.int64)
            chosen_syn = rng.choice(syn_idx, size=target_syn, replace=False) if target_syn > 0 else np.array([], dtype=np.int64)

            chosen = np.concatenate([chosen_real, chosen_syn])

            if len(chosen) < target:
                chosen_set = set(chosen.tolist())
                unused_real = np.array([x for x in real_idx if x not in chosen_set], dtype=np.int64)
                unused_syn = np.array([x for x in syn_idx if x not in chosen_set], dtype=np.int64)

                remaining = target - len(chosen)

                if len(unused_real) > 0 and remaining > 0:
                    take = min(remaining, len(unused_real))
                    extra = rng.choice(unused_real, size=take, replace=False)
                    chosen = np.concatenate([chosen, extra])
                    remaining -= take

                if len(unused_syn) > 0 and remaining > 0:
                    take = min(remaining, len(unused_syn))
                    extra = rng.choice(unused_syn, size=take, replace=False)
                    chosen = np.concatenate([chosen, extra])

        rng.shuffle(chosen)

        sampled_real = int((df.loc[chosen, "is_synthetic"] == False).sum())
        sampled_syn = int((df.loc[chosen, "is_synthetic"] == True).sum())

        info = {
            "dialect": dialect,
            "available": int(available),
            "available_real": int(real_count),
            "available_synthetic": int(syn_count),
            "target": int(target),
            "sampled_real": int(sampled_real),
            "sampled_synthetic": int(sampled_syn),
            "sampled_total": int(len(chosen)),
            "synthetic_ratio": float(sampled_syn / max(1, len(chosen))),
        }

        return chosen.astype(np.int64), info

    def build(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, Path, pd.DataFrame]:
        print("\n" + "=" * 100)
        print("STEP 1: BUILD BALANCED VOCABULARY CORPUS")
        print("=" * 100)

        sampled_indices = []
        plan_rows = []

        for d in ALL_DIALECTS:
            chosen, info = self._sample_one_dialect(df, d)
            sampled_indices.append(chosen)
            plan_rows.append(info)

            print(
                f"[{now()}] {d}: available={info['available']:,}, "
                f"sampled={info['sampled_total']:,}, "
                f"real={info['sampled_real']:,}, "
                f"syn={info['sampled_synthetic']:,}, "
                f"syn_ratio={info['synthetic_ratio']:.3f}"
            )

        sampled_idx = np.concatenate(sampled_indices)
        rng = np.random.default_rng(SEED)
        rng.shuffle(sampled_idx)

        sampled = df.loc[sampled_idx].copy().reset_index(drop=True)

        corpus_path = CORPUS_DIR / "tokenizer_training_corpus_wordpiece.txt"
        meta_path = CORPUS_DIR / "tokenizer_training_corpus_wordpiece_metadata.csv"
        plan_path = CORPUS_DIR / "training_corpus_wordpiece_sampling_plan.csv"

        with open(corpus_path, "w", encoding="utf-8") as f:
            for _, row in sampled.iterrows():
                tag = make_dialect_tag(str(row["dialect"]))
                text = clean_line(row["text"])
                if self.cfg.prefix_dialect_tag:
                    f.write(f"{tag} {text}\n")
                else:
                    f.write(text + "\n")

        sampled[["dialect", "is_synthetic", "split_original", "text"]].to_csv(
            meta_path, index=False, encoding="utf-8"
        )

        plan = pd.DataFrame(plan_rows).sort_values("dialect").reset_index(drop=True)
        plan.to_csv(plan_path, index=False, encoding="utf-8")

        print(f"\nSampled rows: {len(sampled):,}")
        print(f"Saved: {corpus_path}")
        print(f"Saved: {meta_path}")
        print(f"Saved: {plan_path}")

        return sampled, corpus_path, plan

sampler = BalancedCorpusSampler(CFG)
df_vocab_corpus, CORPUS_PATH, corpus_plan = sampler.build(df_train)


# ============================================================
# 6. Deterministic WordPiece Vocabulary Builder
# ============================================================

class DeterministicWordPieceVocabBuilder:
    def __init__(
        self,
        cfg: Portion4WPConfig,
        protected: Dict[str, List[str]],
        train_df: pd.DataFrame,
        holdout_df: pd.DataFrame,
        vocab_df: pd.DataFrame,
    ):
        self.cfg = cfg
        self.protected = protected
        self.train_df = train_df
        self.holdout_df = holdout_df
        self.vocab_df = vocab_df

        self.token_counter = Counter()
        self.token_real_counter = Counter()
        self.token_syn_counter = Counter()
        self.token_dialect_counter = Counter()
        self.subword_counter = Counter()
        self.char_set = set()

    def build_counts(self) -> None:
        print("\n" + "=" * 100)
        print("STEP 2: COUNT WORDS, SUBWORDS, AND CHARACTERS")
        print("=" * 100)

        # Characters from train + holdout + protected.
        for df in [self.train_df, self.holdout_df]:
            if len(df) == 0:
                continue
            for text in df["text"].astype(str).tolist():
                for ch in unicodedata.normalize("NFC", text):
                    if ch.isspace():
                        continue
                    if unicodedata.category(ch).startswith("C"):
                        continue
                    self.char_set.add(ch)

        # Full Bengali block for stronger fallback.
        for code in range(0x0980, 0x0A00):
            ch = chr(code)
            if unicodedata.category(ch).startswith("C"):
                continue
            self.char_set.add(ch)

        # ASCII fallback.
        for ch in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789":
            self.char_set.add(ch)

        for ch in "।॥.,!?;:-_()[]{}'\"/\\@#$%^&*+=|~`":
            self.char_set.add(ch)

        for token in self.protected["strict_all"] + self.protected["expanded_all"] + SPECIAL_PLUS_TAGS:
            for ch in unicodedata.normalize("NFC", str(token)):
                if ch.isspace():
                    continue
                if unicodedata.category(ch).startswith("C"):
                    continue
                self.char_set.add(ch)

        # Token and subword counts from balanced vocab corpus.
        for i, row in self.vocab_df.iterrows():
            text = row["text"]
            dialect = row["dialect"]
            is_syn = safe_bool(row["is_synthetic"])

            toks = lexical_tokenize(text)

            for tok in toks:
                tok = unicodedata.normalize("NFC", tok)
                if not tok:
                    continue

                self.token_counter[tok] += 1
                self.token_dialect_counter[(tok, dialect)] += 1

                if is_syn:
                    self.token_syn_counter[tok] += 1
                else:
                    self.token_real_counter[tok] += 1

            if (i + 1) % 25000 == 0:
                print(f"[COUNT] Processed {i + 1:,} rows...")

        self.mine_subwords()

        print(f"Unique lexical tokens : {len(self.token_counter):,}")
        print(f"Subword candidates    : {len(self.subword_counter):,}")
        print(f"Character fallback set: {len(self.char_set):,}")

    def mine_subwords(self) -> None:
        for tok, count in self.token_counter.items():
            if not has_bangla(tok):
                continue

            tok = unicodedata.normalize("NFC", tok)
            L = len(tok)

            if L < self.cfg.min_subword_len + 1:
                continue
            if L > self.cfg.max_word_chars_for_subword_mining:
                continue

            weight = min(int(count), 100)

            # Prefix pieces without ##.
            for n in range(self.cfg.min_subword_len, min(self.cfg.max_subword_len, L) + 1):
                piece = tok[:n]
                self.subword_counter[piece] += weight

            # Continuation pieces with ##.
            for start in range(1, L):
                max_n = min(self.cfg.max_subword_len, L - start)
                for n in range(self.cfg.min_subword_len, max_n + 1):
                    piece = self.cfg.wordpiece_prefix + tok[start:start+n]
                    self.subword_counter[piece] += weight

    def token_score(self, tok: str, count: int) -> float:
        real = self.token_real_counter[tok]
        syn = self.token_syn_counter[tok]
        real_factor = 0.50 + 0.50 * (real / max(1, count))

        dialect_coverage = sum(
            self.token_dialect_counter[(tok, d)] > 0
            for d in ALL_DIALECTS
        )

        coverage_factor = 0.70 + 0.30 * (dialect_coverage / len(ALL_DIALECTS))

        length_bonus = min(1.25, max(0.85, len(tok) / 8.0))

        if is_numeric_like(tok):
            numeric_penalty = 0.40
        else:
            numeric_penalty = 1.0

        return float(np.log1p(count) * real_factor * coverage_factor * length_bonus * numeric_penalty)

    def build_base_tokens(self) -> List[str]:
        tokens = []

        # 1. Special tokens and dialect tags.
        tokens.extend(SPECIAL_TOKENS)
        tokens.extend(DIALECT_TAGS)

        # 2. Strict protected lexical tokens from Portion 3.
        tokens.extend(self.protected["strict_lexical"])

        # 3. Character fallback: base char and continuation char.
        chars = sorted(self.char_set)
        for ch in chars:
            if ch and not ch.isspace():
                tokens.append(ch)

        for ch in chars:
            if ch and not ch.isspace():
                tokens.append(self.cfg.wordpiece_prefix + ch)

        return dedup_keep_order(tokens)

    def build_vocab(self, vocab_size: int) -> Tuple[Dict[str, int], pd.DataFrame]:
        base_tokens = self.build_base_tokens()

        if len(base_tokens) > vocab_size:
            raise ValueError(
                f"Base required tokens={len(base_tokens):,} exceed vocab_size={vocab_size:,}. "
                f"Increase vocab size."
            )

        remaining = vocab_size - len(base_tokens)

        whole_budget = int(round(remaining * self.cfg.whole_token_budget_ratio))
        subword_budget = remaining - whole_budget

        # Whole lexical words.
        whole_rows = []
        protected_set = set(self.protected["strict_lexical"])

        for tok, count in self.token_counter.items():
            if tok in protected_set:
                continue
            if len(tok) > self.cfg.max_whole_token_chars:
                continue
            if count < self.cfg.min_whole_token_count:
                continue

            score = self.token_score(tok, count)

            whole_rows.append({
                "token": tok,
                "count": int(count),
                "real_count": int(self.token_real_counter[tok]),
                "synthetic_count": int(self.token_syn_counter[tok]),
                "score": score,
                "token_type": "whole_word",
            })

        whole_df = pd.DataFrame(whole_rows)
        if len(whole_df):
            whole_df = whole_df.sort_values(
                ["score", "count", "real_count"],
                ascending=False,
                kind="mergesort",
            ).reset_index(drop=True)
        else:
            whole_df = pd.DataFrame(columns=["token", "count", "real_count", "synthetic_count", "score", "token_type"])

        selected_whole = whole_df.head(whole_budget)["token"].tolist()

        # Subword pieces.
        sub_rows = []
        selected_set = set(base_tokens + selected_whole)

        for piece, count in self.subword_counter.items():
            if piece in selected_set:
                continue
            if count < self.cfg.min_subword_count:
                continue

            # Avoid pure numeric continuation noise.
            raw = piece[len(self.cfg.wordpiece_prefix):] if piece.startswith(self.cfg.wordpiece_prefix) else piece
            if raw.isdigit():
                continue

            score = float(np.log1p(count) * min(1.25, max(0.80, len(raw) / 5.0)))

            sub_rows.append({
                "token": piece,
                "count": int(count),
                "score": score,
                "token_type": "subword",
            })

        sub_df = pd.DataFrame(sub_rows)
        if len(sub_df):
            sub_df = sub_df.sort_values(
                ["score", "count"],
                ascending=False,
                kind="mergesort",
            ).head(self.cfg.max_subword_candidates_keep).reset_index(drop=True)
        else:
            sub_df = pd.DataFrame(columns=["token", "count", "score", "token_type"])

        selected_sub = sub_df.head(subword_budget)["token"].tolist()

        vocab_tokens = dedup_keep_order(base_tokens + selected_whole + selected_sub)

        # If still not full, fill from remaining whole/subword candidates.
        if len(vocab_tokens) < vocab_size:
            current = set(vocab_tokens)

            extra_whole = [t for t in whole_df["token"].tolist() if t not in current]
            needed = vocab_size - len(vocab_tokens)
            vocab_tokens.extend(extra_whole[:needed])
            vocab_tokens = dedup_keep_order(vocab_tokens)

        if len(vocab_tokens) < vocab_size:
            current = set(vocab_tokens)
            extra_sub = [t for t in sub_df["token"].tolist() if t not in current]
            needed = vocab_size - len(vocab_tokens)
            vocab_tokens.extend(extra_sub[:needed])
            vocab_tokens = dedup_keep_order(vocab_tokens)

        vocab_tokens = vocab_tokens[:vocab_size]
        vocab = {tok: i for i, tok in enumerate(vocab_tokens)}

        provenance_rows = []
        base_set = set(base_tokens)
        whole_set = set(selected_whole)
        sub_set = set(selected_sub)

        for tok, idx in vocab.items():
            if tok in SPECIAL_TOKENS:
                typ = "special"
            elif tok in DIALECT_TAGS:
                typ = "dialect_tag"
            elif tok in self.protected["strict_lexical"]:
                typ = "strict_protected"
            elif tok in base_set and tok.startswith(self.cfg.wordpiece_prefix):
                typ = "char_continuation_fallback"
            elif tok in base_set:
                typ = "char_base_fallback"
            elif tok in whole_set:
                typ = "whole_word"
            elif tok in sub_set:
                typ = "subword"
            else:
                typ = "extra"

            provenance_rows.append({
                "token": tok,
                "id": idx,
                "vocab_type": typ,
            })

        prov = pd.DataFrame(provenance_rows)

        return vocab, prov


vocab_builder = DeterministicWordPieceVocabBuilder(
    cfg=CFG,
    protected=protected,
    train_df=df_train,
    holdout_df=df_holdout,
    vocab_df=df_vocab_corpus,
)

vocab_builder.build_counts()


# ============================================================
# 7. Candidate Build
# ============================================================

@dataclass(frozen=True)
class TokenizerCandidate:
    name: str
    backend: str
    model_type: str
    vocab_size: int
    protected_level: str
    description: str


CANDIDATES = [
    TokenizerCandidate(
        name=f"manual_wordpiece_{size//1000}k_strict",
        backend="hf_tokenizers",
        model_type="deterministic_wordpiece",
        vocab_size=size,
        protected_level="strict_in_vocab",
        description=f"Deterministic WordPiece {size} vocab with strict protected tokens and char fallback.",
    )
    for size in CFG.candidate_vocab_sizes
]


class DeterministicTokenizerBuilder:
    def __init__(self, cfg: Portion4WPConfig, protected: Dict[str, List[str]], vocab_builder: DeterministicWordPieceVocabBuilder):
        self.cfg = cfg
        self.protected = protected
        self.vocab_builder = vocab_builder

    def candidate_dir(self, cand: TokenizerCandidate) -> Path:
        p = CANDIDATE_DIR / cand.name
        p.mkdir(parents=True, exist_ok=True)
        return p

    def build_one(self, cand: TokenizerCandidate) -> Dict[str, Any]:
        out_dir = self.candidate_dir(cand)
        tokenizer_json_path = out_dir / "tokenizer.json"
        vocab_txt_path = out_dir / "vocab.txt"
        provenance_path = out_dir / "vocab_provenance.csv"
        meta_path = out_dir / "candidate_metadata.json"

        if tokenizer_json_path.exists() and meta_path.exists():
            print(f"[{now()}] [SKIP] Existing candidate: {cand.name}")
            with open(meta_path, "r", encoding="utf-8") as f:
                return json.load(f)

        print("\n" + "-" * 100)
        print(f"[{now()}] BUILD {cand.name}")
        print(f"vocab_size: {cand.vocab_size:,}")
        print("-" * 100)

        start = time.time()

        vocab, provenance = self.vocab_builder.build_vocab(cand.vocab_size)

        tokenizer = Tokenizer(
            models.WordPiece(
                vocab=vocab,
                unk_token=self.cfg.unk_token,
                continuing_subword_prefix=self.cfg.wordpiece_prefix,
                max_input_chars_per_word=self.cfg.max_input_chars_per_word,
            )
        )

        tokenizer.normalizer = normalizers.Sequence([normalizers.NFC()])
        tokenizer.pre_tokenizer = pre_tokenizers.BertPreTokenizer()
        tokenizer.decoder = make_wordpiece_decoder()

        special_added = [
            make_added_token(t, special=True)
            for t in SPECIAL_PLUS_TAGS
        ]

        tokenizer.add_special_tokens(special_added)

        tokenizer.save(str(tokenizer_json_path), pretty=True)

        # Save vocab in ID order.
        id_to_tok = sorted(vocab.items(), key=lambda x: x[1])
        write_lines(vocab_txt_path, [tok for tok, _ in id_to_tok])

        provenance.to_csv(provenance_path, index=False, encoding="utf-8")

        meta = {
            "candidate": asdict(cand),
            "backend": cand.backend,
            "status": "built",
            "tokenizer_json_path": str(tokenizer_json_path),
            "vocab_txt_path": str(vocab_txt_path),
            "vocab_provenance_path": str(provenance_path),
            "actual_vocab_size": int(tokenizer.get_vocab_size(with_added_tokens=True)),
            "base_model_vocab_size": int(tokenizer.get_vocab_size(with_added_tokens=False)),
            "special_tokens": SPECIAL_TOKENS,
            "dialect_tags": DIALECT_TAGS,
            "wordpiece_prefix": self.cfg.wordpiece_prefix,
            "normalizer": "NFC",
            "pre_tokenizer": "BertPreTokenizer",
            "decoder": "WordPiece",
            "build_seconds": float(time.time() - start),
            "vocab_type_counts": provenance["vocab_type"].value_counts().to_dict(),
        }

        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)

        print(f"[{now()}] DONE {cand.name} in {meta['build_seconds']:.2f}s")
        print(f"Actual vocab size: {meta['actual_vocab_size']:,}")
        print("Vocab type counts:")
        print(pd.Series(meta["vocab_type_counts"]).to_string())

        return meta

    def build_all(self, candidates: List[TokenizerCandidate]) -> List[Dict[str, Any]]:
        print("\n" + "=" * 100)
        print("STEP 3: BUILD DETERMINISTIC WORDPIECE CANDIDATES")
        print("=" * 100)

        metadata = []

        for cand in candidates:
            try:
                metadata.append(self.build_one(cand))
            except Exception as e:
                print(f"[FAILED] {cand.name}: {repr(e)}")
                fail_dir = self.candidate_dir(cand)
                with open(fail_dir / "candidate_metadata_failed.json", "w", encoding="utf-8") as f:
                    json.dump(
                        {
                            "candidate": asdict(cand),
                            "backend": cand.backend,
                            "status": "failed",
                            "error": repr(e),
                        },
                        f,
                        ensure_ascii=False,
                        indent=2,
                    )
            gc.collect()

        if not metadata:
            raise RuntimeError("No deterministic WordPiece tokenizer candidate built successfully.")

        return metadata


builder = DeterministicTokenizerBuilder(CFG, protected, vocab_builder)
candidate_metadata = builder.build_all(CANDIDATES)


# ============================================================
# 8. Tokenizer Wrapper
# ============================================================

class LoadedWPTokenizer:
    def __init__(self, meta: Dict[str, Any]):
        self.meta = meta
        self.name = meta["candidate"]["name"]
        self.backend = meta["backend"]
        self.model_type = meta["candidate"]["model_type"]
        self.protected_level = meta["candidate"]["protected_level"]
        self.tokenizer = Tokenizer.from_file(meta["tokenizer_json_path"])

    def vocab_size(self) -> int:
        return int(self.tokenizer.get_vocab_size(with_added_tokens=True))

    def encode_ids(self, text: str) -> List[int]:
        return list(self.tokenizer.encode(str(text)).ids)

    def encode_tokens(self, text: str) -> List[str]:
        return list(self.tokenizer.encode(str(text)).tokens)

    def token_to_id(self, token: str) -> Optional[int]:
        return self.tokenizer.token_to_id(token)

    def vocab_contains(self, token: str) -> bool:
        return self.token_to_id(token) is not None

    def unk_id(self) -> int:
        idx = self.tokenizer.token_to_id(CFG.unk_token)
        assert idx is not None
        return int(idx)

    def decode_ids(self, ids: List[int], skip_special_tokens: bool = False) -> str:
        return self.tokenizer.decode(ids, skip_special_tokens=skip_special_tokens)


loaded_tokenizers = {
    meta["candidate"]["name"]: LoadedWPTokenizer(meta)
    for meta in candidate_metadata
    if meta.get("status") == "built"
}

print("\nLoaded candidates:")
for name, tok in loaded_tokenizers.items():
    print(f"{name}: vocab={tok.vocab_size():,}, model_type={tok.model_type}")


# ============================================================
# 9. Validation Set
# ============================================================

def build_validation_set(train: pd.DataFrame, holdout: pd.DataFrame) -> pd.DataFrame:
    print("\n" + "=" * 100)
    print("STEP 4: BUILD VALIDATION SET")
    print("=" * 100)

    parts = []

    if len(holdout):
        h = holdout.copy()
        h["validation_origin"] = "real_holdout"
        parts.append(h)

    samples = []
    for d in ALL_DIALECTS:
        sub = train[train["dialect"] == d]
        n = min(CFG.validation_train_sample_per_dialect, len(sub))
        if n > 0:
            samples.append(sub.sample(n=n, random_state=SEED + stable_int(d)))

    train_sample = pd.concat(samples, axis=0).reset_index(drop=True)
    train_sample["validation_origin"] = "train_sample"
    parts.append(train_sample)

    val = pd.concat(parts, axis=0).reset_index(drop=True)

    val["validation_text"] = val.apply(
        lambda r: f"{make_dialect_tag(r['dialect'])} {clean_line(r['text'])}",
        axis=1,
    )

    path = VALIDATION_DIR / "tokenizer_validation_set.csv"
    val.to_csv(path, index=False, encoding="utf-8")

    print(f"Validation rows: {len(val):,}")
    print(val.groupby(["validation_origin", "dialect"]).size().to_string())
    print(f"Saved: {path}")

    return val

df_validation = build_validation_set(df_train, df_holdout)


# ============================================================
# 10. Validation
# ============================================================

class WPTokenizerValidator:
    def __init__(
        self,
        cfg: Portion4WPConfig,
        protected: Dict[str, List[str]],
        tokenizers: Dict[str, LoadedWPTokenizer],
        validation_df: pd.DataFrame,
    ):
        self.cfg = cfg
        self.protected = protected
        self.tokenizers = tokenizers
        self.validation_df = validation_df

    def symbol_presence_report(
        self,
        tok: LoadedWPTokenizer,
        tokens: List[str],
        max_n: Optional[int] = None,
    ) -> Tuple[float, float, pd.DataFrame]:

        tokens = dedup_keep_order(tokens)

        if max_n is not None and len(tokens) > max_n:
            rng = random.Random(SEED)
            tokens = rng.sample(tokens, k=max_n)

        rows = []

        for t in tokens:
            token_id = tok.token_to_id(t)
            enc_tokens = tok.encode_tokens(t)
            atomic = (len(enc_tokens) == 1 and enc_tokens[0] == t)

            rows.append({
                "token": t,
                "token_id": token_id,
                "vocab_present": token_id is not None,
                "atomic_encode": bool(atomic),
                "encoded_tokens": " ".join(enc_tokens[:30]),
                "num_encoded_tokens": len(enc_tokens),
            })

        df = pd.DataFrame(rows)
        vocab_rate = float(df["vocab_present"].mean()) if len(df) else 1.0
        atomic_rate = float(df["atomic_encode"].mean()) if len(df) else 1.0

        return vocab_rate, atomic_rate, df

    def validate_texts(self, name: str, tok: LoadedWPTokenizer) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
        rows = []
        unk_id = tok.unk_id()

        for i, r in self.validation_df.iterrows():
            text = str(r["validation_text"])
            raw = clean_line(r["text"])

            ids = tok.encode_ids(text)
            tokens = tok.encode_tokens(text)
            unk_count = sum(1 for x in ids if x == unk_id)

            ws_units = max(1, len(raw.split()) + 1)
            char_len = max(1, len(raw.replace(" ", "")))

            rows.append({
                "candidate": name,
                "dialect": r["dialect"],
                "validation_origin": r["validation_origin"],
                "row_index": i,
                "whitespace_units_with_tag": ws_units,
                "char_len_without_spaces": char_len,
                "subword_tokens": len(tokens),
                "unk_count": unk_count,
                "unk_rate_row": unk_count / max(1, len(tokens)),
                "fertility": len(tokens) / ws_units,
                "chars_per_subword": char_len / max(1, len(tokens)),
                "tokens_preview": " ".join(tokens[:60]),
            })

        row_df = pd.DataFrame(rows)

        dialect_df = (
            row_df.groupby(["candidate", "dialect"])
            .agg(
                rows=("row_index", "count"),
                total_subword_tokens=("subword_tokens", "sum"),
                total_unk=("unk_count", "sum"),
                mean_unk_rate=("unk_rate_row", "mean"),
                mean_fertility=("fertility", "mean"),
                std_fertility=("fertility", "std"),
                mean_chars_per_subword=("chars_per_subword", "mean"),
            )
            .reset_index()
        )

        dialect_df["unk_rate"] = dialect_df["total_unk"] / dialect_df["total_subword_tokens"].clip(lower=1)

        summary = {
            "candidate": name,
            "backend": "hf_tokenizers",
            "model_type": tok.model_type,
            "protected_level": tok.protected_level,
            "vocab_size": tok.vocab_size(),
            "rows": len(row_df),
            "total_subword_tokens": int(row_df["subword_tokens"].sum()),
            "total_unk": int(row_df["unk_count"].sum()),
            "unk_rate": float(row_df["unk_count"].sum() / max(1, row_df["subword_tokens"].sum())),
            "mean_fertility": float(row_df["fertility"].mean()),
            "std_fertility_by_row": float(row_df["fertility"].std()),
            "fertility_std_across_dialects": float(dialect_df["mean_fertility"].std()),
            "mean_chars_per_subword": float(row_df["chars_per_subword"].mean()),
            "max_dialect_fertility": float(dialect_df["mean_fertility"].max()),
            "min_dialect_fertility": float(dialect_df["mean_fertility"].min()),
        }

        return row_df, dialect_df, summary

    def validate_all(self) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, Dict[str, pd.DataFrame]]]:
        print("\n" + "=" * 100)
        print("STEP 5: VALIDATE CANDIDATES")
        print("=" * 100)

        summaries = []
        row_tables = []
        dialect_tables = []
        symbol_reports = {}

        for name, tok in self.tokenizers.items():
            print(f"\n[{now()}] VALIDATE {name}")

            row_df, dialect_df, text_summary = self.validate_texts(name, tok)

            special_vocab_rate, special_atomic_rate, special_df = self.symbol_presence_report(tok, SPECIAL_TOKENS)
            tag_vocab_rate, tag_atomic_rate, tag_df = self.symbol_presence_report(tok, DIALECT_TAGS)

            strict_vocab_rate, strict_atomic_rate, strict_df = self.symbol_presence_report(
                tok,
                self.protected["strict_lexical"],
                max_n=min(self.cfg.max_symbols_to_validate, len(self.protected["strict_lexical"])),
            )

            expanded_vocab_rate, expanded_atomic_rate, expanded_df = self.symbol_presence_report(
                tok,
                self.protected["expanded_lexical"],
                max_n=min(self.cfg.max_symbols_to_validate, len(self.protected["expanded_lexical"])),
            )

            summary = {
                **text_summary,
                "special_vocab_rate": special_vocab_rate,
                "special_atomic_rate": special_atomic_rate,
                "dialect_tag_vocab_rate": tag_vocab_rate,
                "dialect_tag_atomic_rate": tag_atomic_rate,
                "strict_protected_vocab_rate": strict_vocab_rate,
                "strict_protected_atomic_rate": strict_atomic_rate,
                "expanded_protected_vocab_rate": expanded_vocab_rate,
                "expanded_protected_atomic_rate": expanded_atomic_rate,
                "special_missing_count": int((special_df["vocab_present"] == False).sum()),
                "dialect_tag_missing_count": int((tag_df["vocab_present"] == False).sum()),
                "strict_protected_missing_count": int((strict_df["vocab_present"] == False).sum()),
                "strict_protected_non_atomic_count": int((strict_df["atomic_encode"] == False).sum()),
                "expanded_protected_missing_count": int((expanded_df["vocab_present"] == False).sum()),
            }

            smaller_bonus = 1.0 if summary["vocab_size"] <= 33000 else 0.0

            summary["selection_score"] = float(
                100.0
                - 6000.0 * summary["unk_rate"]
                + 20.0 * summary["dialect_tag_vocab_rate"]
                + 20.0 * summary["dialect_tag_atomic_rate"]
                + 15.0 * summary["special_vocab_rate"]
                + 35.0 * summary["strict_protected_vocab_rate"]
                + 25.0 * summary["strict_protected_atomic_rate"]
                - 8.0 * max(0.0, summary["mean_fertility"] - 1.50)
                - 12.0 * summary["fertility_std_across_dialects"]
                + 2.0 * min(5.0, summary["mean_chars_per_subword"])
                + smaller_bonus
            )

            summaries.append(summary)
            row_tables.append(row_df)
            dialect_tables.append(dialect_df)

            for df_sym in [special_df, tag_df, strict_df, expanded_df]:
                df_sym.insert(0, "candidate", name)

            symbol_reports[name] = {
                "special": special_df,
                "dialect_tags": tag_df,
                "strict_protected": strict_df,
                "expanded_protected": expanded_df,
            }

            print(
                f"vocab={summary['vocab_size']:,} | "
                f"unk={summary['unk_rate']:.8f} | "
                f"fertility={summary['mean_fertility']:.4f} | "
                f"fert_std_dialect={summary['fertility_std_across_dialects']:.4f} | "
                f"tag_atomic={summary['dialect_tag_atomic_rate']:.4f} | "
                f"strict_atomic={summary['strict_protected_atomic_rate']:.4f} | "
                f"score={summary['selection_score']:.4f}"
            )

        summary_df = pd.DataFrame(summaries)
        row_df = pd.concat(row_tables, axis=0).reset_index(drop=True)
        dialect_df = pd.concat(dialect_tables, axis=0).reset_index(drop=True)

        return summary_df, row_df, dialect_df, symbol_reports


validator = WPTokenizerValidator(CFG, protected, loaded_tokenizers, df_validation)
candidate_summary_df, candidate_row_metrics_df, candidate_dialect_metrics_df, symbol_reports = validator.validate_all()


# ============================================================
# 11. Candidate Selection
# ============================================================

def select_candidate(summary_df: pd.DataFrame) -> Tuple[str, pd.DataFrame]:
    df = summary_df.copy()

    df["passes_hard_requirements"] = (
        (df["unk_rate"] <= CFG.max_allowed_unk_rate)
        & (df["special_vocab_rate"] >= CFG.min_special_vocab_rate)
        & (df["dialect_tag_vocab_rate"] >= CFG.min_dialect_tag_vocab_rate)
        & (df["dialect_tag_atomic_rate"] >= CFG.min_dialect_tag_atomic_rate)
        & (df["strict_protected_vocab_rate"] >= CFG.min_strict_protected_vocab_rate)
        & (df["strict_protected_atomic_rate"] >= CFG.min_strict_protected_atomic_rate)
    )

    passing = df[df["passes_hard_requirements"] == True].copy()

    if len(passing):
        ranked_passing = passing.sort_values("selection_score", ascending=False).reset_index(drop=True)

        if CFG.prefer_smaller_vocab_if_close and len(ranked_passing) > 1:
            best_score = ranked_passing.iloc[0]["selection_score"]
            close = ranked_passing[ranked_passing["selection_score"] >= best_score - CFG.close_score_margin].copy()
            selected = close.sort_values("vocab_size", ascending=True).iloc[0]["candidate"]
        else:
            selected = ranked_passing.iloc[0]["candidate"]
    else:
        selected = df.sort_values("selection_score", ascending=False).iloc[0]["candidate"]

    ranked = df.sort_values(
        ["passes_hard_requirements", "selection_score"],
        ascending=False,
    ).reset_index(drop=True)

    return selected, ranked

selected_candidate, candidate_ranked = select_candidate(candidate_summary_df)

print("\n" + "=" * 100)
print("STEP 6: CANDIDATE SELECTION")
print("=" * 100)
print(f"Selected candidate: {selected_candidate}")

display_cols = [
    "candidate",
    "backend",
    "model_type",
    "protected_level",
    "vocab_size",
    "passes_hard_requirements",
    "selection_score",
    "unk_rate",
    "mean_fertility",
    "fertility_std_across_dialects",
    "mean_chars_per_subword",
    "special_vocab_rate",
    "dialect_tag_vocab_rate",
    "dialect_tag_atomic_rate",
    "strict_protected_vocab_rate",
    "strict_protected_atomic_rate",
    "expanded_protected_vocab_rate",
]

print(candidate_ranked[display_cols].round(6).to_string(index=False))


# ============================================================
# 12. Export Reports and Final Tokenizer
# ============================================================

def export_symbol_reports(symbol_reports: Dict[str, Dict[str, pd.DataFrame]]) -> None:
    for cand_name, reports in symbol_reports.items():
        out_dir = VALIDATION_DIR / cand_name
        out_dir.mkdir(parents=True, exist_ok=True)

        for report_name, df in reports.items():
            df.to_csv(out_dir / f"{report_name}_symbol_validation.csv", index=False, encoding="utf-8")

def copy_final_tokenizer(selected_name: str, metadata: List[Dict[str, Any]]) -> Dict[str, Any]:
    print("\n" + "=" * 100)
    print("STEP 7: EXPORT FINAL TOKENIZER")
    print("=" * 100)

    meta_by_name = {m["candidate"]["name"]: m for m in metadata}
    selected_meta = meta_by_name[selected_name]

    src_dir = CANDIDATE_DIR / selected_name

    if FINAL_DIR.exists():
        shutil.rmtree(FINAL_DIR)
    FINAL_DIR.mkdir(parents=True, exist_ok=True)

    for item in src_dir.iterdir():
        dst = FINAL_DIR / item.name
        if item.is_file():
            shutil.copy2(item, dst)
        elif item.is_dir():
            shutil.copytree(item, dst)

    assert (FINAL_DIR / "tokenizer.json").exists(), "Final tokenizer.json missing"

    for fname in [
        CFG.strict_protected_file,
        CFG.expanded_protected_file,
        CFG.special_tokens_file,
        CFG.dialect_tags_file,
    ]:
        src = P3_DIR / fname
        if src.exists():
            shutil.copy2(src, FINAL_DIR / fname)

    tokenizer_config = {
        "tokenizer_project": "Bangla dialect-aware SLM/LLM tokenizer",
        "portion": "tokenizer_p4_deterministic_wordpiece_final",
        "selected_candidate": selected_name,
        "backend": selected_meta["backend"],
        "model_type": selected_meta["candidate"]["model_type"],
        "protected_level": selected_meta["candidate"]["protected_level"],
        "vocab_size_requested": selected_meta["candidate"]["vocab_size"],
        "vocab_size_actual": selected_meta["actual_vocab_size"],
        "wordpiece_prefix": selected_meta["wordpiece_prefix"],
        "normalizer": selected_meta["normalizer"],
        "pre_tokenizer": selected_meta["pre_tokenizer"],
        "decoder": selected_meta["decoder"],
        "special_tokens": {
            "pad_token": CFG.pad_token,
            "unk_token": CFG.unk_token,
            "bos_token": CFG.bos_token,
            "eos_token": CFG.eos_token,
        },
        "dialect_tags": DIALECT_TAGS,
        "dialects": ALL_DIALECTS,
        "source_metadata": selected_meta,
    }

    with open(FINAL_DIR / "tokenizer_config.json", "w", encoding="utf-8") as f:
        json.dump(tokenizer_config, f, ensure_ascii=False, indent=2)

    special_tokens_map = {
        "pad_token": CFG.pad_token,
        "unk_token": CFG.unk_token,
        "bos_token": CFG.bos_token,
        "eos_token": CFG.eos_token,
        "additional_special_tokens": DIALECT_TAGS,
    }

    with open(FINAL_DIR / "special_tokens_map.json", "w", encoding="utf-8") as f:
        json.dump(special_tokens_map, f, ensure_ascii=False, indent=2)

    with open(FINAL_DIR / "README_tokenizer.txt", "w", encoding="utf-8") as f:
        f.write("Bangla Dialect-Aware Deterministic WordPiece Tokenizer\n")
        f.write("=" * 70 + "\n\n")
        f.write(f"Selected candidate: {selected_name}\n")
        f.write(f"Backend: {selected_meta['backend']}\n")
        f.write(f"Model type: {selected_meta['candidate']['model_type']}\n")
        f.write(f"Protected level: {selected_meta['candidate']['protected_level']}\n")
        f.write(f"Actual vocab size: {selected_meta['actual_vocab_size']}\n\n")
        f.write("Load with:\n")
        f.write("from tokenizers import Tokenizer\n")
        f.write("tok = Tokenizer.from_file('tokenizer.json')\n\n")
        f.write("Next: Tokenizer Portion 5 — final validation, token-ID routing calibration, and freeze.\n")

    print(f"Saved final tokenizer directory: {FINAL_DIR}")
    return tokenizer_config

candidate_ranked.to_csv(P4_DIR / "tokenizer_candidate_comparison.csv", index=False, encoding="utf-8")
candidate_row_metrics_df.to_csv(VALIDATION_DIR / "candidate_metrics_by_row.csv", index=False, encoding="utf-8")
candidate_dialect_metrics_df.to_csv(VALIDATION_DIR / "candidate_metrics_by_dialect.csv", index=False, encoding="utf-8")
corpus_plan.to_csv(CORPUS_DIR / "training_corpus_wordpiece_sampling_plan.csv", index=False, encoding="utf-8")

export_symbol_reports(symbol_reports)
final_config = copy_final_tokenizer(selected_candidate, candidate_metadata)

selection_summary = {
    "portion": "tokenizer_p4_deterministic_wordpiece_candidate_build_validation_selection",
    "seed": SEED,
    "config": asdict(CFG),
    "selected_candidate": selected_candidate,
    "candidate_ranking": candidate_ranked.to_dict(orient="records"),
    "candidate_metadata": candidate_metadata,
    "training_corpus_path": str(CORPUS_PATH),
    "training_corpus_plan": str(CORPUS_DIR / "training_corpus_wordpiece_sampling_plan.csv"),
    "validation_rows": int(len(df_validation)),
    "outputs": {
        "candidate_comparison": str(P4_DIR / "tokenizer_candidate_comparison.csv"),
        "candidate_metrics_by_row": str(VALIDATION_DIR / "candidate_metrics_by_row.csv"),
        "candidate_metrics_by_dialect": str(VALIDATION_DIR / "candidate_metrics_by_dialect.csv"),
        "final_tokenizer_dir": str(FINAL_DIR),
    },
}

with open(P4_DIR / "tokenizer_selection_summary.json", "w", encoding="utf-8") as f:
    json.dump(selection_summary, f, ensure_ascii=False, indent=2)

print(f"Saved: {P4_DIR / 'tokenizer_candidate_comparison.csv'}")
print(f"Saved: {VALIDATION_DIR / 'candidate_metrics_by_row.csv'}")
print(f"Saved: {VALIDATION_DIR / 'candidate_metrics_by_dialect.csv'}")
print(f"Saved: {P4_DIR / 'tokenizer_selection_summary.json'}")


# ============================================================
# 13. Plots
# ============================================================

def save_plots(summary_df: pd.DataFrame, dialect_df: pd.DataFrame) -> None:
    if not CFG.save_plots:
        return

    print("\n" + "=" * 100)
    print("STEP 8: SAVE PLOTS")
    print("=" * 100)

    s = summary_df.sort_values("selection_score", ascending=False)

    plt.figure(figsize=(10, 5))
    plt.bar(s["candidate"], s["selection_score"])
    plt.title("Tokenizer Candidate Selection Score")
    plt.xlabel("Candidate")
    plt.ylabel("Selection score")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    p = VALIDATION_DIR / "candidate_selection_scores.png"
    plt.savefig(p, dpi=200)
    plt.close()
    print(f"Saved: {p}")

    plt.figure(figsize=(10, 5))
    plt.bar(s["candidate"], s["mean_fertility"])
    plt.title("Mean Fertility by Candidate")
    plt.xlabel("Candidate")
    plt.ylabel("Subword pieces per whitespace unit")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    p = VALIDATION_DIR / "candidate_mean_fertility.png"
    plt.savefig(p, dpi=200)
    plt.close()
    print(f"Saved: {p}")

    sub = dialect_df[dialect_df["candidate"] == selected_candidate].copy().sort_values("dialect")

    plt.figure(figsize=(14, 6))
    plt.bar(sub["dialect"], sub["mean_fertility"])
    plt.title(f"Selected Candidate Fertility by Dialect — {selected_candidate}")
    plt.xlabel("Dialect")
    plt.ylabel("Subword pieces per whitespace unit")
    plt.xticks(rotation=45)
    plt.tight_layout()
    p = VALIDATION_DIR / "selected_candidate_fertility_by_dialect.png"
    plt.savefig(p, dpi=200)
    plt.close()
    print(f"Saved: {p}")

save_plots(candidate_ranked, candidate_dialect_metrics_df)


# ============================================================
# 14. Final Tests
# ============================================================

def run_tests() -> Dict[str, Any]:
    print("\n" + "=" * 100)
    print("PORTION 4 TESTS")
    print("=" * 100)

    failures = []

    row = candidate_ranked[candidate_ranked["candidate"] == selected_candidate].iloc[0].to_dict()

    if row["unk_rate"] > CFG.max_allowed_unk_rate:
        failures.append(f"UNK rate too high: {row['unk_rate']}")

    if row["special_vocab_rate"] < CFG.min_special_vocab_rate:
        failures.append(f"Special vocab rate too low: {row['special_vocab_rate']}")

    if row["dialect_tag_vocab_rate"] < CFG.min_dialect_tag_vocab_rate:
        failures.append(f"Dialect tag vocab rate too low: {row['dialect_tag_vocab_rate']}")

    if row["dialect_tag_atomic_rate"] < CFG.min_dialect_tag_atomic_rate:
        failures.append(f"Dialect tag atomic rate too low: {row['dialect_tag_atomic_rate']}")

    if row["strict_protected_vocab_rate"] < CFG.min_strict_protected_vocab_rate:
        failures.append(f"Strict protected vocab rate too low: {row['strict_protected_vocab_rate']}")

    if row["strict_protected_atomic_rate"] < CFG.min_strict_protected_atomic_rate:
        failures.append(f"Strict protected atomic rate too low: {row['strict_protected_atomic_rate']}")

    if not FINAL_DIR.exists():
        failures.append(f"Final dir missing: {FINAL_DIR}")

    if not (FINAL_DIR / "tokenizer.json").exists():
        failures.append("Final tokenizer.json missing")

    if not (FINAL_DIR / "tokenizer_config.json").exists():
        failures.append("tokenizer_config.json missing")

    if not (FINAL_DIR / "special_tokens_map.json").exists():
        failures.append("special_tokens_map.json missing")

    tok = Tokenizer.from_file(str(FINAL_DIR / "tokenizer.json"))
    unk_id = tok.token_to_id(CFG.unk_token)

    smoke_texts = [
        "<dial:BAR> আমি এখন যাই না",
        "<dial:CHI> আঁই ঘরত যাই",
        "<dial:SYL> তাইন কিলা আছেন",
        "<dial:RAJ> আমি এইডা করবো",
        "<dial:KHU> রাখছস এইজায়গায়",
    ]

    smoke_rows = []
    for text in smoke_texts:
        enc = tok.encode(text)
        ids = enc.ids
        tokens = enc.tokens
        smoke_rows.append({
            "text": text,
            "tokens": tokens,
            "ids": ids,
            "unk_count": int(sum(x == unk_id for x in ids)),
        })

    smoke_unk = sum(x["unk_count"] for x in smoke_rows)
    if smoke_unk > 0:
        failures.append(f"Smoke test produced UNK count: {smoke_unk}")

    smoke_path = VALIDATION_DIR / "final_tokenizer_smoke_test.json"
    with open(smoke_path, "w", encoding="utf-8") as f:
        json.dump(smoke_rows, f, ensure_ascii=False, indent=2)

    results = {
        "selected_candidate": selected_candidate,
        "backend": row["backend"],
        "model_type": row["model_type"],
        "protected_level": row["protected_level"],
        "selected_vocab_size": int(row["vocab_size"]),
        "selected_unk_rate": float(row["unk_rate"]),
        "selected_mean_fertility": float(row["mean_fertility"]),
        "selected_fertility_std_across_dialects": float(row["fertility_std_across_dialects"]),
        "selected_special_vocab_rate": float(row["special_vocab_rate"]),
        "selected_dialect_tag_vocab_rate": float(row["dialect_tag_vocab_rate"]),
        "selected_dialect_tag_atomic_rate": float(row["dialect_tag_atomic_rate"]),
        "selected_strict_protected_vocab_rate": float(row["strict_protected_vocab_rate"]),
        "selected_strict_protected_atomic_rate": float(row["strict_protected_atomic_rate"]),
        "selected_expanded_protected_vocab_rate": float(row["expanded_protected_vocab_rate"]),
        "final_dir": str(FINAL_DIR),
        "smoke_test_path": str(smoke_path),
        "hard_failures": failures,
        "passed": len(failures) == 0,
    }

    for k, v in results.items():
        if k != "hard_failures":
            print(f"{k:65s}: {v}")

    if failures:
        print("\n[FAILED]")
        for f in failures:
            print(f"  - {f}")
        raise AssertionError("Tokenizer Portion 4 tests failed.")

    print("\n[OK] Portion 4 tests passed.")
    return results

test_results = run_tests()

with open(P4_DIR / "p4_test_results.json", "w", encoding="utf-8") as f:
    json.dump(test_results, f, ensure_ascii=False, indent=2)


# ============================================================
# 15. Checkpoint
# ============================================================

print("\n" + "=" * 100)
print("CHECKPOINT: TOKENIZER PORTION 4")
print("=" * 100)

print(f"Selected tokenizer: {selected_candidate}")
print(f"Final tokenizer dir: {FINAL_DIR}")

print("\nCandidate comparison:")
print(candidate_ranked[display_cols].round(6).to_string(index=False))

print("\nSelected candidate fertility by dialect:")
print(
    candidate_dialect_metrics_df[candidate_dialect_metrics_df["candidate"] == selected_candidate][
        [
            "dialect",
            "rows",
            "unk_rate",
            "mean_fertility",
            "std_fertility",
            "mean_chars_per_subword",
        ]
    ].round(6).sort_values("dialect").to_string(index=False)
)

print("\nSaved artifacts:")
print(P4_DIR / "tokenizer_candidate_comparison.csv")
print(VALIDATION_DIR / "candidate_metrics_by_row.csv")
print(VALIDATION_DIR / "candidate_metrics_by_dialect.csv")
print(P4_DIR / "tokenizer_selection_summary.json")
print(P4_DIR / "p4_test_results.json")
print(FINAL_DIR)

print("\n✅ TOKENIZER PORTION 4 COMPLETE")
print("Next: Tokenizer Portion 5 — final tokenizer validation, token-ID routing calibration, and freeze.")

TOKENIZER PORTION 4 — DETERMINISTIC WORDPIECE FINAL VERSION
Train input : artifacts/tokenizer_p21/tokenizer_train_clean_p21_recommended.csv
Holdout     : artifacts/tokenizer_p2/tokenizer_eval_holdout_clean.csv
P3 dir      : artifacts/tokenizer_p3
Output dir  : artifacts/tokenizer_p4_wordpiece
Dialects    : ['BAR', 'CHI', 'KHU', 'KIS', 'MYM', 'NAR', 'NOA', 'NSD', 'RAJ', 'RAN', 'STD', 'SYL', 'TAN']
GPU needed  : NO — deterministic CPU vocabulary construction.

STEP 0: LOAD MINIMAL TRAIN DATA
[2026-05-13 13:00:45] Train rows loaded: 121,503
dialect
BAR     8466
CHI    14592
KHU     8126
KIS     9899
MYM     8993
NAR     8952
NOA     8663
NSD     9486
RAJ     8955
RAN     8449
STD     9011
SYL     9558
TAN     8353

STEP 0.1: LOAD HOLDOUT DATA
[2026-05-13 13:00:45] Holdout rows loaded: 2,875
dialect
BAR    375
CHI    625
MYM    625
NOA    625
SYL    625

STEP 0.2: LOAD PROTECTED SYMBOLS
Special tokens           : 4
Dialect tags             : 13
Strict lexical protected : 3,679
Expanded lex

In [7]:
# %% [markdown]
# ============================================================
# TOKENIZER PORTION 5 — FINAL FREEZE, VALIDATION, ROUTING CALIBRATION
# Bangla Dialect-Aware Deterministic WordPiece Tokenizer
# ============================================================
#
# Purpose:
#   1. Load the final Portion 4 tokenizer.
#   2. Validate special tokens, dialect tags, protected tokens, and UNK behavior.
#   3. Run train/holdout fertility + row-level outlier audit.
#   4. Build token-ID routing tables for dialect-conditioned SLM/LLM training.
#   5. Freeze the tokenizer into a model-ready directory with hashes/manifests.
#   6. Export a tokenizer card for paper/project documentation.
#
# GPU:
#   Not needed. This is CPU-only validation and metadata generation.
#
# Primary input:
#   artifacts/tokenizer_p4_wordpiece/final_tokenizer/tokenizer.json
#
# Main outputs:
#   artifacts/tokenizer_p5_freeze/final_tokenizer_frozen/
#   artifacts/tokenizer_p5_freeze/tokenizer_manifest.json
#   artifacts/tokenizer_p5_freeze/model_io_config.json
#   artifacts/tokenizer_p5_freeze/dialect_token_id_map.json
#   artifacts/tokenizer_p5_freeze/protected_token_id_table.csv
#   artifacts/tokenizer_p5_freeze/fertility_by_dialect.csv
#   artifacts/tokenizer_p5_freeze/fertility_outlier_rows.csv
#   artifacts/tokenizer_p5_freeze/tokenizer_card.md
# ============================================================

# %%
from __future__ import annotations

import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TOKENIZERS_PARALLELISM"] = "true"

import re
import gc
import json
import time
import shutil
import random
import hashlib
import warnings
import unicodedata
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    from tokenizers import Tokenizer
except Exception as e:
    raise ImportError("Missing package: tokenizers. Run: !pip install -q tokenizers") from e

SEED = 42

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)


# %% [markdown]
# ## Cell 1 — Configuration

# %%
@dataclass(frozen=True)
class Portion5Config:
    # Final tokenizer from Portion 4.
    p4_final_dir: str = "artifacts/tokenizer_p4_wordpiece/final_tokenizer"

    # Upstream artifacts.
    train_path: str = "artifacts/tokenizer_p21/tokenizer_train_clean_p21_recommended.csv"
    holdout_path: str = "artifacts/tokenizer_p2/tokenizer_eval_holdout_clean.csv"
    p3_dir: str = "artifacts/tokenizer_p3"
    p4_dir: str = "artifacts/tokenizer_p4_wordpiece"

    # Output.
    output_dir: str = "artifacts/tokenizer_p5_freeze"
    frozen_dir_name: str = "final_tokenizer_frozen"

    # Protected token files from Portion 3 / copied in Portion 4.
    strict_protected_file: str = "protected_tokens_strict.txt"
    expanded_protected_file: str = "protected_tokens_expanded.txt"
    special_tokens_file: str = "special_tokens.txt"
    dialect_tags_file: str = "dialect_tags.txt"

    # P3 dialect-token score files.
    dialect_token_file_pattern: str = "dialect_tokens_{dialect}_strict.csv"
    core_tokens_file: str = "core_tokens_strict.csv"
    adaptive_review_file: str = "adaptive_synthetic_assisted_strict_tokens.csv"

    # Validation sampling.
    train_validation_sample_per_dialect: int = 1200
    holdout_all: bool = True
    row_outlier_quantile: float = 0.99
    absolute_fertility_outlier_threshold: float = 3.0

    # Hard validation thresholds.
    max_allowed_unk_rate: float = 0.001
    max_allowed_smoke_unk_count: int = 0
    min_special_vocab_rate: float = 1.0
    min_dialect_tag_vocab_rate: float = 1.0
    min_dialect_tag_atomic_rate: float = 1.0
    min_strict_protected_vocab_rate: float = 1.0
    min_strict_protected_atomic_rate: float = 0.995
    max_mean_fertility: float = 1.75
    max_dialect_mean_fertility: float = 2.25

    # Model I/O defaults for SLM/LLM.
    max_seq_len_default: int = 512
    padding_side: str = "right"
    truncation_side: str = "right"

    # Special tokens.
    pad_token: str = "<pad>"
    unk_token: str = "<unk>"
    bos_token: str = "<bos>"
    eos_token: str = "<eos>"

    target_dialects_10: Tuple[str, ...] = (
        "BAR", "CHI", "KHU", "MYM", "NAR",
        "NOA", "RAJ", "RAN", "STD", "SYL",
    )
    auxiliary_dialects: Tuple[str, ...] = ("KIS", "TAN", "NSD")

    save_plots: bool = True


CFG = Portion5Config()

P4_FINAL_DIR = Path(CFG.p4_final_dir)
P3_DIR = Path(CFG.p3_dir)
P4_DIR = Path(CFG.p4_dir)
P5_DIR = Path(CFG.output_dir)
FROZEN_DIR = P5_DIR / CFG.frozen_dir_name
VALIDATION_DIR = P5_DIR / "validation"
ROUTING_DIR = P5_DIR / "routing"
REPORT_DIR = P5_DIR / "reports"
PLOT_DIR = P5_DIR / "plots"

for p in [P5_DIR, FROZEN_DIR, VALIDATION_DIR, ROUTING_DIR, REPORT_DIR, PLOT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

ALL_DIALECTS = sorted(set(CFG.target_dialects_10 + CFG.auxiliary_dialects))
SPECIAL_TOKENS = [CFG.pad_token, CFG.unk_token, CFG.bos_token, CFG.eos_token]
DIALECT_TAGS = [f"<dial:{d}>" for d in ALL_DIALECTS]
SPECIAL_PLUS_TAGS = SPECIAL_TOKENS + DIALECT_TAGS

print("=" * 100)
print("TOKENIZER PORTION 5 — FINAL FREEZE, VALIDATION, ROUTING CALIBRATION")
print("=" * 100)
print(f"P4 final dir : {P4_FINAL_DIR}")
print(f"P3 dir       : {P3_DIR}")
print(f"Output dir   : {P5_DIR}")
print(f"Frozen dir   : {FROZEN_DIR}")
print(f"Dialects     : {ALL_DIALECTS}")
print("GPU needed   : NO — CPU-only validation and manifest generation.")
print("=" * 100)

assert P4_FINAL_DIR.exists(), f"Missing final tokenizer dir: {P4_FINAL_DIR}"
assert (P4_FINAL_DIR / "tokenizer.json").exists(), f"Missing tokenizer.json in {P4_FINAL_DIR}"
assert P3_DIR.exists(), f"Missing Portion 3 dir: {P3_DIR}"


# %% [markdown]
# ## Cell 2 — Utility Functions

# %%
def now() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")


def safe_bool(x: Any) -> bool:
    if isinstance(x, bool):
        return x
    if pd.isna(x):
        return False
    return str(x).strip().lower() in {"true", "1", "yes", "y"}


def stable_hash(text: str) -> str:
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()


def file_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def clean_line(text: Any) -> str:
    if pd.isna(text):
        return ""
    s = str(text)
    s = unicodedata.normalize("NFC", s)
    s = re.sub(r"[\x00-\x1f\x7f-\x9f]", "", s)
    s = re.sub(r"[\u200b\u200c\u200d\u200e\u200f\ufeff]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def read_lines(path: Path) -> List[str]:
    assert path.exists(), f"Missing file: {path}"
    with open(path, "r", encoding="utf-8") as f:
        return [x.strip() for x in f if x.strip()]


def write_lines(path: Path, lines: List[str]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for x in lines:
            x = str(x).strip()
            if x:
                f.write(x + "\n")


def write_json(path: Path, obj: Any) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def read_json(path: Path) -> Dict[str, Any]:
    assert path.exists(), f"Missing JSON file: {path}"
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def dedup_keep_order(xs: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in xs:
        x = str(x).strip()
        if not x:
            continue
        if x in seen:
            continue
        seen.add(x)
        out.append(x)
    return out


def make_dialect_tag(dialect: str) -> str:
    return f"<dial:{dialect}>"


def is_dialect_tag(token: str) -> bool:
    return bool(re.fullmatch(r"<dial:[A-Z]{3}>", str(token)))


def token_preview(tokens: List[str], n: int = 80) -> str:
    return " ".join(tokens[:n])


def compact_text(text: Any, n: int = 160) -> str:
    s = clean_line(text)
    if len(s) <= n:
        return s
    return s[:n] + "..."


# %% [markdown]
# ## Cell 3 — Load Final Tokenizer and Upstream Symbol Lists

# %%
print("\n" + "=" * 100)
print("CELL 3: LOAD TOKENIZER + SYMBOL LISTS")
print("=" * 100)

TOKENIZER_JSON = P4_FINAL_DIR / "tokenizer.json"
TOKENIZER_CONFIG_PATH = P4_FINAL_DIR / "tokenizer_config.json"
SPECIAL_MAP_PATH = P4_FINAL_DIR / "special_tokens_map.json"
VOCAB_TXT_PATH = P4_FINAL_DIR / "vocab.txt"
VOCAB_PROVENANCE_PATH = P4_FINAL_DIR / "vocab_provenance.csv"

final_tokenizer = Tokenizer.from_file(str(TOKENIZER_JSON))

p4_tokenizer_config = read_json(TOKENIZER_CONFIG_PATH) if TOKENIZER_CONFIG_PATH.exists() else {}
p4_special_map = read_json(SPECIAL_MAP_PATH) if SPECIAL_MAP_PATH.exists() else {}

strict_protected_path = P4_FINAL_DIR / CFG.strict_protected_file
expanded_protected_path = P4_FINAL_DIR / CFG.expanded_protected_file
special_tokens_path = P4_FINAL_DIR / CFG.special_tokens_file
dialect_tags_path = P4_FINAL_DIR / CFG.dialect_tags_file

# Fallback to P3 if the file was not copied into final tokenizer dir.
if not strict_protected_path.exists():
    strict_protected_path = P3_DIR / CFG.strict_protected_file
if not expanded_protected_path.exists():
    expanded_protected_path = P3_DIR / CFG.expanded_protected_file
if not special_tokens_path.exists():
    special_tokens_path = P3_DIR / CFG.special_tokens_file
if not dialect_tags_path.exists():
    dialect_tags_path = P3_DIR / CFG.dialect_tags_file

strict_all = read_lines(strict_protected_path)
expanded_all = read_lines(expanded_protected_path)
special_from_file = read_lines(special_tokens_path) if special_tokens_path.exists() else SPECIAL_TOKENS
dialect_tags_from_file = read_lines(dialect_tags_path) if dialect_tags_path.exists() else DIALECT_TAGS

special_from_file = dedup_keep_order(special_from_file)
dialect_tags_from_file = dedup_keep_order(dialect_tags_from_file)
reserved = set(special_from_file + dialect_tags_from_file)

strict_lexical = dedup_keep_order([
    t for t in strict_all
    if t not in reserved and not is_dialect_tag(t)
])
expanded_lexical = dedup_keep_order([
    t for t in expanded_all
    if t not in reserved and not is_dialect_tag(t)
])

vocab_size = final_tokenizer.get_vocab_size(with_added_tokens=True)
unk_id = final_tokenizer.token_to_id(CFG.unk_token)
pad_id = final_tokenizer.token_to_id(CFG.pad_token)
bos_id = final_tokenizer.token_to_id(CFG.bos_token)
eos_id = final_tokenizer.token_to_id(CFG.eos_token)

print(f"Tokenizer vocab size     : {vocab_size:,}")
print(f"Special tokens from file : {special_from_file}")
print(f"Dialect tags from file   : {len(dialect_tags_from_file)}")
print(f"Strict all symbols       : {len(strict_all):,}")
print(f"Strict lexical symbols   : {len(strict_lexical):,}")
print(f"Expanded lexical symbols : {len(expanded_lexical):,}")
print(f"pad/unk/bos/eos IDs      : {pad_id}, {unk_id}, {bos_id}, {eos_id}")

assert unk_id is not None, "UNK token ID is missing."
assert pad_id is not None, "PAD token ID is missing."
assert bos_id is not None, "BOS token ID is missing."
assert eos_id is not None, "EOS token ID is missing."


# %% [markdown]
# ## Cell 4 — Load Validation Data

# %%
print("\n" + "=" * 100)
print("CELL 4: LOAD TRAIN/HOLDOUT VALIDATION DATA")
print("=" * 100)

TRAIN_PATH = Path(CFG.train_path)
HOLDOUT_PATH = Path(CFG.holdout_path)

assert TRAIN_PATH.exists(), f"Missing train file: {TRAIN_PATH}"


def load_train_for_validation(path: Path) -> pd.DataFrame:
    cols = {"text", "dialect", "is_synthetic", "split_original", "eligible_for_tokenizer_train"}
    df = pd.read_csv(path, usecols=lambda c: c in cols)
    for col in ["is_synthetic", "eligible_for_tokenizer_train"]:
        if col in df.columns:
            df[col] = df[col].map(safe_bool)
    df = df[df["eligible_for_tokenizer_train"] == True].copy()
    df = df[~df["split_original"].isin(["validation", "test"])].copy()
    df = df[df["dialect"].isin(ALL_DIALECTS)].copy()
    df = df[df["text"].notna()].copy()
    df["text"] = df["text"].map(clean_line)
    df = df[df["text"].str.len() > 0].copy()
    return df.reset_index(drop=True)


def load_holdout_for_validation(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=["text", "dialect", "is_synthetic", "split_original"])
    cols = {"text", "dialect", "is_synthetic", "split_original"}
    df = pd.read_csv(path, usecols=lambda c: c in cols)
    if "is_synthetic" in df.columns:
        df["is_synthetic"] = df["is_synthetic"].map(safe_bool)
    else:
        df["is_synthetic"] = False
    df = df[df["dialect"].isin(ALL_DIALECTS)].copy()
    df = df[df["text"].notna()].copy()
    df["text"] = df["text"].map(clean_line)
    df = df[df["text"].str.len() > 0].copy()
    return df.reset_index(drop=True)


df_train = load_train_for_validation(TRAIN_PATH)
df_holdout = load_holdout_for_validation(HOLDOUT_PATH)

print(f"Train rows   : {len(df_train):,}")
print(f"Holdout rows : {len(df_holdout):,}")
print("\nTrain dialect counts:")
print(df_train["dialect"].value_counts().sort_index().to_string())
if len(df_holdout):
    print("\nHoldout dialect counts:")
    print(df_holdout["dialect"].value_counts().sort_index().to_string())


def build_validation_frame(train: pd.DataFrame, holdout: pd.DataFrame) -> pd.DataFrame:
    parts = []

    if CFG.holdout_all and len(holdout):
        h = holdout.copy()
        h["validation_origin"] = "real_holdout"
        parts.append(h)

    samples = []
    for d in ALL_DIALECTS:
        sub = train[train["dialect"] == d]
        n = min(CFG.train_validation_sample_per_dialect, len(sub))
        if n > 0:
            samples.append(sub.sample(n=n, random_state=SEED + int(stable_hash(d)[:6], 16)))

    train_sample = pd.concat(samples, axis=0).reset_index(drop=True)
    train_sample["validation_origin"] = "train_sample"
    parts.append(train_sample)

    val = pd.concat(parts, axis=0).reset_index(drop=True)
    val["text"] = val["text"].map(clean_line)
    val["input_text"] = val.apply(lambda r: f"{make_dialect_tag(r['dialect'])} {r['text']}", axis=1)
    val["row_uid"] = [stable_hash(f"{i}|{d}|{t}")[:16] for i, (d, t) in enumerate(zip(val["dialect"], val["text"]))]
    return val


df_validation = build_validation_frame(df_train, df_holdout)
validation_path = VALIDATION_DIR / "portion5_validation_frame.csv"
df_validation.to_csv(validation_path, index=False, encoding="utf-8")

print(f"\nValidation rows: {len(df_validation):,}")
print(df_validation.groupby(["validation_origin", "dialect"]).size().to_string())
print(f"Saved: {validation_path}")


# %% [markdown]
# ## Cell 5 — Token Inventory and Atomicity Validation

# %%
print("\n" + "=" * 100)
print("CELL 5: TOKEN INVENTORY + ATOMICITY VALIDATION")
print("=" * 100)


def encode_tokens(text: str) -> List[str]:
    return list(final_tokenizer.encode(str(text)).tokens)


def encode_ids(text: str) -> List[int]:
    return list(final_tokenizer.encode(str(text)).ids)


def token_to_id(token: str) -> Optional[int]:
    return final_tokenizer.token_to_id(str(token))


def token_atomic(token: str) -> bool:
    toks = encode_tokens(token)
    return len(toks) == 1 and toks[0] == token


def build_symbol_table(tokens: List[str], symbol_type: str, max_n: Optional[int] = None) -> pd.DataFrame:
    tokens = dedup_keep_order(tokens)
    if max_n is not None and len(tokens) > max_n:
        rng = random.Random(SEED)
        tokens = rng.sample(tokens, k=max_n)

    rows = []
    for t in tokens:
        tid = token_to_id(t)
        enc_toks = encode_tokens(t)
        enc_ids = encode_ids(t)
        rows.append({
            "token": t,
            "symbol_type": symbol_type,
            "token_id": tid,
            "vocab_present": tid is not None,
            "atomic_encode": len(enc_toks) == 1 and enc_toks[0] == t,
            "encoded_tokens": " ".join(enc_toks[:40]),
            "encoded_ids": " ".join(map(str, enc_ids[:40])),
            "num_encoded_tokens": len(enc_toks),
        })
    return pd.DataFrame(rows)


special_table = build_symbol_table(SPECIAL_TOKENS, "special")
dialect_tag_table = build_symbol_table(DIALECT_TAGS, "dialect_tag")
strict_table = build_symbol_table(strict_lexical, "strict_protected")
expanded_table = build_symbol_table(expanded_lexical, "expanded_protected")

protected_token_id_table = pd.concat(
    [special_table, dialect_tag_table, strict_table, expanded_table],
    axis=0,
    ignore_index=True,
)

protected_token_id_table.to_csv(REPORT_DIR / "protected_token_id_table.csv", index=False, encoding="utf-8")
special_table.to_csv(REPORT_DIR / "special_token_id_table.csv", index=False, encoding="utf-8")
dialect_tag_table.to_csv(REPORT_DIR / "dialect_tag_id_table.csv", index=False, encoding="utf-8")
strict_table.to_csv(REPORT_DIR / "strict_protected_token_id_table.csv", index=False, encoding="utf-8")
expanded_table.to_csv(REPORT_DIR / "expanded_protected_token_id_table.csv", index=False, encoding="utf-8")

symbol_summary = {
    "special_vocab_rate": float(special_table["vocab_present"].mean()),
    "special_atomic_rate": float(special_table["atomic_encode"].mean()),
    "dialect_tag_vocab_rate": float(dialect_tag_table["vocab_present"].mean()),
    "dialect_tag_atomic_rate": float(dialect_tag_table["atomic_encode"].mean()),
    "strict_protected_vocab_rate": float(strict_table["vocab_present"].mean()),
    "strict_protected_atomic_rate": float(strict_table["atomic_encode"].mean()),
    "expanded_protected_vocab_rate": float(expanded_table["vocab_present"].mean()),
    "expanded_protected_atomic_rate": float(expanded_table["atomic_encode"].mean()),
    "strict_missing_count": int((strict_table["vocab_present"] == False).sum()),
    "strict_non_atomic_count": int((strict_table["atomic_encode"] == False).sum()),
    "expanded_missing_count": int((expanded_table["vocab_present"] == False).sum()),
    "expanded_non_atomic_count": int((expanded_table["atomic_encode"] == False).sum()),
}

write_json(REPORT_DIR / "symbol_validation_summary.json", symbol_summary)

print(json.dumps(symbol_summary, ensure_ascii=False, indent=2))
print(f"Saved: {REPORT_DIR / 'protected_token_id_table.csv'}")


# %% [markdown]
# ## Cell 6 — Dialect Tag ID Map and Routing Tables

# %%
print("\n" + "=" * 100)
print("CELL 6: DIALECT TAG ID MAP + ROUTING CALIBRATION")
print("=" * 100)

# Dialect tag IDs.
dialect_token_id_map = {}
for d in ALL_DIALECTS:
    tag = make_dialect_tag(d)
    tid = token_to_id(tag)
    assert tid is not None, f"Missing dialect tag ID: {tag}"
    dialect_token_id_map[d] = {
        "dialect": d,
        "tag": tag,
        "tag_id": int(tid),
        "taxonomy_role": "auxiliary" if d in CFG.auxiliary_dialects else "target",
    }

write_json(ROUTING_DIR / "dialect_token_id_map.json", dialect_token_id_map)

# Load P3 dialect-specific strict token tables and map to IDs.
dialect_specific_rows = []
for d in ALL_DIALECTS:
    p = P3_DIR / CFG.dialect_token_file_pattern.format(dialect=d)
    if not p.exists():
        continue
    src = pd.read_csv(p)
    if "token" not in src.columns:
        continue
    for _, r in src.iterrows():
        tok = str(r["token"])
        tid = token_to_id(tok)
        dialect_specific_rows.append({
            "dialect": d,
            "token": tok,
            "token_id": tid,
            "vocab_present": tid is not None,
            "atomic_encode": token_atomic(tok),
            "count_td": int(r.get("count_td", 0)) if not pd.isna(r.get("count_td", np.nan)) else 0,
            "real_count_td": int(r.get("real_count_td", 0)) if not pd.isna(r.get("real_count_td", np.nan)) else 0,
            "synthetic_count_td": int(r.get("synthetic_count_td", 0)) if not pd.isna(r.get("synthetic_count_td", np.nan)) else 0,
            "candidate_tier": str(r.get("candidate_tier", "")),
            "association_score": float(r.get("association_score", 0.0)) if not pd.isna(r.get("association_score", np.nan)) else 0.0,
        })

dialect_specific_token_ids = pd.DataFrame(dialect_specific_rows)
if len(dialect_specific_token_ids):
    dialect_specific_token_ids.to_csv(ROUTING_DIR / "dialect_specific_token_ids.csv", index=False, encoding="utf-8")

# Core token IDs.
core_token_ids = pd.DataFrame()
core_path = P3_DIR / CFG.core_tokens_file
if core_path.exists():
    core = pd.read_csv(core_path)
    if "token" in core.columns:
        core_rows = []
        for _, r in core.iterrows():
            tok = str(r["token"])
            tid = token_to_id(tok)
            core_rows.append({
                "token": tok,
                "token_id": tid,
                "vocab_present": tid is not None,
                "atomic_encode": token_atomic(tok),
                "total_count": int(r.get("total_count", 0)) if not pd.isna(r.get("total_count", np.nan)) else 0,
                "real_count": int(r.get("real_count", 0)) if not pd.isna(r.get("real_count", np.nan)) else 0,
                "core_score": float(r.get("core_score", 0.0)) if not pd.isna(r.get("core_score", np.nan)) else 0.0,
            })
        core_token_ids = pd.DataFrame(core_rows)
        core_token_ids.to_csv(ROUTING_DIR / "core_token_ids.csv", index=False, encoding="utf-8")

# ID lists for model routing / optional embedding analysis.
special_token_ids = {t: int(token_to_id(t)) for t in SPECIAL_TOKENS}
dialect_tag_ids = {d: int(v["tag_id"]) for d, v in dialect_token_id_map.items()}

routing_manifest = {
    "special_token_ids": special_token_ids,
    "dialect_tag_ids": dialect_tag_ids,
    "target_dialects_10": list(CFG.target_dialects_10),
    "auxiliary_dialects": list(CFG.auxiliary_dialects),
    "all_dialects": ALL_DIALECTS,
    "dialect_specific_token_id_csv": str(ROUTING_DIR / "dialect_specific_token_ids.csv"),
    "core_token_id_csv": str(ROUTING_DIR / "core_token_ids.csv"),
    "notes": [
        "Use dialect_tag_ids to prefix each sample before SLM/LLM tokenization.",
        "Dialect-specific token IDs are for routing diagnostics and optional embedding analysis.",
        "Do not mask dialect tags during causal LM training unless running a tag-ablation experiment.",
    ],
}

write_json(ROUTING_DIR / "routing_manifest.json", routing_manifest)

print("Dialect tag IDs:")
print(pd.DataFrame(dialect_token_id_map.values()).sort_values("dialect").to_string(index=False))
if len(dialect_specific_token_ids):
    print("\nDialect-specific token ID counts:")
    print(dialect_specific_token_ids.groupby("dialect").size().to_string())
print(f"\nSaved: {ROUTING_DIR / 'routing_manifest.json'}")


# %% [markdown]
# ## Cell 7 — Corpus-Level Tokenizer Metrics

# %%
print("\n" + "=" * 100)
print("CELL 7: CORPUS-LEVEL VALIDATION METRICS")
print("=" * 100)


def evaluate_rows(df: pd.DataFrame, name: str) -> pd.DataFrame:
    rows = []
    for i, r in df.iterrows():
        dialect = str(r["dialect"])
        text = clean_line(r["text"])
        input_text = f"{make_dialect_tag(dialect)} {text}"

        enc = final_tokenizer.encode(input_text)
        ids = list(enc.ids)
        toks = list(enc.tokens)
        unk_count = int(sum(x == unk_id for x in ids))

        ws_units = max(1, len(text.split()) + 1)  # +1 for dialect tag
        char_len = max(1, len(text.replace(" ", "")))

        rows.append({
            "eval_set": name,
            "row_index": i,
            "row_uid": r.get("row_uid", stable_hash(f"{name}|{i}|{dialect}|{text}")[:16]),
            "dialect": dialect,
            "validation_origin": r.get("validation_origin", name),
            "is_synthetic": safe_bool(r.get("is_synthetic", False)),
            "text": text,
            "input_text": input_text,
            "char_len_without_spaces": char_len,
            "whitespace_units_with_tag": ws_units,
            "subword_tokens": len(toks),
            "unk_count": unk_count,
            "unk_rate_row": unk_count / max(1, len(toks)),
            "fertility": len(toks) / ws_units,
            "chars_per_subword": char_len / max(1, len(toks)),
            "tokens_preview": token_preview(toks, n=80),
            "ids_preview": " ".join(map(str, ids[:80])),
        })

    return pd.DataFrame(rows)


eval_rows = evaluate_rows(df_validation, "portion5_validation")
eval_rows_path = VALIDATION_DIR / "tokenizer_eval_rows.csv"
eval_rows.to_csv(eval_rows_path, index=False, encoding="utf-8")

fertility_by_dialect = (
    eval_rows.groupby(["dialect", "validation_origin"], dropna=False)
    .agg(
        rows=("row_index", "count"),
        total_subword_tokens=("subword_tokens", "sum"),
        total_unk=("unk_count", "sum"),
        mean_unk_rate=("unk_rate_row", "mean"),
        mean_fertility=("fertility", "mean"),
        median_fertility=("fertility", "median"),
        p95_fertility=("fertility", lambda x: float(np.quantile(x, 0.95))),
        p99_fertility=("fertility", lambda x: float(np.quantile(x, 0.99))),
        std_fertility=("fertility", "std"),
        mean_chars_per_subword=("chars_per_subword", "mean"),
    )
    .reset_index()
)
fertility_by_dialect["unk_rate"] = fertility_by_dialect["total_unk"] / fertility_by_dialect["total_subword_tokens"].clip(lower=1)

fertility_overall = (
    eval_rows.groupby(["validation_origin"], dropna=False)
    .agg(
        rows=("row_index", "count"),
        total_subword_tokens=("subword_tokens", "sum"),
        total_unk=("unk_count", "sum"),
        mean_unk_rate=("unk_rate_row", "mean"),
        mean_fertility=("fertility", "mean"),
        median_fertility=("fertility", "median"),
        p95_fertility=("fertility", lambda x: float(np.quantile(x, 0.95))),
        p99_fertility=("fertility", lambda x: float(np.quantile(x, 0.99))),
        std_fertility=("fertility", "std"),
        mean_chars_per_subword=("chars_per_subword", "mean"),
    )
    .reset_index()
)
fertility_overall["unk_rate"] = fertility_overall["total_unk"] / fertility_overall["total_subword_tokens"].clip(lower=1)

fertility_by_dialect.to_csv(REPORT_DIR / "fertility_by_dialect.csv", index=False, encoding="utf-8")
fertility_overall.to_csv(REPORT_DIR / "fertility_overall.csv", index=False, encoding="utf-8")

print("Overall fertility:")
print(fertility_overall.round(6).to_string(index=False))
print("\nFertility by dialect:")
print(
    fertility_by_dialect[
        ["dialect", "validation_origin", "rows", "unk_rate", "mean_fertility", "p95_fertility", "p99_fertility", "mean_chars_per_subword"]
    ].round(6).sort_values(["dialect", "validation_origin"]).to_string(index=False)
)
print(f"\nSaved: {eval_rows_path}")
print(f"Saved: {REPORT_DIR / 'fertility_by_dialect.csv'}")


# %% [markdown]
# ## Cell 8 — Fertility Outlier Audit

# %%
print("\n" + "=" * 100)
print("CELL 8: FERTILITY OUTLIER AUDIT")
print("=" * 100)

q_threshold = float(eval_rows["fertility"].quantile(CFG.row_outlier_quantile))
outlier_threshold = max(CFG.absolute_fertility_outlier_threshold, q_threshold)

fertility_outliers = eval_rows[eval_rows["fertility"] >= outlier_threshold].copy()
fertility_outliers = fertility_outliers.sort_values(
    ["fertility", "subword_tokens"], ascending=False, kind="mergesort"
).reset_index(drop=True)

fertility_outliers["text_preview"] = fertility_outliers["text"].map(lambda x: compact_text(x, 200))

outlier_path = REPORT_DIR / "fertility_outlier_rows.csv"
fertility_outliers.to_csv(outlier_path, index=False, encoding="utf-8")

outlier_summary = (
    fertility_outliers.groupby("dialect")
    .agg(
        outlier_rows=("row_index", "count"),
        max_fertility=("fertility", "max"),
        mean_fertility=("fertility", "mean"),
        mean_subword_tokens=("subword_tokens", "mean"),
    )
    .reset_index()
    .sort_values("outlier_rows", ascending=False)
)
outlier_summary.to_csv(REPORT_DIR / "fertility_outlier_summary_by_dialect.csv", index=False, encoding="utf-8")

print(f"Outlier threshold: {outlier_threshold:.4f}")
print(f"Outlier rows     : {len(fertility_outliers):,}")
if len(outlier_summary):
    print("\nOutlier summary by dialect:")
    print(outlier_summary.round(4).to_string(index=False))
    print("\nTop 20 outlier rows:")
    print(
        fertility_outliers[
            ["dialect", "validation_origin", "fertility", "subword_tokens", "whitespace_units_with_tag", "text_preview", "tokens_preview"]
        ].head(20).round(4).to_string(index=False)
    )
print(f"\nSaved: {outlier_path}")


# %% [markdown]
# ## Cell 9 — Smoke Tests and Decode Checks

# %%
print("\n" + "=" * 100)
print("CELL 9: SMOKE TESTS + DECODE CHECKS")
print("=" * 100)

smoke_texts = [
    "<dial:BAR> আমি এখন যাই না",
    "<dial:CHI> আঁই ঘরত যাই",
    "<dial:SYL> তাইন কিলা আছেন",
    "<dial:RAJ> আমি এইডা করবো",
    "<dial:KHU> রাখছস এইজায়গায়",
    "<dial:KIS> আঁই আইজ বাজার যাইতাছি",
    "<dial:NSD> অহন এইহানে কাম করতাছি",
    "<dial:TAN> ক্যান এত দেরি হইলো",
    "<dial:STD> আমি আজকে বাজারে যাচ্ছি",
]

smoke_rows = []
for text in smoke_texts:
    enc = final_tokenizer.encode(text)
    ids = list(enc.ids)
    toks = list(enc.tokens)
    decoded = final_tokenizer.decode(ids, skip_special_tokens=False)
    smoke_rows.append({
        "text": text,
        "ids": ids,
        "tokens": toks,
        "decoded": decoded,
        "unk_count": int(sum(x == unk_id for x in ids)),
        "starts_with_atomic_dialect_tag": bool(len(toks) > 0 and is_dialect_tag(toks[0])),
    })

smoke_path = VALIDATION_DIR / "final_smoke_tests.json"
write_json(smoke_path, smoke_rows)

smoke_df = pd.DataFrame([
    {
        "text": r["text"],
        "num_tokens": len(r["tokens"]),
        "unk_count": r["unk_count"],
        "starts_with_atomic_dialect_tag": r["starts_with_atomic_dialect_tag"],
        "tokens_preview": " ".join(r["tokens"]),
        "decoded": r["decoded"],
    }
    for r in smoke_rows
])
smoke_df.to_csv(VALIDATION_DIR / "final_smoke_tests.csv", index=False, encoding="utf-8")

print(smoke_df.to_string(index=False))
print(f"Saved: {smoke_path}")


# %% [markdown]
# ## Cell 10 — Freeze Final Tokenizer Directory

# %%
print("\n" + "=" * 100)
print("CELL 10: FREEZE FINAL TOKENIZER DIRECTORY")
print("=" * 100)

if FROZEN_DIR.exists():
    shutil.rmtree(FROZEN_DIR)
FROZEN_DIR.mkdir(parents=True, exist_ok=True)

# Copy all final tokenizer files from Portion 4.
for item in P4_FINAL_DIR.iterdir():
    dst = FROZEN_DIR / item.name
    if item.is_file():
        shutil.copy2(item, dst)
    elif item.is_dir():
        shutil.copytree(item, dst)

# Add Portion 5 reports into frozen dir for portability.
for src in [
    REPORT_DIR / "protected_token_id_table.csv",
    REPORT_DIR / "special_token_id_table.csv",
    REPORT_DIR / "strict_protected_token_id_table.csv",
    REPORT_DIR / "fertility_by_dialect.csv",
    REPORT_DIR / "fertility_overall.csv",
    ROUTING_DIR / "routing_manifest.json",
    ROUTING_DIR / "dialect_token_id_map.json",
]:
    if src.exists():
        shutil.copy2(src, FROZEN_DIR / src.name)

# Ensure core files exist.
assert (FROZEN_DIR / "tokenizer.json").exists(), "Frozen tokenizer.json missing."
assert (FROZEN_DIR / "tokenizer_config.json").exists(), "Frozen tokenizer_config.json missing."
assert (FROZEN_DIR / "special_tokens_map.json").exists(), "Frozen special_tokens_map.json missing."

# Compute file hashes.
frozen_hashes = {}
for p in sorted(FROZEN_DIR.rglob("*")):
    if p.is_file():
        frozen_hashes[str(p.relative_to(FROZEN_DIR))] = file_sha256(p)

write_json(FROZEN_DIR / "file_hashes_sha256.json", frozen_hashes)
write_json(P5_DIR / "file_hashes_sha256.json", frozen_hashes)

print(f"Frozen tokenizer saved to: {FROZEN_DIR}")
print("Frozen files:")
for k in frozen_hashes.keys():
    print(" -", k)


# %% [markdown]
# ## Cell 11 — Model I/O Config and Tokenizer Manifest

# %%
print("\n" + "=" * 100)
print("CELL 11: MODEL I/O CONFIG + TOKENIZER MANIFEST")
print("=" * 100)

selected_overall = fertility_overall.copy()
mean_fertility = float(eval_rows["fertility"].mean())
unk_rate = float(eval_rows["unk_count"].sum() / max(1, eval_rows["subword_tokens"].sum()))
dialect_mean_fertility = fertility_by_dialect.groupby("dialect")["mean_fertility"].mean().to_dict()
max_dialect_mean_fertility = float(max(dialect_mean_fertility.values())) if dialect_mean_fertility else 0.0

model_io_config = {
    "tokenizer_type": "deterministic_wordpiece",
    "tokenizer_backend": "huggingface_tokenizers",
    "tokenizer_file": "tokenizer.json",
    "vocab_size": int(vocab_size),
    "special_token_ids": {
        "pad_token_id": int(pad_id),
        "unk_token_id": int(unk_id),
        "bos_token_id": int(bos_id),
        "eos_token_id": int(eos_id),
    },
    "special_tokens": {
        "pad_token": CFG.pad_token,
        "unk_token": CFG.unk_token,
        "bos_token": CFG.bos_token,
        "eos_token": CFG.eos_token,
    },
    "dialect_tag_ids": {d: int(v["tag_id"]) for d, v in dialect_token_id_map.items()},
    "dialects": ALL_DIALECTS,
    "target_dialects_10": list(CFG.target_dialects_10),
    "auxiliary_dialects": list(CFG.auxiliary_dialects),
    "padding_side": CFG.padding_side,
    "truncation_side": CFG.truncation_side,
    "max_seq_len_default": CFG.max_seq_len_default,
    "input_format": "<dial:XXX> text",
    "causal_lm_training_notes": [
        "Prefix every sequence with exactly one dialect tag.",
        "Use bos/eos according to the SLM/LLM data collator policy.",
        "Do not replace dialect tags with UNK; tags are atomic vocabulary entries.",
        "For ablation, remove dialect tags only in a separate no-tag experiment.",
    ],
}

write_json(P5_DIR / "model_io_config.json", model_io_config)
write_json(FROZEN_DIR / "model_io_config.json", model_io_config)

manifest = {
    "portion": "tokenizer_p5_final_freeze_validation_routing_calibration",
    "created_at": now(),
    "seed": SEED,
    "config": asdict(CFG),
    "tokenizer_name": "Bangla Dialect-Aware Deterministic WordPiece Tokenizer",
    "tokenizer_backend": "huggingface_tokenizers",
    "tokenizer_path": str(FROZEN_DIR / "tokenizer.json"),
    "vocab_size": int(vocab_size),
    "file_hashes_sha256": frozen_hashes,
    "special_token_ids": model_io_config["special_token_ids"],
    "dialect_tag_ids": model_io_config["dialect_tag_ids"],
    "symbol_validation": symbol_summary,
    "corpus_validation": {
        "validation_rows": int(len(eval_rows)),
        "unk_rate": unk_rate,
        "mean_fertility": mean_fertility,
        "max_dialect_mean_fertility": max_dialect_mean_fertility,
        "dialect_mean_fertility": {k: float(v) for k, v in dialect_mean_fertility.items()},
        "fertility_outlier_threshold": float(outlier_threshold),
        "fertility_outlier_rows": int(len(fertility_outliers)),
    },
    "routing_outputs": {
        "routing_manifest": str(ROUTING_DIR / "routing_manifest.json"),
        "dialect_specific_token_ids": str(ROUTING_DIR / "dialect_specific_token_ids.csv"),
        "core_token_ids": str(ROUTING_DIR / "core_token_ids.csv"),
    },
    "frozen_outputs": {
        "frozen_dir": str(FROZEN_DIR),
        "model_io_config": str(FROZEN_DIR / "model_io_config.json"),
        "file_hashes": str(FROZEN_DIR / "file_hashes_sha256.json"),
    },
}

write_json(P5_DIR / "tokenizer_manifest.json", manifest)
write_json(FROZEN_DIR / "tokenizer_manifest.json", manifest)

print(f"Saved: {P5_DIR / 'model_io_config.json'}")
print(f"Saved: {P5_DIR / 'tokenizer_manifest.json'}")
print(f"Copied into frozen dir: {FROZEN_DIR}")


# %% [markdown]
# ## Cell 12 — Tokenizer Card

# %%
print("\n" + "=" * 100)
print("CELL 12: TOKENIZER CARD")
print("=" * 100)

card = f"""# Bangla Dialect-Aware Deterministic WordPiece Tokenizer

## Summary

This tokenizer is the frozen tokenizer for the Bangla dialect-aware SLM/LLM pipeline. It is a deterministic WordPiece tokenizer built from cleaned 13-label tokenizer-training data: 10 target dialect/control labels and 3 auxiliary regional varieties.

## Dialect space

Target/control dialect labels:

```text
{', '.join(CFG.target_dialects_10)}
```

Auxiliary regional labels:

```text
{', '.join(CFG.auxiliary_dialects)}
```

Input format:

```text
<dial:XXX> utterance text
```

## Tokenizer design

The tokenizer uses:

1. Atomic special tokens: `{', '.join(SPECIAL_TOKENS)}`.
2. Atomic dialect tags for all dialect labels.
3. Strict protected lexical tokens from Portion 3 dialect-aware token mining.
4. Bengali character-level base and continuation fallback tokens.
5. Frequency-ranked whole-word and mined subword pieces.

This design avoids iterative SentencePiece/BPE training instability while preserving dialect tags and protected dialect markers as atomic vocabulary entries.

## Frozen vocabulary

- Vocab size: `{vocab_size}`
- Backend: `huggingface_tokenizers`
- Tokenizer type: `deterministic_wordpiece`
- Tokenizer file: `tokenizer.json`

Special token IDs:

```json
{json.dumps(model_io_config['special_token_ids'], ensure_ascii=False, indent=2)}
```

## Validation summary

- Validation rows: `{len(eval_rows)}`
- UNK rate: `{unk_rate:.8f}`
- Mean fertility: `{mean_fertility:.6f}`
- Max dialect mean fertility: `{max_dialect_mean_fertility:.6f}`
- Special vocab rate: `{symbol_summary['special_vocab_rate']:.6f}`
- Dialect tag vocab rate: `{symbol_summary['dialect_tag_vocab_rate']:.6f}`
- Dialect tag atomic rate: `{symbol_summary['dialect_tag_atomic_rate']:.6f}`
- Strict protected vocab rate: `{symbol_summary['strict_protected_vocab_rate']:.6f}`
- Strict protected atomic rate: `{symbol_summary['strict_protected_atomic_rate']:.6f}`

## Model integration notes

For causal SLM/LLM training:

1. Prefix each sample with exactly one dialect tag.
2. Use `pad_token_id` only for batch padding and mask padding labels with `-100`.
3. Use `bos_token_id` and `eos_token_id` according to the model data-collator policy.
4. Keep dialect tags visible during normal training.
5. For dialect-conditioning ablation, create a separate no-tag dataset rather than deleting tags from this tokenizer.

## Generated artifacts

Core files:

```text
final_tokenizer_frozen/tokenizer.json
final_tokenizer_frozen/tokenizer_config.json
final_tokenizer_frozen/special_tokens_map.json
final_tokenizer_frozen/model_io_config.json
final_tokenizer_frozen/tokenizer_manifest.json
final_tokenizer_frozen/file_hashes_sha256.json
```

Diagnostics:

```text
reports/fertility_by_dialect.csv
reports/fertility_outlier_rows.csv
reports/protected_token_id_table.csv
routing/routing_manifest.json
routing/dialect_specific_token_ids.csv
```
"""

card_path = P5_DIR / "tokenizer_card.md"
with open(card_path, "w", encoding="utf-8") as f:
    f.write(card)

shutil.copy2(card_path, FROZEN_DIR / "tokenizer_card.md")
print(f"Saved: {card_path}")


# %% [markdown]
# ## Cell 13 — Plots

# %%
print("\n" + "=" * 100)
print("CELL 13: PLOTS")
print("=" * 100)

if CFG.save_plots:
    # Plot 1: mean fertility by dialect.
    plot_df = fertility_by_dialect.groupby("dialect", as_index=False)["mean_fertility"].mean().sort_values("dialect")
    plt.figure(figsize=(14, 6))
    plt.bar(plot_df["dialect"], plot_df["mean_fertility"])
    plt.title("Final Tokenizer Mean Fertility by Dialect")
    plt.xlabel("Dialect")
    plt.ylabel("Mean fertility")
    plt.xticks(rotation=45)
    plt.tight_layout()
    p = PLOT_DIR / "final_mean_fertility_by_dialect.png"
    plt.savefig(p, dpi=200)
    plt.close()
    print(f"Saved: {p}")

    # Plot 2: p95 fertility by dialect.
    p95_df = fertility_by_dialect.groupby("dialect", as_index=False)["p95_fertility"].mean().sort_values("dialect")
    plt.figure(figsize=(14, 6))
    plt.bar(p95_df["dialect"], p95_df["p95_fertility"])
    plt.title("Final Tokenizer P95 Fertility by Dialect")
    plt.xlabel("Dialect")
    plt.ylabel("P95 fertility")
    plt.xticks(rotation=45)
    plt.tight_layout()
    p = PLOT_DIR / "final_p95_fertility_by_dialect.png"
    plt.savefig(p, dpi=200)
    plt.close()
    print(f"Saved: {p}")

    # Plot 3: outlier rows by dialect.
    if len(outlier_summary):
        out_plot = outlier_summary.sort_values("dialect")
        plt.figure(figsize=(14, 6))
        plt.bar(out_plot["dialect"], out_plot["outlier_rows"])
        plt.title("Fertility Outlier Rows by Dialect")
        plt.xlabel("Dialect")
        plt.ylabel("Outlier rows")
        plt.xticks(rotation=45)
        plt.tight_layout()
        p = PLOT_DIR / "fertility_outlier_rows_by_dialect.png"
        plt.savefig(p, dpi=200)
        plt.close()
        print(f"Saved: {p}")
else:
    print("Plot saving disabled.")


# %% [markdown]
# ## Cell 14 — Final Tests

# %%
print("\n" + "=" * 100)
print("CELL 14: FINAL PORTION 5 TESTS")
print("=" * 100)

failures = []
warnings_list = []

# Symbol tests.
if symbol_summary["special_vocab_rate"] < CFG.min_special_vocab_rate:
    failures.append(f"Special vocab rate too low: {symbol_summary['special_vocab_rate']}")
if symbol_summary["dialect_tag_vocab_rate"] < CFG.min_dialect_tag_vocab_rate:
    failures.append(f"Dialect tag vocab rate too low: {symbol_summary['dialect_tag_vocab_rate']}")
if symbol_summary["dialect_tag_atomic_rate"] < CFG.min_dialect_tag_atomic_rate:
    failures.append(f"Dialect tag atomic rate too low: {symbol_summary['dialect_tag_atomic_rate']}")
if symbol_summary["strict_protected_vocab_rate"] < CFG.min_strict_protected_vocab_rate:
    failures.append(f"Strict protected vocab rate too low: {symbol_summary['strict_protected_vocab_rate']}")
if symbol_summary["strict_protected_atomic_rate"] < CFG.min_strict_protected_atomic_rate:
    failures.append(f"Strict protected atomic rate too low: {symbol_summary['strict_protected_atomic_rate']}")

# Corpus tests.
if unk_rate > CFG.max_allowed_unk_rate:
    failures.append(f"UNK rate too high: {unk_rate}")
if mean_fertility > CFG.max_mean_fertility:
    failures.append(f"Mean fertility too high: {mean_fertility}")
if max_dialect_mean_fertility > CFG.max_dialect_mean_fertility:
    failures.append(f"Max dialect mean fertility too high: {max_dialect_mean_fertility}")

# Smoke tests.
smoke_unk = int(smoke_df["unk_count"].sum())
if smoke_unk > CFG.max_allowed_smoke_unk_count:
    failures.append(f"Smoke test UNK count too high: {smoke_unk}")
if not bool(smoke_df["starts_with_atomic_dialect_tag"].all()):
    failures.append("Some smoke tests do not start with an atomic dialect tag.")

# File tests.
required_frozen_files = [
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "model_io_config.json",
    "tokenizer_manifest.json",
    "file_hashes_sha256.json",
    "tokenizer_card.md",
]
for fname in required_frozen_files:
    if not (FROZEN_DIR / fname).exists():
        failures.append(f"Missing frozen file: {fname}")

# Routing tests.
if len(dialect_token_id_map) != len(ALL_DIALECTS):
    failures.append("Dialect token ID map is incomplete.")
if len(set(v["tag_id"] for v in dialect_token_id_map.values())) != len(ALL_DIALECTS):
    failures.append("Dialect tag IDs are not unique.")

# Warnings, not hard failures.
if len(fertility_outliers) > 0:
    warnings_list.append(f"Fertility outlier rows present: {len(fertility_outliers)}. Review reports/fertility_outlier_rows.csv.")
if symbol_summary["expanded_protected_vocab_rate"] < 1.0:
    warnings_list.append(f"Expanded protected vocab rate < 1.0: {symbol_summary['expanded_protected_vocab_rate']:.6f}. Not a hard failure because strict tokens are final.")

portion5_test_results = {
    "passed": len(failures) == 0,
    "hard_failures": failures,
    "warnings": warnings_list,
    "vocab_size": int(vocab_size),
    "unk_rate": float(unk_rate),
    "mean_fertility": float(mean_fertility),
    "max_dialect_mean_fertility": float(max_dialect_mean_fertility),
    "smoke_unk_count": int(smoke_unk),
    "symbol_summary": symbol_summary,
    "frozen_dir": str(FROZEN_DIR),
}

write_json(P5_DIR / "p5_test_results.json", portion5_test_results)
write_json(FROZEN_DIR / "p5_test_results.json", portion5_test_results)

for k, v in portion5_test_results.items():
    if k not in {"hard_failures", "warnings", "symbol_summary"}:
        print(f"{k:45s}: {v}")

if warnings_list:
    print("\n[WARNINGS]")
    for w in warnings_list:
        print(" -", w)

if failures:
    print("\n[FAILED]")
    for f in failures:
        print(" -", f)
    raise AssertionError("Tokenizer Portion 5 tests failed.")

print("\n[OK] Portion 5 tests passed.")


# %% [markdown]
# ## Cell 15 — Final Checkpoint

# %%
print("\n" + "=" * 100)
print("CHECKPOINT: TOKENIZER PORTION 5 COMPLETE")
print("=" * 100)

print(f"Frozen tokenizer dir       : {FROZEN_DIR}")
print(f"Tokenizer JSON             : {FROZEN_DIR / 'tokenizer.json'}")
print(f"Tokenizer manifest         : {FROZEN_DIR / 'tokenizer_manifest.json'}")
print(f"Model I/O config           : {FROZEN_DIR / 'model_io_config.json'}")
print(f"Tokenizer card             : {FROZEN_DIR / 'tokenizer_card.md'}")
print(f"Vocab size                 : {vocab_size:,}")
print(f"UNK rate                   : {unk_rate:.8f}")
print(f"Mean fertility             : {mean_fertility:.6f}")
print(f"Max dialect mean fertility : {max_dialect_mean_fertility:.6f}")
print(f"Strict vocab rate          : {symbol_summary['strict_protected_vocab_rate']:.6f}")
print(f"Strict atomic rate         : {symbol_summary['strict_protected_atomic_rate']:.6f}")
print(f"Dialect tag atomic rate    : {symbol_summary['dialect_tag_atomic_rate']:.6f}")

print("\nDialect tag IDs:")
print(pd.DataFrame(dialect_token_id_map.values()).sort_values("dialect").to_string(index=False))

print("\nMean fertility by dialect:")
print(
    fertility_by_dialect.groupby("dialect", as_index=False)["mean_fertility"]
    .mean()
    .round(6)
    .sort_values("dialect")
    .to_string(index=False)
)

print("\nMain artifacts:")
for p in [
    P5_DIR / "tokenizer_manifest.json",
    P5_DIR / "model_io_config.json",
    P5_DIR / "p5_test_results.json",
    REPORT_DIR / "fertility_by_dialect.csv",
    REPORT_DIR / "fertility_outlier_rows.csv",
    REPORT_DIR / "protected_token_id_table.csv",
    ROUTING_DIR / "routing_manifest.json",
    ROUTING_DIR / "dialect_specific_token_ids.csv",
    P5_DIR / "tokenizer_card.md",
    FROZEN_DIR,
]:
    print(p)

print("\n✅ TOKENIZER PORTION 5 COMPLETE — TOKENIZER FROZEN")
print("Next: SLM/LLM Portion 1 — model-ready data pipeline, packing, batch sampling, and tokenizer integration.")


TOKENIZER PORTION 5 — FINAL FREEZE, VALIDATION, ROUTING CALIBRATION
P4 final dir : artifacts/tokenizer_p4_wordpiece/final_tokenizer
P3 dir       : artifacts/tokenizer_p3
Output dir   : artifacts/tokenizer_p5_freeze
Frozen dir   : artifacts/tokenizer_p5_freeze/final_tokenizer_frozen
Dialects     : ['BAR', 'CHI', 'KHU', 'KIS', 'MYM', 'NAR', 'NOA', 'NSD', 'RAJ', 'RAN', 'STD', 'SYL', 'TAN']
GPU needed   : NO — CPU-only validation and manifest generation.

CELL 3: LOAD TOKENIZER + SYMBOL LISTS
Tokenizer vocab size     : 32,000
Special tokens from file : ['<pad>', '<unk>', '<bos>', '<eos>']
Dialect tags from file   : 13
Strict all symbols       : 3,696
Strict lexical symbols   : 3,679
Expanded lexical symbols : 6,774
pad/unk/bos/eos IDs      : 0, 1, 2, 3

CELL 4: LOAD TRAIN/HOLDOUT VALIDATION DATA
Train rows   : 121,503
Holdout rows : 2,875

Train dialect counts:
dialect
BAR     8466
CHI    14592
KHU     8126
KIS     9899
MYM     8993
NAR     8952
NOA     8663
NSD     9486
RAJ     8955
RAN  

## Bangla Dialect-Aware Deterministic WordPiece Tokenizer